# ICEHG DS — Final Reviewer-Rebuild v3 (Reviewer 3 Ready)

**Purpose.** This notebook is the single authoritative analysis source for the revised manuscript. It rebuilds the study from the public ICEHG DS v1.0.1 files, regenerates the primary CWT scalograms, runs the corrected pregnancy-wise nested cross-validation, and executes all reviewer-requested sensitivity, robustness, confounder, ablation, bootstrap, and visualization analyses.

## Prespecified analysis hierarchy

- **Primary pipeline:** manuscript-consistent legacy Morlet representation: 0.08–5 Hz filtered EHG, 120-s windows with 60-s stride, `morl`, integer scales 1–128, per-segment/channel min–max mapping to uint8, bilinear resize to 224×224, model input `/255`.
- **Reviewer sensitivity pipelines:** evaluated without changing the primary pipeline based on held-out test performance:
  1. legacy scales + bicubic resize;
  2. physical 0.08–5 Hz Morlet frequency grid + uint8/bilinear;
  3. physical 0.08–5 Hz Morlet grid + `log1p(power)` floating-point representation.
- **Primary discrimination summary:** arithmetic mean of the five independently calculated held-out outer-fold raw ROC-AUC values; sample SD across folds.
- **Pooled ROC-AUC:** secondary descriptive result only.
- **Nested validation correction:** each inner development split is divided into pregnancy-disjoint **fit**, **early-stopping**, and **untouched evaluation** subsets. Fusion weights, Platt calibration, and decision threshold are selected only from untouched inner-evaluation OOF predictions.
- **No pipeline/seed selection from outer-test performance.**

## Reviewer coverage included in this notebook

Paired-pregnancy analysis; paired vs singleton subgroup analysis; clinical/acquisition metadata confounding; delivery-mode subgroups; signal-integrity audit; metadata-only diagnostic baseline; fold-macro AUC; pregnancy-cluster bootstrap CIs; repeated-seed sensitivity; CWT/frequency/normalization/resizing sensitivity; simple spectral baselines; CNN-only/Transformer-only/full CNN–Transformer ablation; equal vs optimized fusion; paired bootstrap comparisons with Holm adjustment; calibration; ROC/confusion/reliability figures; and score-based, heatmap-independent Grad-CAM case selection.

All outputs are written under a **new Drive root** and the notebook is resume-safe.

## Clean-run rule

This is the **single clean master notebook** for the revised analysis. Do not mix cells from older ICEHG notebooks into this run. Start from a fresh Colab runtime and execute cells in order. All generated caches, models, predictions, tables, figures, and reviewer analyses are written to the new isolated Drive root `ICEHG_FINAL_REVIEWER_REBUILD_CLEAN_V2_2026`. Only the existing public raw-data store is reused. The raw-data download is resume-safe: existing valid files are preserved, the official PhysioNet AWS mirror is used to complete missing files, and modeling cannot start until the original WFDB cohort is verified as exactly **126 records / 91 pregnancies / 62 early / 64 later / 35 paired / 56 singleton**.

The MATLAB-format duplicate directory `icehgdsmat/` is excluded from the primary raw-data scan.


> **Reviewer-3-ready resume policy.** Run this notebook after the currently running CLEAN v2 notebook has completed. It uses the **same project root** and keeps all rebuild/retrain flags `False`, so completed primary folds, sensitivity variants, ablations, repeated-seed outputs, and baseline outputs are reused. The new work added in v3 is the support-based CWT boundary-effect sensitivity analysis plus reviewer-facing reproducibility/audit exports and a larger Figure 8. Do not delete the existing CLEAN_V2 project folder before running this notebook.


In [ ]:

# ============================================================
# 1. PACKAGES, DRIVE, IMPORTS
# ============================================================
import sys, subprocess, importlib.util, os

PACKAGES = {
    "wfdb": "wfdb",
    "pywt": "PyWavelets",
    "skimage": "scikit-image",
    "PIL": "Pillow",
    "openpyxl": "openpyxl",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
}

missing = [pip_name for module_name, pip_name in PACKAGES.items()
           if importlib.util.find_spec(module_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from google.colab import drive
drive.mount("/content/drive")

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import gc, re, json, time, zlib, random, shutil, warnings, platform, hashlib
from pathlib import Path
from itertools import product
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

import scipy
from scipy.signal import butter, filtfilt, welch
from scipy.stats import binomtest

import pywt
import wfdb
from PIL import Image
from skimage.transform import resize

import sklearn
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, brier_score_loss,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, models, backend as K

warnings.filterwarnings("ignore", category=FutureWarning)

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
subprocess.run(["nvidia-smi"], check=False)


In [ ]:

# ============================================================
# 2. SINGLE NEW PROJECT ROOT + PRESPECIFIED SETTINGS
# ============================================================

# Clean v2 outputs are isolated from every older model/cache/result folder.
# Only the verified public raw ICEHG download is reused so it does not need to
# be downloaded again.
PROJECT_ROOT = Path("/content/drive/MyDrive/ICEHG_FINAL_REVIEWER_REBUILD_CLEAN_V2_2026")
DATA_ROOT = Path("/content/drive/MyDrive/ICEHG_FINAL_REVIEWER_REBUILD_2026/01_raw_icehg")
FILTERED_ROOT = PROJECT_ROOT / "02_filtered_signals"
CACHE_ROOT = PROJECT_ROOT / "03_scalograms"
SPLIT_ROOT = PROJECT_ROOT / "04_splits"
MODEL_DIR = PROJECT_ROOT / "05_models"
PRED_DIR = PROJECT_ROOT / "06_predictions"
TABLE_DIR = PROJECT_ROOT / "07_tables"
FIG_DIR = PROJECT_ROOT / "08_figures"
LOG_DIR = PROJECT_ROOT / "09_logs"
SENSITIVITY_ROOT = PROJECT_ROOT / "10_sensitivity"
BASELINE_ROOT = PROJECT_ROOT / "11_simple_baselines"
META_ROOT = PROJECT_ROOT / "12_metadata_confounder"
REPEAT_ROOT = PROJECT_ROOT / "13_repeated_seed"
ABLATION_ROOT = PROJECT_ROOT / "14_architecture_ablation"
GRADCAM_ROOT = PROJECT_ROOT / "15_gradcam"
EXPORT_DIR = PROJECT_ROOT / "16_zenodo_export"

for p in [
    PROJECT_ROOT, DATA_ROOT, FILTERED_ROOT, CACHE_ROOT, SPLIT_ROOT,
    MODEL_DIR, PRED_DIR, TABLE_DIR, FIG_DIR, LOG_DIR, SENSITIVITY_ROOT,
    BASELINE_ROOT, META_ROOT, REPEAT_ROOT, ABLATION_ROOT, GRADCAM_ROOT,
    EXPORT_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Data / preprocessing
# -----------------------------
PHYSIONET_URL = "https://physionet.org/files/icehg-ds/1.0.1/"
EXPECTED_RECORDS = 126
EXPECTED_PREGNANCIES = 91
EXPECTED_EARLY = 62
EXPECTED_LATER = 64
EXPECTED_PAIRED_PREGNANCIES = 35
EXPECTED_SINGLETON_PREGNANCIES = 56

FS = 20.0
FILTER_LOW_HZ = 0.08
FILTER_HIGH_HZ = 5.0
FILTER_ORDER = 4

WINDOW_SECONDS = 120
STRIDE_SECONDS = 60
WINDOW_SAMPLES = int(WINDOW_SECONDS * FS)
STRIDE_SAMPLES = int(STRIDE_SECONDS * FS)

WAVELET = "morl"
N_FREQUENCIES = 128
IMAGE_SIZE = 224

# Primary/current manuscript representation.
LEGACY_SCALES = np.arange(1, 129, dtype=float)
LEGACY_CWT_METHOD = "conv"

# Physical-band sensitivity representation.
SAMPLING_PERIOD = 1.0 / FS
BAND_FREQ_DESC = np.geomspace(FILTER_HIGH_HZ, FILTER_LOW_HZ, N_FREQUENCIES)
BAND_SCALES = (
    pywt.central_frequency(WAVELET)
    / (BAND_FREQ_DESC * SAMPLING_PERIOD)
)

# Reviewer 3: finite-segment CWT boundary-effect sensitivity.
# PyWavelets' Morlet wavelet has finite nominal support given by
# ContinuousWavelet.lower_bound / upper_bound (typically -8 / +8 for `morl`).
# Boundary-affected coefficients are masked before resizing in one additional
# sensitivity representation. This is a robustness analysis only.
BOUNDARY_VARIANT = "bandlimited_boundary_masked_uint8"
BOUNDARY_NEUTRAL_UINT8 = 128

# -----------------------------
# Model/training
# -----------------------------
RANDOM_STATE = 42
BASELINE_RANDOM_STATE = 42
REPEATED_SEEDS = [2026, 2027, 2028, 2029, 2030]

BATCH_SIZE = 16
EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 2e-5
DROPOUT = 0.40
L2_WEIGHT = 3e-4

EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 256
NUM_TRANSFORMER_BLOCKS = 2

N_OUTER_SPLITS = 5
N_INNER_SPLITS = 3
REPRESENTATIONS = ["RGB", "S1", "S2", "S3"]

# -----------------------------
# Fusion/calibration/threshold
# -----------------------------
WEIGHT_STEP = 0.05
MIN_WEIGHT = 0.10
MAX_WEIGHT = 0.50
TOP_WEIGHT_CANDIDATES = 50

THRESHOLD_MIN = 0.05
THRESHOLD_MAX = 0.95
THRESHOLD_STEP = 0.005
ROBUST_BA_TOLERANCE = 0.01
PLATT_C = 1.0

# -----------------------------
# Statistical resampling
# -----------------------------
N_BOOTSTRAP_THRESHOLD = 5000
N_BOOTSTRAP_PRIMARY_AUC = 20000
N_BOOTSTRAP_PAIRED = 20000

PRIMARY_BOOTSTRAP_SEED = 20261019
THRESHOLD_BOOTSTRAP_SEED = 20261020
PAIRED_BOOTSTRAP_SEED = 20261021

# -----------------------------
# Resume / rebuild controls
# -----------------------------
FORCE_REDOWNLOAD_DATA = False
FORCE_REBUILD_FILTERED = False
FORCE_REBUILD_PRIMARY_SCALOGRAMS = False
FORCE_REBUILD_SENSITIVITY_SCALOGRAMS = False
FORCE_RETRAIN = False
FORCE_RETRAIN_SENSITIVITY = False
FORCE_RETRAIN_ABLATION = False
RUN_REPEATED_SEEDS = True
RUN_CWT_SENSITIVITY = True
RUN_SIMPLE_BASELINES = True
RUN_METADATA_CONFOUNDER = True
RUN_ARCHITECTURE_ABLATION = True
RUN_GRADCAM = True

FIG_DPI = 600

tf.keras.mixed_precision.set_global_policy("float32")
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Policy:", tf.keras.mixed_precision.global_policy())


In [ ]:

# ============================================================
# 3. REPRODUCIBILITY, ENVIRONMENT, SAVE HELPERS
# ============================================================

def set_global_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    return int(seed)

set_global_seed(RANDOM_STATE)

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "tensorflow": tf.__version__,
    "tensorflow_build_info": tf.sysconfig.get_build_info(),
    "gpu_devices": [d.name for d in tf.config.list_physical_devices("GPU")],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "pywavelets": pywt.__version__,
    "wfdb": wfdb.__version__,
    "pillow": __import__("PIL").__version__,
    "scikit_image": __import__("skimage").__version__,
    "dataset": "PhysioNet ICEHG DS",
    "dataset_version": "1.0.1",
    "dataset_source": PHYSIONET_URL,
    "primary_cwt_method": LEGACY_CWT_METHOD,
    "primary_wavelet": WAVELET,
    "primary_scales": "1..128 integers",
    "filter_hz": [FILTER_LOW_HZ, FILTER_HIGH_HZ],
    "sampling_hz": FS,
    "window_seconds": WINDOW_SECONDS,
    "stride_seconds": STRIDE_SECONDS,
    "random_state_primary": RANDOM_STATE,
    "repeated_seeds": REPEATED_SEEDS,
}

with open(PROJECT_ROOT / "environment_versions.json", "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2)

with open(PROJECT_ROOT / "pip_freeze.txt", "w", encoding="utf-8") as fh:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=fh, check=False)

def save_table(df: pd.DataFrame, name: str, index=False, directory=TABLE_DIR):
    path = Path(directory) / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    return path

def save_figure(fig, stem: str, directory=FIG_DIR):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    png = directory / f"{stem}.png"
    pdf = directory / f"{stem}.pdf"
    fig.savefig(png, dpi=FIG_DPI, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)
    return png, pdf

def save_json(obj, name: str, directory=PROJECT_ROOT):
    path = Path(directory) / f"{name}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    return path

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

print(pd.DataFrame([environment]).T)


In [ ]:
# ============================================================
# 4. DOWNLOAD / VERIFY PUBLIC ICEHG DS v1.0.1
#    CLEAN + RESUME-SAFE VERSION
# ============================================================

# Official PhysioNet public AWS mirror:
#   aws s3 sync --no-sign-request s3://physionet-open/icehg-ds/1.0.1/ DESTINATION
#
# Policy:
# - Never delete already downloaded records unless FORCE_REDOWNLOAD_DATA=True.
# - Re-scan the disk after every download/sync operation.
# - Use only the original WFDB folders for the primary analysis.
# - Exclude the duplicate MATLAB-format copy under icehgdsmat/.
# - Require exactly 126 unique .hea files and matching .dat files.

from collections import Counter
import shutil
import subprocess
import sys
from pathlib import Path


RAW_FOLDER_EXPECTED = {
    "early_induced": 38,
    "early_cesarean": 11,
    "early_induced-cesarean": 13,
    "later_induced": 43,
    "later_cesarean": 8,
    "later_induced-cesarean": 13,
}

PHYSIONET_S3 = "s3://physionet-open/icehg-ds/1.0.1/"


def dataset_header_files():
    """Return only the 126 original WFDB ICEHG headers, excluding icehgdsmat."""
    files = []
    for p in DATA_ROOT.rglob("*.hea"):
        parts_lower = {part.lower() for part in p.parts}
        parent = p.parent.name.lower()

        if "icehgdsmat" in parts_lower:
            continue

        if parent not in RAW_FOLDER_EXPECTED:
            continue

        files.append(p)

    return sorted(files, key=lambda p: (p.parent.name.lower(), p.stem.lower()))


def dataset_audit(headers):
    """Print and validate the raw WFDB cohort currently present on disk."""
    print("\n" + "=" * 88)
    print("ICEHG DS v1.0.1 — LOCAL RAW-DATA AUDIT")
    print("=" * 88)
    print("DATA_ROOT:", DATA_ROOT)

    folder_counts = Counter(p.parent.name.lower() for p in headers)
    names = [p.stem.lower() for p in headers]
    unique_names = set(names)

    print("\nFolder counts")
    for folder, expected in RAW_FOLDER_EXPECTED.items():
        found = int(folder_counts.get(folder, 0))
        print(f"  {folder:28s} {found:3d} / {expected:3d}")

    print("\nTotal .hea files       :", len(headers))
    print("Unique record names    :", len(unique_names))

    duplicate_names = sorted(
        name for name, n in Counter(names).items() if n > 1
    )

    missing_dat = []
    for hea in headers:
        dat = hea.with_suffix(".dat")
        if not dat.exists():
            missing_dat.append((hea, dat))

    print("Duplicate record names :", len(duplicate_names))
    print("Missing .dat companions:", len(missing_dat))

    problems = []

    if len(headers) != EXPECTED_RECORDS:
        problems.append(
            f"Expected {EXPECTED_RECORDS} original WFDB headers, found {len(headers)}."
        )

    if len(unique_names) != EXPECTED_RECORDS:
        problems.append(
            f"Expected {EXPECTED_RECORDS} unique record names, found {len(unique_names)}."
        )

    for folder, expected in RAW_FOLDER_EXPECTED.items():
        found = int(folder_counts.get(folder, 0))
        if found != expected:
            problems.append(
                f"{folder}: expected {expected} headers, found {found}."
            )

    if duplicate_names:
        problems.append(
            "Duplicate record names: " + ", ".join(duplicate_names[:20])
        )

    if missing_dat:
        problems.append(
            f"{len(missing_dat)} original WFDB header(s) have no matching .dat file."
        )

    if problems:
        print("\nAUDIT STATUS: INCOMPLETE")
        for p in problems:
            print(" -", p)
        return False

    print("\nAUDIT STATUS: COMPLETE")
    print("✓ 126 original WFDB headers found.")
    print("✓ 126 unique record names found.")
    print("✓ All six raw folders have the expected record counts.")
    print("✓ Every original .hea file has its corresponding .dat file.")
    print("=" * 88)
    return True


# ------------------------------------------------------------
# Optional full reset — normally KEEP FALSE
# ------------------------------------------------------------

if FORCE_REDOWNLOAD_DATA and DATA_ROOT.exists():
    print("FORCE_REDOWNLOAD_DATA=True -> deleting raw data root.")
    shutil.rmtree(DATA_ROOT)

DATA_ROOT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# First local audit
# ------------------------------------------------------------

headers = dataset_header_files()
print(f"Local raw ICEHG headers found: {len(headers)} / {EXPECTED_RECORDS}")
complete = dataset_audit(headers)


# ------------------------------------------------------------
# Resume-safe official AWS sync only if needed
# ------------------------------------------------------------

if not complete:
    print("\nICEHG DS v1.0.1 is incomplete locally.")
    print("Completing/resuming download from the official PhysioNet public AWS mirror...")

    if shutil.which("aws") is None:
        print("AWS CLI not found; installing awscli...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "awscli"],
            check=True,
        )

    aws_cmd = [
        "aws", "s3", "sync",
        "--no-sign-request",
        PHYSIONET_S3,
        str(DATA_ROOT),
        "--only-show-errors",
    ]

    aws_ok = False

    try:
        subprocess.run(aws_cmd, check=True)
        aws_ok = True
    except Exception as exc:
        print("\nAWS sync did not complete successfully:")
        print(repr(exc))
        print("\nFalling back to resume-capable wget from PhysioNet...")

    # Conservative fallback. Existing files are preserved and partial files resume.
    if not aws_ok:
        wget_cmd = [
            "wget",
            "-r", "-N", "-c", "-np", "-nH",
            "--cut-dirs=3",
            "-R", "index.html*",
            "--timeout=60",
            "--read-timeout=60",
            "--tries=20",
            "--retry-connrefused",
            PHYSIONET_URL,
            "-P", str(DATA_ROOT),
        ]
        subprocess.run(wget_cmd, check=True)

    # CRITICAL: refresh the file list from disk after download.
    headers = dataset_header_files()
    complete = dataset_audit(headers)


# ------------------------------------------------------------
# Final strict gate
# ------------------------------------------------------------

if not complete:
    raise RuntimeError(
        "ICEHG DS v1.0.1 is still incomplete after the resume attempt. "
        "Do not continue to metadata parsing or model training. "
        "Re-run this cell; the download is resume-safe."
    )

# Always rebuild the in-memory list from the current disk state.
headers = dataset_header_files()

print("\nFINAL DATA GATE PASSED")
print(f"headers refreshed from disk: {len(headers)}")
print("Primary analysis can proceed to Cell 5.")


In [ ]:
# ============================================================
# 5. HEADER METADATA, LABELS, PAIRING, PREGNANCY IDs
#    CLEAN + DIAGNOSTIC VERSION
# ============================================================

# Never trust a stale in-memory header list from a previous failed download.
headers = dataset_header_files()

if len(headers) != EXPECTED_RECORDS:
    raise RuntimeError(
        f"Cell 5 requires {EXPECTED_RECORDS} verified raw headers; "
        f"currently found {len(headers)}. Re-run Cell 4 first."
    )

print(f"Cell 5 starting with {len(headers)} freshly scanned raw headers.")

KNOWN_KEYS = [
    "RecID", "RecType", "Gestation", "Rectime", "Age", "Weight",
    "Placental_position", "Height", "Newborn_weight", "Pair_RecID"
]


def parse_comments(comments):
    result = {}

    for raw in comments or []:
        s = str(raw).lstrip("#").strip()

        if not s:
            continue

        matched = False

        for key in KNOWN_KEYS:
            m = re.match(
                rf"^{re.escape(key)}\s*[:=]?\s*(.*)$",
                s,
                flags=re.I,
            )

            if m:
                result[key] = m.group(1).strip()
                matched = True
                break

        if not matched and ":" in s:
            key, value = s.split(":", 1)
            result[key.strip()] = value.strip()

    return result


def normalize_record_id(value):
    if value is None:
        return None

    s = str(value).strip().lower()

    if s in {"", "none", "nan", "na", "n/a", "-", "0"}:
        return None

    digits = re.findall(r"\d+", s)

    if not digits:
        return None

    return f"icehg{int(digits[-1])}"


def infer_period_and_delivery(path: Path):
    folder = path.parent.name.lower()

    if folder.startswith("early_"):
        return (
            "early",
            1,
            folder[len("early_"):].replace("_", "-"),
        )

    if folder.startswith("later_"):
        return (
            "later",
            0,
            folder[len("later_"):].replace("_", "-"),
        )

    raise ValueError(f"Unrecognized ICEHG folder: {folder}")


# ------------------------------------------------------------
# Parse all 126 raw headers
# ------------------------------------------------------------

rows = []

for hea_path in headers:
    base_path = hea_path.with_suffix("")

    try:
        hdr = wfdb.rdheader(str(base_path))
    except Exception as exc:
        raise RuntimeError(
            f"WFDB header read failed for: {hea_path}"
        ) from exc

    comments = parse_comments(hdr.comments)

    period, label, delivery = infer_period_and_delivery(hea_path)

    record_id = (
        normalize_record_id(comments.get("RecID"))
        or normalize_record_id(hea_path.stem)
    )

    pair_id = normalize_record_id(comments.get("Pair_RecID"))

    rows.append({
        "record_id": record_id,
        "record_name": hea_path.stem,
        "record_base_path": str(base_path),
        "header_path": str(hea_path),
        "period": period,
        "label_num": int(label),
        "label_text": period,
        "delivery_mode": delivery,
        "pair_record_id": pair_id,
        "fs": float(hdr.fs),
        "sig_len": int(hdr.sig_len),
        "n_sig": int(hdr.n_sig),
        "signal_names": "|".join(hdr.sig_name or []),
        "units": "|".join(hdr.units or []),
        "gestation_delivery_week_raw": comments.get("Gestation"),
        "recording_week_rectime_raw": comments.get("Rectime"),
        "maternal_age_raw": comments.get("Age"),
        "maternal_weight_raw": comments.get("Weight"),
        "placental_position": comments.get("Placental_position"),
        "maternal_height_raw": comments.get("Height"),
        "newborn_weight_raw": comments.get("Newborn_weight"),
    })


record_meta = (
    pd.DataFrame(rows)
    .sort_values("record_name")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Record-level structural audit before pregnancy reconstruction
# ------------------------------------------------------------

if len(record_meta) != EXPECTED_RECORDS:
    raise RuntimeError(
        f"Metadata parser produced {len(record_meta)} records; "
        f"expected {EXPECTED_RECORDS}."
    )

if record_meta["record_name"].nunique() != EXPECTED_RECORDS:
    dup = record_meta.loc[
        record_meta["record_name"].duplicated(keep=False),
        ["record_name", "header_path"],
    ]
    raise RuntimeError(
        "Duplicate record_name values detected:\n"
        + dup.to_string(index=False)
    )

if record_meta["record_id"].isna().any():
    bad = record_meta.loc[
        record_meta["record_id"].isna(),
        ["record_name", "header_path"],
    ]
    raise RuntimeError(
        "Record ID could not be reconstructed for:\n"
        + bad.to_string(index=False)
    )

if record_meta["record_id"].nunique() != EXPECTED_RECORDS:
    dup = record_meta.loc[
        record_meta["record_id"].duplicated(keep=False),
        ["record_name", "record_id", "header_path"],
    ]
    raise RuntimeError(
        "Duplicate normalized record_id values detected:\n"
        + dup.to_string(index=False)
    )


# ------------------------------------------------------------
# Pregnancy reconstruction using Pair_RecID relationships
# ------------------------------------------------------------

class UnionFind:
    def __init__(self, items):
        self.parent = {x: x for x in items}
        self.rank = {x: 0 for x in items}

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)

        if ra == rb:
            return

        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra

        self.parent[rb] = ra

        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


ids = set(record_meta["record_id"])
uf = UnionFind(ids)

unknown_pair_refs = []

for row in record_meta.itertuples():
    if row.pair_record_id:
        if row.pair_record_id in ids:
            uf.union(row.record_id, row.pair_record_id)
        else:
            unknown_pair_refs.append(
                (row.record_name, row.record_id, row.pair_record_id)
            )

if unknown_pair_refs:
    print("\nWARNING: Pair_RecID references absent from local record IDs:")
    for x in unknown_pair_refs:
        print(x)


components = defaultdict(list)

for rid in ids:
    components[uf.find(rid)].append(rid)


canonical = {}

for members in components.values():

    def num_key(x):
        m = re.findall(r"\d+", str(x))
        return int(m[-1]) if m else 10**12

    pid = f"preg_{min(members, key=num_key)}"

    for rid in members:
        canonical[rid] = pid


record_meta["pregnancy_id"] = record_meta["record_id"].map(canonical)

record_meta["recording_duration_minutes"] = (
    record_meta["sig_len"]
    / record_meta["fs"]
    / 60.0
)

record_meta["expected_120s_segments"] = (
    ((record_meta["sig_len"] - WINDOW_SAMPLES) // STRIDE_SAMPLES) + 1
).clip(lower=0).astype(int)


# ------------------------------------------------------------
# Pregnancy composition
# ------------------------------------------------------------

preg_rows = []

for pid, g in record_meta.groupby("pregnancy_id"):

    has_e = bool((g["label_num"] == 1).any())
    has_l = bool((g["label_num"] == 0).any())

    typ = "paired" if (has_e and has_l) else "singleton"

    preg_rows.append({
        "pregnancy_id": pid,
        "pairing_status": typ,
        "n_records": len(g),
        "early_records": int((g["label_num"] == 1).sum()),
        "later_records": int((g["label_num"] == 0).sum()),
    })


preg_comp = pd.DataFrame(preg_rows)

record_meta = record_meta.merge(
    preg_comp[["pregnancy_id", "pairing_status"]],
    on="pregnancy_id",
    how="left",
    validate="many_to_one",
)


# ------------------------------------------------------------
# Transparent cohort audit BEFORE the strict gate
# ------------------------------------------------------------

cohort_actual = {
    "records": int(len(record_meta)),
    "pregnancies": int(record_meta["pregnancy_id"].nunique()),
    "early_records": int((record_meta["label_num"] == 1).sum()),
    "later_records": int((record_meta["label_num"] == 0).sum()),
    "paired_pregnancies": int(
        (preg_comp["pairing_status"] == "paired").sum()
    ),
    "singleton_pregnancies": int(
        (preg_comp["pairing_status"] == "singleton").sum()
    ),
}

cohort_expected = {
    "records": EXPECTED_RECORDS,
    "pregnancies": EXPECTED_PREGNANCIES,
    "early_records": EXPECTED_EARLY,
    "later_records": EXPECTED_LATER,
    "paired_pregnancies": EXPECTED_PAIRED_PREGNANCIES,
    "singleton_pregnancies": EXPECTED_SINGLETON_PREGNANCIES,
}

print("\n" + "=" * 88)
print("COHORT AUDIT BEFORE MODELING")
print("=" * 88)

for key in cohort_expected:
    print(
        f"{key:24s}: "
        f"{cohort_actual[key]:3d} / expected {cohort_expected[key]:3d}"
    )

print(
    f"{'sampling frequency':24s}: "
    f"{sorted(record_meta['fs'].unique().tolist())}"
)


# ------------------------------------------------------------
# Strict cohort gate with descriptive errors
# ------------------------------------------------------------

mismatches = []

for key, expected in cohort_expected.items():
    actual = cohort_actual[key]

    if actual != expected:
        mismatches.append(
            f"{key}: expected {expected}, found {actual}"
        )

if not np.allclose(record_meta["fs"], FS):
    mismatches.append(
        f"sampling frequency mismatch: expected all {FS} Hz, "
        f"found {sorted(record_meta['fs'].unique().tolist())}"
    )

if mismatches:
    print("\nCOHORT GATE FAILED")
    for item in mismatches:
        print(" -", item)

    raise RuntimeError(
        "ICEHG cohort reconstruction does not match the prespecified "
        "126-record / 91-pregnancy study cohort. "
        "Do not continue until the items printed above are resolved."
    )

print("\nCOHORT GATE PASSED")
print("✓ 126 records")
print("✓ 91 pregnancies")
print("✓ 62 early / 64 later records")
print("✓ 35 paired / 56 singleton pregnancies")
print(f"✓ all records sampled at {FS:g} Hz")


# ------------------------------------------------------------
# Save authoritative metadata
# ------------------------------------------------------------

record_meta.to_csv(
    CACHE_ROOT / "record_metadata.csv",
    index=False,
)

preg_comp.to_csv(
    CACHE_ROOT / "pregnancy_composition.csv",
    index=False,
)


# ------------------------------------------------------------
# Dataset-distribution table
# ------------------------------------------------------------

table_dataset = (
    record_meta.pivot_table(
        index="delivery_mode",
        columns="period",
        values="record_name",
        aggfunc="count",
        fill_value=0,
    )
    .rename(
        columns={
            "early": "Early-recorded, n",
            "later": "Later-recorded, n",
        }
    )
    .reset_index()
)

table_dataset["Total, n"] = (
    table_dataset.get("Early-recorded, n", 0)
    + table_dataset.get("Later-recorded, n", 0)
)

table_dataset = pd.concat(
    [
        table_dataset,
        pd.DataFrame([
            {
                "delivery_mode": "Total",
                "Early-recorded, n": EXPECTED_EARLY,
                "Later-recorded, n": EXPECTED_LATER,
                "Total, n": EXPECTED_RECORDS,
            }
        ]),
    ],
    ignore_index=True,
)

save_table(
    table_dataset,
    "table_01_dataset_distribution",
)

display(table_dataset)


In [ ]:

# ============================================================
# 6. RAW SIGNAL READING, FILTERING, SIGNAL-INTEGRITY AUDIT
#    Per-record filtered signals are cached once and reused.
# ============================================================

def locate_original_channels(sig_names, n_sig):
    names = [str(x).lower().replace("-", " ").replace("_", " ") for x in sig_names]
    selected = []
    for target in ("s1", "s2", "s3"):
        candidates = [
            i for i, name in enumerate(names)
            if target in name and "filter" not in name and "filt" not in name
        ]
        if candidates:
            selected.append(candidates[0])
    if len(selected) == 3:
        return selected
    # Documented ICEHG order: raw S1, filtered S1, raw S2, filtered S2, raw S3, filtered S3.
    if n_sig >= 6:
        return [0, 2, 4]
    raise RuntimeError(f"Raw S1/S2/S3 could not be identified: {sig_names}")

def read_raw_ehg(row):
    rec = wfdb.rdrecord(str(row.record_base_path))
    if rec.p_signal is None:
        raise RuntimeError(f"Physical signal unavailable: {row.record_name}")
    idx = locate_original_channels(rec.sig_name or [], rec.n_sig)
    x = np.asarray(rec.p_signal[:, idx], dtype=np.float64)
    return x, rec

FILTER_B, FILTER_A = butter(
    FILTER_ORDER, [FILTER_LOW_HZ, FILTER_HIGH_HZ],
    btype="bandpass", fs=FS
)

def bandpass_filter(x):
    return filtfilt(FILTER_B, FILTER_A, x, axis=0)

integrity_rows = []
for i, row in enumerate(record_meta.itertuples(), start=1):
    cache_path = FILTERED_ROOT / f"{row.record_name}.npz"
    try:
        raw, rec = read_raw_ehg(row)
        if FORCE_REBUILD_FILTERED or not cache_path.exists():
            filtered = bandpass_filter(raw).astype(np.float32)
            np.savez_compressed(cache_path, filtered=filtered)
        else:
            filtered = np.load(cache_path)["filtered"]

        integrity_rows.append({
            "record_name": row.record_name,
            "read_success": True,
            "missing_raw_channel": False,
            "nonfinite_raw": bool(~np.isfinite(raw).all()),
            "constant_S1": bool(np.nanstd(raw[:, 0]) == 0),
            "constant_S2": bool(np.nanstd(raw[:, 1]) == 0),
            "constant_S3": bool(np.nanstd(raw[:, 2]) == 0),
            "duration_min": raw.shape[0] / FS / 60.0,
            "expected_120s_windows": len(range(0, raw.shape[0] - WINDOW_SAMPLES + 1, STRIDE_SAMPLES)),
        })
    except Exception as exc:
        integrity_rows.append({
            "record_name": row.record_name,
            "read_success": False,
            "missing_raw_channel": True,
            "nonfinite_raw": np.nan,
            "constant_S1": np.nan,
            "constant_S2": np.nan,
            "constant_S3": np.nan,
            "duration_min": np.nan,
            "expected_120s_windows": np.nan,
            "error": repr(exc),
        })
        raise
    if i % 20 == 0:
        print(f"Filtered/audited {i}/{len(record_meta)} records")

integrity_df = pd.DataFrame(integrity_rows)
save_table(integrity_df, "table_S3A_signal_integrity_detail")

signal_audit_summary = pd.DataFrame([
    {"Audit item": "Total records assessed", "Result": len(integrity_df)},
    {"Audit item": "Records successfully read", "Result": int(integrity_df["read_success"].sum())},
    {"Audit item": "Records with read failure", "Result": int((~integrity_df["read_success"]).sum())},
    {"Audit item": "Records missing any raw S1/S2/S3 channel", "Result": int(integrity_df["missing_raw_channel"].sum())},
    {"Audit item": "Records with non-finite raw samples", "Result": int(integrity_df["nonfinite_raw"].fillna(False).sum())},
    {"Audit item": "Records with any completely constant raw channel",
     "Result": int((integrity_df[["constant_S1","constant_S2","constant_S3"]].fillna(False).any(axis=1)).sum())},
    {"Audit item": "Recording duration, min, mean ± SD",
     "Result": f'{integrity_df["duration_min"].mean():.2f} ± {integrity_df["duration_min"].std(ddof=1):.2f}'},
    {"Audit item": "Recording duration, min, range",
     "Result": f'{integrity_df["duration_min"].min():.2f}–{integrity_df["duration_min"].max():.2f}'},
    {"Audit item": "Expected 120-s windows per record, mean",
     "Result": f'{integrity_df["expected_120s_windows"].mean():.2f}'},
    {"Audit item": "Expected 120-s windows per record, range",
     "Result": f'{int(integrity_df["expected_120s_windows"].min())}–{int(integrity_df["expected_120s_windows"].max())}'},
    {"Audit item": "Expert-labeled artifact annotation available", "Result": "No"},
    {"Audit item": "Explicit acquisition-batch identifier available", "Result": "No"},
    {"Audit item": "Equipment/device-state information available", "Result": "No"},
])
save_table(signal_audit_summary, "table_S3A_signal_integrity_summary")
display(signal_audit_summary)


In [ ]:

# ============================================================
# 7. SEGMENT INDEX + PRIMARY LEGACY CWT/SCALOGRAM REBUILD
#    Primary: morl, scales 1..128, conv, magnitude,
#    per-segment/channel min-max -> uint8, PIL bilinear 224x224.
# ============================================================

record_df = (
    record_meta[
        ["record_name", "pregnancy_id", "label_text", "label_num"]
    ]
    .drop_duplicates("record_name")
    .sort_values("record_name")
    .reset_index(drop=True)
)

segment_rows = []
global_idx = 0
for row in record_meta.sort_values("record_name").itertuples():
    filt = np.load(FILTERED_ROOT / f"{row.record_name}.npz")["filtered"]
    starts = list(range(0, filt.shape[0] - WINDOW_SAMPLES + 1, STRIDE_SAMPLES))
    for seg_no, start in enumerate(starts):
        segment_rows.append({
            "segment_index": global_idx,
            "record_name": row.record_name,
            "pregnancy_id": row.pregnancy_id,
            "label_num": int(row.label_num),
            "label_text": row.label_text,
            "delivery_mode": row.delivery_mode,
            "segment_in_record": seg_no,
            "start_sample": int(start),
            "stop_sample": int(start + WINDOW_SAMPLES),
            "start_seconds": float(start / FS),
            "stop_seconds": float((start + WINDOW_SAMPLES) / FS),
        })
        global_idx += 1

segment_df = pd.DataFrame(segment_rows).sort_values("segment_index").reset_index(drop=True)
if len(segment_df) != 3558:
    print(f"WARNING: prior A2 run had 3558 segments; rebuilt dataset has {len(segment_df)}.")

SEGMENT_METADATA_PATH = CACHE_ROOT / "segment_metadata_120s_60stride.csv"
segment_df.to_csv(SEGMENT_METADATA_PATH, index=False)

PRIMARY_SCALOGRAM_DIR = CACHE_ROOT / "primary_legacy_uint8_bilinear"
PRIMARY_SCALOGRAM_DIR.mkdir(parents=True, exist_ok=True)

SCALOGRAM_PATHS = {
    "RGB": PRIMARY_SCALOGRAM_DIR / "RGB_scalograms_120s.npy",
    "S1": PRIMARY_SCALOGRAM_DIR / "S1_scalograms_120s.npy",
    "S2": PRIMARY_SCALOGRAM_DIR / "S2_scalograms_120s.npy",
    "S3": PRIMARY_SCALOGRAM_DIR / "S3_scalograms_120s.npy",
}
PRIMARY_COMPLETE = PRIMARY_SCALOGRAM_DIR / "_COMPLETE.json"

def legacy_uint8_scalogram(segment_1d, resample=Image.Resampling.BILINEAR):
    coeff, freq = pywt.cwt(
        np.asarray(segment_1d, dtype=np.float64),
        LEGACY_SCALES,
        WAVELET,
        sampling_period=SAMPLING_PERIOD,
        method=LEGACY_CWT_METHOD,
    )
    power = (np.abs(coeff) ** 2).astype(np.float32)
    lo = float(np.nanmin(power))
    hi = float(np.nanmax(power))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        scaled = np.zeros_like(power, dtype=np.uint8)
    else:
        scaled = np.clip((power - lo) / (hi - lo), 0, 1)
        scaled = np.round(scaled * 255.0).astype(np.uint8)
    img = Image.fromarray(scaled, mode="L")
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE), resample=resample)
    return np.asarray(img, dtype=np.uint8)

def build_primary_legacy_scalograms():
    expected_shapes = {
        "S1": (len(segment_df), IMAGE_SIZE, IMAGE_SIZE, 1),
        "S2": (len(segment_df), IMAGE_SIZE, IMAGE_SIZE, 1),
        "S3": (len(segment_df), IMAGE_SIZE, IMAGE_SIZE, 1),
        "RGB": (len(segment_df), IMAGE_SIZE, IMAGE_SIZE, 3),
    }

    if (
        not FORCE_REBUILD_PRIMARY_SCALOGRAMS
        and PRIMARY_COMPLETE.exists()
        and all(p.exists() for p in SCALOGRAM_PATHS.values())
    ):
        print("Primary legacy scalograms already complete -> reuse.")
        return

    arrays = {
        rep: np.lib.format.open_memmap(
            path, mode="w+", dtype=np.uint8, shape=expected_shapes[rep]
        )
        for rep, path in SCALOGRAM_PATHS.items()
    }

    metadata_by_record = {
        k: g.sort_values("segment_in_record")
        for k, g in segment_df.groupby("record_name", sort=False)
    }

    for rec_no, row in enumerate(record_meta.sort_values("record_name").itertuples(), start=1):
        filtered = np.load(FILTERED_ROOT / f"{row.record_name}.npz")["filtered"]
        g = metadata_by_record[row.record_name]

        for seg in g.itertuples():
            start, stop, idx = int(seg.start_sample), int(seg.stop_sample), int(seg.segment_index)
            chans = []
            for ch in range(3):
                sc = legacy_uint8_scalogram(filtered[start:stop, ch])
                arrays[f"S{ch+1}"][idx, :, :, 0] = sc
                chans.append(sc)
            arrays["RGB"][idx] = np.stack(chans, axis=-1)

        for arr in arrays.values():
            arr.flush()
        print(f"[primary scalograms {rec_no:03d}/{len(record_meta)}] {row.record_name}")

    for arr in arrays.values():
        arr.flush()
    del arrays
    gc.collect()

    audit = {
        "pipeline": "primary_legacy_uint8_bilinear",
        "wavelet": WAVELET,
        "scales": "integer 1..128",
        "pywt_method": LEGACY_CWT_METHOD,
        "input_filter_hz": [FILTER_LOW_HZ, FILTER_HIGH_HZ],
        "segment_seconds": WINDOW_SECONDS,
        "stride_seconds": STRIDE_SECONDS,
        "scalogram_value": "wavelet power |CWT|^2",
        "normalization": "per-segment/channel min-max to uint8",
        "resize": "PIL bilinear",
        "image_size": IMAGE_SIZE,
        "model_input": "float32 / 255",
        "segments": int(len(segment_df)),
    }
    save_json(audit, "_COMPLETE", PRIMARY_SCALOGRAM_DIR)

build_primary_legacy_scalograms()

# Array audit
array_rows = []
for rep, path in SCALOGRAM_PATHS.items():
    arr = np.load(path, mmap_mode="r")
    array_rows.append({
        "representation": rep,
        "shape": str(arr.shape),
        "dtype": str(arr.dtype),
        "min_sample": float(np.min(arr[:min(10,len(arr))])),
        "max_sample": float(np.max(arr[:min(10,len(arr))])),
    })
array_audit_df = pd.DataFrame(array_rows)
save_table(array_audit_df, "table_S_primary_scalogram_array_audit")
display(array_audit_df)

print("Segment count:", len(segment_df))
print("Record count:", record_df["record_name"].nunique())
print("Pregnancy count:", record_df["pregnancy_id"].nunique())


In [ ]:

# ============================================================
# 8. CWT SCALE–FREQUENCY AUDIT + SENSITIVITY CACHE REBUILD
# ============================================================

legacy_freq = pywt.scale2frequency(WAVELET, LEGACY_SCALES) / SAMPLING_PERIOD
band_actual_freq = pywt.scale2frequency(WAVELET, BAND_SCALES) / SAMPLING_PERIOD

cwt_audit = pd.DataFrame({
    "legacy_scale": LEGACY_SCALES,
    "legacy_pseudo_frequency_hz": legacy_freq,
    "legacy_within_input_passband": (
        (legacy_freq >= FILTER_LOW_HZ) & (legacy_freq <= FILTER_HIGH_HZ)
    ),
    "band_scale": BAND_SCALES,
    "band_requested_frequency_hz": BAND_FREQ_DESC,
    "band_actual_frequency_hz": band_actual_freq,
})
save_table(cwt_audit, "table_S_cwt_scale_frequency_audit")
display(cwt_audit.head())
display(cwt_audit.tail())

# ------------------------------------------------------------
# Reviewer 3 boundary-support audit for the 0.08–5.0 Hz grid.
# ------------------------------------------------------------
_cw = pywt.ContinuousWavelet(WAVELET)
MORLET_LOWER_BOUND = float(_cw.lower_bound)
MORLET_UPPER_BOUND = float(_cw.upper_bound)

# At scale s, the nominal finite support expands by s in sample coordinates.
# We use a conservative support-based exclusion width on each side.
boundary_left_samples = np.ceil(
    np.abs(MORLET_LOWER_BOUND) * BAND_SCALES
).astype(int)
boundary_right_samples = np.ceil(
    np.abs(MORLET_UPPER_BOUND) * BAND_SCALES
).astype(int)

boundary_left_samples = np.minimum(boundary_left_samples, WINDOW_SAMPLES)
boundary_right_samples = np.minimum(boundary_right_samples, WINDOW_SAMPLES)
boundary_interior_samples = np.maximum(
    0, WINDOW_SAMPLES - boundary_left_samples - boundary_right_samples
)
boundary_interior_fraction = boundary_interior_samples / float(WINDOW_SAMPLES)

boundary_audit = pd.DataFrame({
    "frequency_row": np.arange(1, N_FREQUENCIES + 1, dtype=int),
    "requested_frequency_hz": BAND_FREQ_DESC,
    "actual_frequency_hz": band_actual_freq,
    "scale": BAND_SCALES,
    "wavelet_lower_bound": MORLET_LOWER_BOUND,
    "wavelet_upper_bound": MORLET_UPPER_BOUND,
    "left_boundary_samples": boundary_left_samples,
    "right_boundary_samples": boundary_right_samples,
    "left_boundary_seconds": boundary_left_samples / FS,
    "right_boundary_seconds": boundary_right_samples / FS,
    "interior_samples": boundary_interior_samples,
    "interior_seconds": boundary_interior_samples / FS,
    "interior_fraction": boundary_interior_fraction,
    "no_fully_interior_region": boundary_interior_samples <= 0,
})
save_table(
    boundary_audit,
    "table_S_cwt_boundary_support_audit",
    directory=SENSITIVITY_ROOT,
)
display(boundary_audit.head())
display(boundary_audit.tail())

SENS_VARIANTS = {
    "legacy_bilinear_minmax_float16": {
        "path": CACHE_ROOT / "sensitivity_RGB_legacy_bilinear_minmax_float16.npy",
        "dtype": np.float16,
    },
    "legacy_bicubic_uint8": {
        "path": CACHE_ROOT / "sensitivity_RGB_legacy_bicubic_uint8.npy",
        "dtype": np.uint8,
    },
    "bandlimited_bilinear_uint8": {
        "path": CACHE_ROOT / "sensitivity_RGB_bandlimited_bilinear_uint8.npy",
        "dtype": np.uint8,
    },
    "bandlimited_logpower_float16": {
        "path": CACHE_ROOT / "sensitivity_RGB_bandlimited_logpower_float16.npy",
        "dtype": np.float16,
    },
    "bandlimited_boundary_masked_uint8": {
        "path": CACHE_ROOT / "sensitivity_RGB_bandlimited_boundary_masked_uint8.npy",
        "dtype": np.uint8,
    },
}

def cwt_uint8(segment_1d, scales, resample):
    coeff, _ = pywt.cwt(
        np.asarray(segment_1d, dtype=np.float64),
        scales, WAVELET, sampling_period=SAMPLING_PERIOD, method="conv"
    )
    power = (np.abs(coeff) ** 2).astype(np.float32)
    lo, hi = float(np.nanmin(power)), float(np.nanmax(power))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        scaled = np.zeros_like(power, dtype=np.uint8)
    else:
        scaled = np.round(np.clip((power-lo)/(hi-lo),0,1)*255).astype(np.uint8)
    return np.asarray(
        Image.fromarray(scaled, mode="L").resize((IMAGE_SIZE, IMAGE_SIZE), resample=resample),
        dtype=np.uint8
    )

def cwt_minmax_float(segment_1d):
    coeff, _ = pywt.cwt(
        np.asarray(segment_1d, dtype=np.float64),
        LEGACY_SCALES, WAVELET, sampling_period=SAMPLING_PERIOD, method=LEGACY_CWT_METHOD
    )
    power = (np.abs(coeff) ** 2).astype(np.float32)
    lo, hi = float(np.nanmin(power)), float(np.nanmax(power))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        scaled = np.zeros_like(power, dtype=np.float32)
    else:
        scaled = np.clip((power-lo)/(hi-lo),0,1).astype(np.float32)
    img = Image.fromarray(scaled, mode="F").resize(
        (IMAGE_SIZE, IMAGE_SIZE), resample=Image.Resampling.BILINEAR
    )
    return np.asarray(img, dtype=np.float16)

def cwt_logpower_float(segment_1d):
    coeff, _ = pywt.cwt(
        np.asarray(segment_1d, dtype=np.float64),
        BAND_SCALES, WAVELET, sampling_period=SAMPLING_PERIOD, method="conv"
    )
    x = np.log1p(np.abs(coeff) ** 2).astype(np.float32)
    # Same high->low row ordering as requested physical frequency grid.
    x = resize(
        x, (IMAGE_SIZE, IMAGE_SIZE),
        order=1, mode="reflect", preserve_range=True, anti_aliasing=True
    ).astype(np.float16)
    return x


def cwt_boundary_masked_uint8(segment_1d):
    """
    Reviewer 3 sensitivity variant:
    0.08–5.0 Hz Morlet grid with scale-dependent finite-support boundary masking.

    The mask is defined from PyWavelets ContinuousWavelet lower/upper bounds.
    Only coefficients whose nominal support remains fully inside the 120-s
    segment contribute to min-max normalization. Boundary-affected coefficients
    are filled with a common mid-gray value (128) before bilinear resizing.
    Rows with no fully interior coefficient region are entirely neutral.
    """
    x = np.asarray(segment_1d, dtype=np.float64)
    coeff, _ = pywt.cwt(
        x,
        BAND_SCALES,
        WAVELET,
        sampling_period=SAMPLING_PERIOD,
        method="conv",
    )
    power = (np.abs(coeff) ** 2).astype(np.float32)
    n_time = power.shape[1]
    valid = np.ones_like(power, dtype=bool)

    for row_idx, (left_n, right_n) in enumerate(
        zip(boundary_left_samples, boundary_right_samples)
    ):
        left_n = int(min(max(left_n, 0), n_time))
        right_n = int(min(max(right_n, 0), n_time))
        if left_n + right_n >= n_time:
            valid[row_idx, :] = False
            continue
        if left_n > 0:
            valid[row_idx, :left_n] = False
        if right_n > 0:
            valid[row_idx, n_time-right_n:] = False

    finite_valid = valid & np.isfinite(power)
    vals = power[finite_valid]

    scaled_float = np.full(
        power.shape,
        BOUNDARY_NEUTRAL_UINT8 / 255.0,
        dtype=np.float32,
    )

    if vals.size:
        lo = float(np.min(vals))
        hi = float(np.max(vals))
        if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
            scaled_float[finite_valid] = np.clip(
                (power[finite_valid] - lo) / (hi - lo),
                0.0,
                1.0,
            )
        else:
            scaled_float[finite_valid] = 0.0

    scaled = np.round(scaled_float * 255.0).astype(np.uint8)
    return np.asarray(
        Image.fromarray(scaled, mode="L").resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            resample=Image.Resampling.BILINEAR,
        ),
        dtype=np.uint8,
    )

def build_sensitivity_rgb_cache(variant):
    cfg = SENS_VARIANTS[variant]
    path = cfg["path"]
    marker = Path(str(path) + ".complete.json")
    if not FORCE_REBUILD_SENSITIVITY_SCALOGRAMS and path.exists() and marker.exists():
        print("Reuse sensitivity cache:", variant)
        return

    mmap = np.lib.format.open_memmap(
        path, mode="w+", dtype=cfg["dtype"],
        shape=(len(segment_df), IMAGE_SIZE, IMAGE_SIZE, 3)
    )
    meta_by_record = {
        k: g.sort_values("segment_in_record")
        for k, g in segment_df.groupby("record_name", sort=False)
    }

    for rec_no, row in enumerate(record_meta.sort_values("record_name").itertuples(), start=1):
        filtered = np.load(FILTERED_ROOT / f"{row.record_name}.npz")["filtered"]
        for seg in meta_by_record[row.record_name].itertuples():
            idx, start, stop = int(seg.segment_index), int(seg.start_sample), int(seg.stop_sample)
            for ch in range(3):
                x = filtered[start:stop, ch]
                if variant == "legacy_bilinear_minmax_float16":
                    img = cwt_minmax_float(x)
                elif variant == "legacy_bicubic_uint8":
                    img = cwt_uint8(x, LEGACY_SCALES, Image.Resampling.BICUBIC)
                elif variant == "bandlimited_bilinear_uint8":
                    img = cwt_uint8(x, BAND_SCALES, Image.Resampling.BILINEAR)
                elif variant == "bandlimited_logpower_float16":
                    img = cwt_logpower_float(x)
                elif variant == "bandlimited_boundary_masked_uint8":
                    img = cwt_boundary_masked_uint8(x)
                else:
                    raise ValueError(variant)
                mmap[idx, :, :, ch] = img
        mmap.flush()
        print(f"[{variant} {rec_no:03d}/{len(record_meta)}] {row.record_name}")

    mmap.flush()
    del mmap
    gc.collect()
    with open(marker, "w", encoding="utf-8") as f:
        json.dump({"variant": variant, "segments": len(segment_df)}, f, indent=2)

if RUN_CWT_SENSITIVITY:
    for variant in SENS_VARIANTS:
        build_sensitivity_rgb_cache(variant)


In [ ]:

# ============================================================
# 9. AUTHORITATIVE PREGNANCY-WISE OUTER/INNER SPLITS
# ============================================================
OUT_DIR = PROJECT_ROOT
FOLD_DIR = PRED_DIR / "fold_results"
COMMENT4_DIR = REPEAT_ROOT
for p in [FOLD_DIR, COMMENT4_DIR]:
    p.mkdir(parents=True, exist_ok=True)


# ============================================================
# ============================================================

def validate_partition(train_df, test_df, name):
    train_preg = set(
        train_df["pregnancy_id"]
    )
    test_preg = set(
        test_df["pregnancy_id"]
    )

    overlap = train_preg.intersection(
        test_preg
    )

    if overlap:
        raise RuntimeError(
            f"{name}: pregnancy leakage: {list(overlap)[:5]}"
        )

    if train_df["label_num"].nunique() != 2:
        raise RuntimeError(
            f"{name}: train tek sınıf içeriyor."
        )

    if test_df["label_num"].nunique() != 2:
        raise RuntimeError(
            f"{name}: test/validation tek sınıf içeriyor."
        )

outer_cv = GroupKFold(
    n_splits=N_OUTER_SPLITS
)

y_record = record_df[
    "label_num"
].astype(int).values

groups_record = record_df[
    "pregnancy_id"
].astype(str).values

outer_splits = list(
    outer_cv.split(
        record_df,
        y_record,
        groups_record
    )
)

outer_assignment = np.full(
    len(record_df),
    -1,
    dtype=int
)

split_rows = []

for outer_fold, (
    outer_train_idx,
    outer_test_idx
) in enumerate(
    outer_splits,
    start=1
):
    outer_assignment[
        outer_test_idx
    ] = outer_fold

    train_df = record_df.iloc[
        outer_train_idx
    ]

    test_df = record_df.iloc[
        outer_test_idx
    ]

    validate_partition(
        train_df,
        test_df,
        f"Outer {outer_fold}"
    )

    # Inner split kontrolü
    inner_cv = GroupKFold(
        n_splits=N_INNER_SPLITS
    )

    for inner_fold, (
        inner_train_local,
        inner_val_local
    ) in enumerate(
        inner_cv.split(
            train_df,
            train_df["label_num"].astype(int).values,
            train_df["pregnancy_id"].astype(str).values
        ),
        start=1
    ):
        validate_partition(
            train_df.iloc[inner_train_local],
            train_df.iloc[inner_val_local],
            f"Outer {outer_fold}/Inner {inner_fold}"
        )

    split_rows.append({
        "outer_fold": outer_fold,
        "train_records": len(train_df),
        "test_records": len(test_df),
        "train_pregnancies": train_df["pregnancy_id"].nunique(),
        "test_pregnancies": test_df["pregnancy_id"].nunique(),
        "test_early": int((test_df["label_num"] == 1).sum()),
        "test_later": int((test_df["label_num"] == 0).sum()),
        "pregnancy_overlap": 0,
    })

record_df["outer_fold"] = outer_assignment

split_df = pd.DataFrame(
    split_rows
)

record_df[
    [
        "record_name",
        "pregnancy_id",
        "label_num",
        "outer_fold"
    ]
].to_csv(
    OUT_DIR / "a2_primary_outer_fold_assignments.csv",
    index=False
)

split_df.to_csv(
    TABLE_DIR / "a2_outer_fold_composition.csv",
    index=False
)

display(split_df)


## Primary corrected nested-CV model

The next cells reproduce the manuscript-consistent A2 CNN–SE–Transformer on the newly rebuilt primary legacy scalograms. No outputs from older notebooks are imported. The three-way inner split is enforced before any fusion/calibration/threshold selection.


In [ ]:

# ============================================================
# 10 — Generic A2 scalogram Sequence
# Eski A2 davranışı: stored scalogram / 255.0
# ============================================================

labels_all = (
    segment_df["label_num"]
    .astype(np.float32)
    .to_numpy()
)

class A2ScalogramSequence(
    keras.utils.Sequence
):
    def __init__(
        self,
        representation,
        segment_indices,
        labels=labels_all,
        batch_size=BATCH_SIZE,
        shuffle=False,
    ):
        super().__init__()

        self.representation = representation

        self.segment_indices = np.asarray(
            segment_indices,
            dtype=int
        ).copy()

        self.labels = np.asarray(
            labels,
            dtype=np.float32
        )

        self.batch_size = int(
            batch_size
        )

        self.shuffle = bool(
            shuffle
        )

        self.X = np.load(
            SCALOGRAM_PATHS[
                representation
            ],
            mmap_mode="r"
        )

        self.on_epoch_end()

    def __len__(self):
        return int(
            np.ceil(
                len(self.segment_indices)
                /
                self.batch_size
            )
        )

    def __getitem__(self, batch_index):
        batch_ids = self.segment_indices[
            batch_index * self.batch_size:
            (batch_index + 1) * self.batch_size
        ]

        X_batch = self.X[
            batch_ids
        ].astype(np.float32)

        # Eski A2 generator ile aynı scaling.
        X_batch = X_batch / 255.0

        if X_batch.ndim == 3:
            X_batch = X_batch[..., np.newaxis]

        y_batch = self.labels[
            batch_ids
        ].astype(np.float32)

        return X_batch, y_batch

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(
                self.segment_indices
            )

print("A2ScalogramSequence hazır.")


In [ ]:

# ============================================================
# 11 — A2 CNN–SE–Transformer mimarisi
# ============================================================

@keras.utils.register_keras_serializable(
    package="ICEHG_A2"
)
class AddPositionEmbedding(
    layers.Layer
):
    def build(self, input_shape):
        sequence_length = int(
            input_shape[1]
        )

        embedding_dimension = int(
            input_shape[2]
        )

        self.position_embedding = (
            self.add_weight(
                name="position_embedding",
                shape=(
                    sequence_length,
                    embedding_dimension
                ),
                initializer="random_normal",
                trainable=True
            )
        )

        super().build(
            input_shape
        )

    def call(self, inputs):
        return (
            inputs
            +
            self.position_embedding[
                tf.newaxis,
                ...
            ]
        )


def se_block(
    x,
    reduction=8,
    name_prefix="se"
):
    filters = int(
        x.shape[-1]
    )

    se = layers.GlobalAveragePooling2D(
        name=f"{name_prefix}_gap"
    )(x)

    se = layers.Dense(
        max(
            filters // reduction,
            1
        ),
        activation="relu",
        name=f"{name_prefix}_dense_1"
    )(se)

    se = layers.Dense(
        filters,
        activation="sigmoid",
        name=f"{name_prefix}_dense_2"
    )(se)

    se = layers.Reshape(
        (
            1,
            1,
            filters
        ),
        name=f"{name_prefix}_reshape"
    )(se)

    return layers.Multiply(
        name=f"{name_prefix}_scale"
    )([x, se])


def residual_cnn_block(
    x,
    filters,
    block_index
):
    prefix = (
        f"residual_block_{block_index}"
    )

    shortcut = x

    x = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        kernel_regularizer=regularizers.l2(
            L2_WEIGHT
        ),
        name=f"{prefix}_conv_1"
    )(x)

    x = layers.BatchNormalization(
        name=f"{prefix}_bn_1"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{prefix}_relu_1"
    )(x)

    x = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        kernel_regularizer=regularizers.l2(
            L2_WEIGHT
        ),
        name=f"{prefix}_conv_2"
    )(x)

    x = layers.BatchNormalization(
        name=f"{prefix}_bn_2"
    )(x)

    x = se_block(
        x,
        reduction=8,
        name_prefix=f"{prefix}_se"
    )

    if int(
        shortcut.shape[-1]
    ) != filters:
        shortcut = layers.Conv2D(
            filters,
            kernel_size=1,
            padding="same",
            kernel_regularizer=regularizers.l2(
                L2_WEIGHT
            ),
            name=f"{prefix}_shortcut_conv"
        )(shortcut)

        shortcut = layers.BatchNormalization(
            name=f"{prefix}_shortcut_bn"
        )(shortcut)

    x = layers.Add(
        name=f"{prefix}_add"
    )([x, shortcut])

    x = layers.Activation(
        "relu",
        name=f"{prefix}_relu_out"
    )(x)

    x = layers.MaxPooling2D(
        pool_size=(2, 2),
        name=f"{prefix}_pool"
    )(x)

    return x


def cnn_backbone(inputs):
    x = residual_cnn_block(
        inputs, 32, 1
    )

    x = residual_cnn_block(
        x, 64, 2
    )

    x = residual_cnn_block(
        x, 128, 3
    )

    x = residual_cnn_block(
        x, 256, 4
    )

    # A2: 14×14 -> 7×7
    x = layers.MaxPooling2D(
        pool_size=(2, 2),
        name="backbone_final_pool"
    )(x)

    return x


def transformer_encoder_block(
    x,
    block_index
):
    prefix = (
        f"transformer_block_{block_index}"
    )

    attention_output = (
        layers.MultiHeadAttention(
            num_heads=NUM_HEADS,
            key_dim=EMBED_DIM // NUM_HEADS,
            dropout=DROPOUT,
            name=f"{prefix}_mha"
        )(x, x)
    )

    x = layers.Add(
        name=f"{prefix}_attention_add"
    )([x, attention_output])

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name=f"{prefix}_attention_norm"
    )(x)

    ffn = layers.Dense(
        FF_DIM,
        activation="relu",
        kernel_regularizer=regularizers.l2(
            L2_WEIGHT
        ),
        name=f"{prefix}_ffn_1"
    )(x)

    ffn = layers.Dropout(
        DROPOUT,
        name=f"{prefix}_ffn_dropout"
    )(ffn)

    ffn = layers.Dense(
        EMBED_DIM,
        kernel_regularizer=regularizers.l2(
            L2_WEIGHT
        ),
        name=f"{prefix}_ffn_2"
    )(ffn)

    x = layers.Add(
        name=f"{prefix}_ffn_add"
    )([x, ffn])

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name=f"{prefix}_ffn_norm"
    )(x)

    return x


def build_a2_cnn_transformer(
    representation
):
    channels = (
        3
        if representation == "RGB"
        else 1
    )

    inputs = layers.Input(
        shape=(
            IMAGE_SIZE,
            IMAGE_SIZE,
            channels
        ),
        name=f"{representation}_scalogram"
    )

    x = cnn_backbone(
        inputs
    )

    x = layers.Conv2D(
        EMBED_DIM,
        kernel_size=1,
        padding="same",
        kernel_regularizer=regularizers.l2(
            L2_WEIGHT
        ),
        name="token_projection"
    )(x)

    x = layers.Reshape(
        (49, EMBED_DIM),
        name="cnn_tokens"
    )(x)

    x = AddPositionEmbedding(
        name="position_embedding"
    )(x)

    for block_index in range(
        1,
        NUM_TRANSFORMER_BLOCKS + 1
    ):
        x = transformer_encoder_block(
            x,
            block_index
        )

    x = layers.GlobalAveragePooling1D(
        name="token_gap"
    )(x)

    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(
            L2_WEIGHT
        ),
        name="head_dense_128"
    )(x)

    x = layers.Dropout(
        DROPOUT,
        name="head_dropout_128"
    )(x)

    x = layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(
            L2_WEIGHT
        ),
        name="head_dense_64"
    )(x)

    x = layers.Dropout(
        DROPOUT,
        name="head_dropout_64"
    )(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="probability"
    )(x)

    model = models.Model(
        inputs,
        outputs,
        name=f"A2_CNN_Transformer_{representation}"
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.AUC(
                name="auc"
            )
        ]
    )

    return model


for rep in REPRESENTATIONS:
    K.clear_session()

    m = build_a2_cnn_transformer(
        rep
    )

    print(
        rep,
        "params:",
        f"{m.count_params():,}"
    )

    del m

K.clear_session()
gc.collect()


In [ ]:

# ============================================================
# 12 — Yardımcı fonksiyonlar
# ============================================================

def set_run_seed(run_name):
    run_offset = (
        zlib.crc32(
            run_name.encode("utf-8")
        )
        %
        100000
    )

    run_seed = int(
        RANDOM_STATE
        +
        run_offset
    )

    random.seed(
        run_seed
    )

    np.random.seed(
        run_seed
    )

    tf.random.set_seed(
        run_seed
    )

    return run_seed


def record_indices_to_segment_indices(
    record_indices
):
    selected_records = set(
        record_df.iloc[
            record_indices
        ]["record_name"].astype(str)
    )

    mask = segment_df[
        "record_name"
    ].isin(
        selected_records
    )

    return segment_df.loc[
        mask,
        "segment_index"
    ].astype(int).values


def aggregate_segment_predictions(
    df
):
    return (
        df
        .groupby(
            "record_name",
            as_index=False
        )
        .agg(
            prob_early_raw=(
                "prob_early_raw",
                "mean"
            ),
            label_num=(
                "label_num",
                "first"
            ),
            label_text=(
                "label_text",
                "first"
            ),
            pregnancy_id=(
                "pregnancy_id",
                "first"
            ),
            n_segments=(
                "segment_index",
                "count"
            ),
        )
    )


def calculate_binary_metrics(
    y_true,
    y_prob,
    y_pred
):
    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    ba = (
        (sensitivity + specificity) / 2.0
        if (
            np.isfinite(sensitivity)
            and
            np.isfinite(specificity)
        )
        else np.nan
    )

    auc_value = (
        roc_auc_score(
            y_true,
            y_prob
        )
        if len(np.unique(y_true)) == 2
        else np.nan
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "balanced_accuracy": ba,
        "auc": auc_value,
        "brier": brier_score_loss(
            y_true,
            np.clip(y_prob, 0, 1)
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def select_robust_threshold(
    y_true,
    y_prob
):
    rows = []

    thresholds = np.arange(
        THRESHOLD_MIN,
        THRESHOLD_MAX
        +
        THRESHOLD_STEP / 2,
        THRESHOLD_STEP
    )

    for threshold in thresholds:
        y_pred = (
            np.asarray(y_prob)
            >=
            threshold
        ).astype(int)

        metrics = calculate_binary_metrics(
            y_true,
            y_prob,
            y_pred
        )

        rows.append({
            "threshold": float(
                threshold
            ),
            **metrics,
            "class_balance_gap": abs(
                metrics["sensitivity"]
                -
                metrics["specificity"]
            ),
            "distance_to_half": abs(
                float(threshold)
                -
                0.5
            ),
        })

    threshold_df = pd.DataFrame(
        rows
    )

    best_ba = threshold_df[
        "balanced_accuracy"
    ].max()

    candidates = threshold_df[
        threshold_df[
            "balanced_accuracy"
        ]
        >=
        best_ba
        -
        ROBUST_BA_TOLERANCE
    ].copy()

    selected = (
        candidates
        .sort_values(
            [
                "class_balance_gap",
                "distance_to_half",
                "f1",
                "accuracy"
            ],
            ascending=[
                True,
                True,
                False,
                False
            ]
        )
        .iloc[0]
        .to_dict()
    )

    return selected, threshold_df


def logit_transform(
    probabilities,
    epsilon=1e-6
):
    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=float
        ),
        epsilon,
        1.0 - epsilon
    )

    return np.log(
        probabilities
        /
        (1.0 - probabilities)
    ).reshape(-1, 1)


def fit_positive_platt(
    y_true,
    raw_prob
):
    calibrator = LogisticRegression(
        C=PLATT_C,
        solver="lbfgs",
        max_iter=2000,
        random_state=RANDOM_STATE
    )

    calibrator.fit(
        logit_transform(
            raw_prob
        ),
        np.asarray(
            y_true,
            dtype=int
        )
    )

    slope = float(
        calibrator.coef_[
            0, 0
        ]
    )

    intercept = float(
        calibrator.intercept_[
            0
        ]
    )

    if slope <= 0:
        return (
            None,
            slope,
            intercept
        )

    return (
        calibrator,
        slope,
        intercept
    )


def apply_platt(
    calibrator,
    raw_prob
):
    if calibrator is None:
        return np.asarray(
            raw_prob,
            dtype=float
        )

    return calibrator.predict_proba(
        logit_transform(
            raw_prob
        )
    )[:, 1]


def merge_rep_record_predictions(
    pred_dict
):
    merged = None

    for rep in REPRESENTATIONS:
        temp = pred_dict[
            rep
        ][
            [
                "record_name",
                "pregnancy_id",
                "label_num",
                "prob_early_raw"
            ]
        ].copy()

        temp = temp.rename(
            columns={
                "prob_early_raw":
                f"prob_{rep}"
            }
        )

        if merged is None:
            merged = temp
        else:
            merged = merged.merge(
                temp[
                    [
                        "record_name",
                        f"prob_{rep}"
                    ]
                ],
                on="record_name",
                how="inner"
            )

    return merged


def generate_weight_grid():
    step_units = int(
        round(
            1.0 / WEIGHT_STEP
        )
    )

    min_units = int(
        round(
            MIN_WEIGHT / WEIGHT_STEP
        )
    )

    max_units = int(
        round(
            MAX_WEIGHT / WEIGHT_STEP
        )
    )

    rows = []

    for units in product(
        range(
            min_units,
            max_units + 1
        ),
        repeat=4
    ):
        if sum(units) != step_units:
            continue

        weights = (
            np.asarray(
                units,
                dtype=float
            )
            *
            WEIGHT_STEP
        )

        rows.append({
            "w_RGB": weights[0],
            "w_S1": weights[1],
            "w_S2": weights[2],
            "w_S3": weights[3],
            "weight_l2_from_equal": float(
                np.sum(
                    (weights - 0.25) ** 2
                )
            ),
        })

    return (
        pd.DataFrame(rows)
        .drop_duplicates()
        .reset_index(drop=True)
    )

WEIGHT_GRID = generate_weight_grid()

print(
    "Weight candidates:",
    len(WEIGHT_GRID)
)


In [ ]:

# ============================================================
# 13 — Tek representation eğitim/prediction görevi + resume
# ============================================================

def train_predict_representation(
    representation,
    train_segment_indices,
    validation_segment_indices,
    test_segment_indices,
    run_name,
    save_final_model=False,
):
    rep_pred_dir = (
        PRED_DIR / representation
    )

    rep_model_dir = (
        MODEL_DIR / representation
    )

    rep_history_dir = (
        OUT_DIR / "histories" / representation
    )

    for p in [
        rep_pred_dir,
        rep_model_dir,
        rep_history_dir
    ]:
        p.mkdir(
            parents=True,
            exist_ok=True
        )

    prediction_path = (
        rep_pred_dir
        /
        f"{run_name}_segment_predictions.csv"
    )

    history_path = (
        rep_history_dir
        /
        f"{run_name}_history.csv"
    )

    model_path = (
        rep_model_dir
        /
        f"{run_name}_best.keras"
    )

    if (
        not FORCE_RETRAIN
        and prediction_path.exists()
        and (
            (not save_final_model)
            or model_path.exists()
        )
    ):
        print(
            "Resume →",
            prediction_path
        )

        return pd.read_csv(
            prediction_path
        )

    run_seed = set_run_seed(
        f"{representation}_{run_name}"
    )

    print(
        f"\n{representation} | {run_name} | seed={run_seed}"
    )

    print(
        "Train/Val/Test segments:",
        len(train_segment_indices),
        len(validation_segment_indices),
        len(test_segment_indices)
    )

    K.clear_session()
    gc.collect()

    model = build_a2_cnn_transformer(
        representation
    )

    train_gen = A2ScalogramSequence(
        representation,
        train_segment_indices,
        labels_all,
        BATCH_SIZE,
        shuffle=True,
    )

    val_gen = A2ScalogramSequence(
        representation,
        validation_segment_indices,
        labels_all,
        BATCH_SIZE,
        shuffle=False,
    )

    test_gen = A2ScalogramSequence(
        representation,
        test_segment_indices,
        labels_all,
        BATCH_SIZE,
        shuffle=False,
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),

        keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1
        ),

        keras.callbacks.TerminateOnNaN()
    ]

    if save_final_model:
        callbacks.append(
            keras.callbacks.ModelCheckpoint(
                filepath=model_path,
                monitor="val_auc",
                mode="max",
                save_best_only=True,
                save_weights_only=False,
                verbose=1
            )
        )

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )

    pd.DataFrame(
        history.history
    ).to_csv(
        history_path,
        index=False
    )

    probabilities = model.predict(
        test_gen,
        verbose=1
    ).reshape(-1)

    metadata_lookup = (
        segment_df
        .set_index(
            "segment_index"
        )
        .loc[
            np.asarray(
                test_segment_indices,
                dtype=int
            )
        ]
        .reset_index()
    )

    pred = pd.DataFrame({
        "segment_index": np.asarray(
            test_segment_indices,
            dtype=int
        ),
        "record_name": metadata_lookup[
            "record_name"
        ].astype(str).values,
        "pregnancy_id": metadata_lookup[
            "pregnancy_id"
        ].astype(str).values,
        "label_text": metadata_lookup[
            "label_text"
        ].astype(str).values,
        "label_num": metadata_lookup[
            "label_num"
        ].astype(int).values,
        "prob_early_raw": probabilities.astype(
            float
        ),
    })

    pred.to_csv(
        prediction_path,
        index=False
    )

    del model
    del train_gen
    del val_gen
    del test_gen
    del history

    K.clear_session()
    gc.collect()

    return pred


In [ ]:

# ============================================================
# 14 — A2 corrected ensemble seçimi
# Yalnız INNER OOF kullanılır.
# ============================================================

def select_a2_corrected_configuration(
    inner_merged
):
    y_inner = inner_merged[
        "label_num"
    ].astype(int).to_numpy()

    prob_matrix = inner_merged[
        [
            "prob_RGB",
            "prob_S1",
            "prob_S2",
            "prob_S3"
        ]
    ].astype(float).to_numpy()

    raw_rows = []

    for weight_row in WEIGHT_GRID.itertuples(
        index=False
    ):
        weights = np.asarray(
            [
                weight_row.w_RGB,
                weight_row.w_S1,
                weight_row.w_S2,
                weight_row.w_S3,
            ],
            dtype=float
        )

        raw_prob = (
            prob_matrix
            @
            weights
        )

        threshold_row, _ = (
            select_robust_threshold(
                y_inner,
                raw_prob
            )
        )

        raw_rows.append({
            "w_RGB": weights[0],
            "w_S1": weights[1],
            "w_S2": weights[2],
            "w_S3": weights[3],
            "weight_l2_from_equal": (
                weight_row.weight_l2_from_equal
            ),
            "raw_threshold": (
                threshold_row["threshold"]
            ),
            "raw_balanced_accuracy": (
                threshold_row[
                    "balanced_accuracy"
                ]
            ),
            "raw_auc": threshold_row[
                "auc"
            ],
            "raw_class_balance_gap": (
                threshold_row[
                    "class_balance_gap"
                ]
            ),
        })

    raw_df = pd.DataFrame(
        raw_rows
    )

    shortlist = (
        raw_df
        .sort_values(
            [
                "raw_balanced_accuracy",
                "raw_auc",
                "raw_class_balance_gap",
                "weight_l2_from_equal"
            ],
            ascending=[
                False,
                False,
                True,
                True
            ]
        )
        .head(
            TOP_WEIGHT_CANDIDATES
        )
        .copy()
    )

    candidate_rows = []
    calibrators = {}
    threshold_tables = {}

    for _, candidate in shortlist.iterrows():
        weights = candidate[
            [
                "w_RGB",
                "w_S1",
                "w_S2",
                "w_S3"
            ]
        ].to_numpy(
            dtype=float
        )

        raw_prob = (
            prob_matrix
            @
            weights
        )

        calibrator, slope, intercept = (
            fit_positive_platt(
                y_inner,
                raw_prob
            )
        )

        # A2 corrected logic:
        # negative/zero slope calibration kabul edilmez.
        if calibrator is None:
            continue

        calibrated = apply_platt(
            calibrator,
            raw_prob
        )

        selected_threshold_row, threshold_df = (
            select_robust_threshold(
                y_inner,
                calibrated
            )
        )

        key = len(
            candidate_rows
        )

        calibrators[
            key
        ] = calibrator

        threshold_tables[
            key
        ] = threshold_df

        candidate_rows.append({
            "candidate_key": key,
            "w_RGB": weights[0],
            "w_S1": weights[1],
            "w_S2": weights[2],
            "w_S3": weights[3],
            "weight_l2_from_equal": (
                candidate[
                    "weight_l2_from_equal"
                ]
            ),
            "platt_slope": slope,
            "platt_intercept": intercept,
            "threshold": (
                selected_threshold_row[
                    "threshold"
                ]
            ),
            "inner_accuracy": (
                selected_threshold_row[
                    "accuracy"
                ]
            ),
            "inner_precision": (
                selected_threshold_row[
                    "precision"
                ]
            ),
            "inner_sensitivity": (
                selected_threshold_row[
                    "sensitivity"
                ]
            ),
            "inner_specificity": (
                selected_threshold_row[
                    "specificity"
                ]
            ),
            "inner_f1": (
                selected_threshold_row[
                    "f1"
                ]
            ),
            "inner_balanced_accuracy": (
                selected_threshold_row[
                    "balanced_accuracy"
                ]
            ),
            "inner_auc": (
                selected_threshold_row[
                    "auc"
                ]
            ),
            "inner_class_balance_gap": (
                selected_threshold_row[
                    "class_balance_gap"
                ]
            ),
            "threshold_distance_to_half": (
                selected_threshold_row[
                    "distance_to_half"
                ]
            ),
        })

    candidate_df = pd.DataFrame(
        candidate_rows
    )

    if candidate_df.empty:
        raise RuntimeError(
            "Pozitif slope Platt candidate bulunamadı."
        )

    selected = (
        candidate_df
        .sort_values(
            [
                "inner_balanced_accuracy",
                "inner_auc",
                "inner_class_balance_gap",
                "weight_l2_from_equal",
                "threshold_distance_to_half"
            ],
            ascending=[
                False,
                False,
                True,
                True,
                True
            ]
        )
        .iloc[0]
        .copy()
    )

    key = int(
        selected[
            "candidate_key"
        ]
    )

    return (
        selected,
        calibrators[key],
        raw_df,
        candidate_df,
        threshold_tables[key]
    )


In [ ]:

# ============================================================
# 15 — Three-way inner split helper
# Outer-development -> inner fit / early-stop / untouched evaluation
# ============================================================

def _has_both_classes(df):
    return df["label_num"].nunique() == 2


def validate_threeway_partition(
    fit_df,
    earlystop_df,
    eval_df,
    name
):
    parts = {
        "fit": fit_df,
        "earlystop": earlystop_df,
        "evaluation": eval_df,
    }

    # Every partition must contain both classes.
    for part_name, part_df in parts.items():
        if not _has_both_classes(part_df):
            raise RuntimeError(
                f"{name}: {part_name} partition does not contain both classes."
            )

    # Pregnancy sets must be pairwise disjoint.
    preg_sets = {
        k: set(v["pregnancy_id"].astype(str))
        for k, v in parts.items()
    }

    pairs = [
        ("fit", "earlystop"),
        ("fit", "evaluation"),
        ("earlystop", "evaluation"),
    ]

    for a, b in pairs:
        overlap = preg_sets[a].intersection(preg_sets[b])
        if overlap:
            raise RuntimeError(
                f"{name}: pregnancy leakage between {a} and {b}: "
                f"{list(overlap)[:5]}"
            )


def make_inner_fit_earlystop_eval_split(
    outer_train_record_idx,
    inner_dev_local,
    inner_eval_local,
    outer_fold,
    inner_fold,
):
    """
    The outer-training records are already split by the 3-fold inner GroupKFold
    into inner-development and untouched inner-evaluation records.

    This function splits inner-development again, pregnancy-wise, into:
      - model-fit records
      - early-stopping records

    Selection among valid GroupKFold sub-splits is deterministic and depends
    only on the inner-fold number, never on model performance.
    """
    outer_train_record_idx = np.asarray(outer_train_record_idx, dtype=int)
    inner_dev_local = np.asarray(inner_dev_local, dtype=int)
    inner_eval_local = np.asarray(inner_eval_local, dtype=int)

    inner_dev_record_idx = outer_train_record_idx[inner_dev_local]
    inner_eval_record_idx = outer_train_record_idx[inner_eval_local]

    inner_dev_df = record_df.iloc[inner_dev_record_idx].copy()
    inner_eval_df = record_df.iloc[inner_eval_record_idx].copy()

    es_splitter = GroupKFold(n_splits=N_INNER_SPLITS)

    candidate_splits = list(
        es_splitter.split(
            inner_dev_df,
            inner_dev_df["label_num"].astype(int).values,
            inner_dev_df["pregnancy_id"].astype(str).values,
        )
    )

    valid_candidates = []

    for candidate_id, (fit_rel, earlystop_rel) in enumerate(
        candidate_splits,
        start=1,
    ):
        fit_record_idx = inner_dev_record_idx[np.asarray(fit_rel, dtype=int)]
        earlystop_record_idx = inner_dev_record_idx[
            np.asarray(earlystop_rel, dtype=int)
        ]

        fit_df = record_df.iloc[fit_record_idx].copy()
        earlystop_df = record_df.iloc[earlystop_record_idx].copy()

        try:
            validate_threeway_partition(
                fit_df,
                earlystop_df,
                inner_eval_df,
                (
                    f"Outer {outer_fold}/Inner {inner_fold}/"
                    f"candidate {candidate_id}"
                ),
            )
        except RuntimeError:
            continue

        valid_candidates.append(
            (
                candidate_id,
                fit_record_idx,
                earlystop_record_idx,
                inner_eval_record_idx,
            )
        )

    if not valid_candidates:
        raise RuntimeError(
            f"Outer {outer_fold}/Inner {inner_fold}: "
            "no valid pregnancy-disjoint fit/early-stop/evaluation split."
        )

    # Deterministic rotation; there is no performance-based split selection.
    selected = valid_candidates[
        (inner_fold - 1) % len(valid_candidates)
    ]

    return selected


def final_outer_fit_earlystop_split(
    outer_train_record_idx,
    outer_fold,
):
    """
    Deterministic pregnancy-wise fit/early-stop split for the final model of an
    outer fold. The outer test set is not involved.
    """
    outer_train_record_idx = np.asarray(
        outer_train_record_idx,
        dtype=int,
    )

    outer_train_df = record_df.iloc[
        outer_train_record_idx
    ].copy()

    splitter = GroupKFold(
        n_splits=N_INNER_SPLITS
    )

    candidates = list(
        splitter.split(
            outer_train_df,
            outer_train_df["label_num"].astype(int).values,
            outer_train_df["pregnancy_id"].astype(str).values,
        )
    )

    valid = []
    for split_id, (fit_local, earlystop_local) in enumerate(
        candidates,
        start=1,
    ):
        fit_idx = outer_train_record_idx[
            np.asarray(fit_local, dtype=int)
        ]
        es_idx = outer_train_record_idx[
            np.asarray(earlystop_local, dtype=int)
        ]

        fit_df = record_df.iloc[fit_idx]
        es_df = record_df.iloc[es_idx]

        if (
            _has_both_classes(fit_df)
            and _has_both_classes(es_df)
            and not (
                set(fit_df["pregnancy_id"])
                & set(es_df["pregnancy_id"])
            )
        ):
            valid.append((split_id, fit_idx, es_idx))

    if not valid:
        raise RuntimeError(
            f"Outer {outer_fold}: no valid final fit/early-stop split."
        )

    # Fixed deterministic choice, independent of outer-test performance.
    return valid[0]


In [ ]:

# ============================================================
# 16 — CORRECTED 5×3 pregnancy-wise nested training
# ============================================================

all_outer_parts = []
all_config_rows = []
all_fold_metric_rows = []
all_inner_split_audit_rows = []

for outer_fold, (
    outer_train_record_idx,
    outer_test_record_idx
) in enumerate(
    outer_splits,
    start=1
):
    print("\n" + "#" * 100)
    print(f"CORRECTED OUTER FOLD {outer_fold}/5")
    print("#" * 100)

    fold_dir = FOLD_DIR / f"outer_{outer_fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    final_fold_path = fold_dir / "outer_record_predictions.csv"
    config_path = fold_dir / "selected_configuration.csv"
    metric_path = fold_dir / "outer_metrics.csv"
    split_audit_path = fold_dir / "inner_threeway_split_audit.csv"

    if (
        not FORCE_RETRAIN
        and final_fold_path.exists()
        and config_path.exists()
        and metric_path.exists()
        and split_audit_path.exists()
    ):
        print("Completed corrected outer fold found -> loading.")

        all_outer_parts.append(pd.read_csv(final_fold_path))
        all_config_rows.append(
            pd.read_csv(config_path).iloc[0].to_dict()
        )
        all_fold_metric_rows.append(
            pd.read_csv(metric_path).iloc[0].to_dict()
        )
        all_inner_split_audit_rows.extend(
            pd.read_csv(split_audit_path).to_dict("records")
        )
        continue

    outer_train_record_idx = np.asarray(
        outer_train_record_idx,
        dtype=int,
    )
    outer_test_record_idx = np.asarray(
        outer_test_record_idx,
        dtype=int,
    )

    outer_train_df = record_df.iloc[
        outer_train_record_idx
    ].copy()

    outer_test_df = record_df.iloc[
        outer_test_record_idx
    ].copy()

    validate_partition(
        outer_train_df,
        outer_test_df,
        f"Corrected outer {outer_fold}",
    )

    # --------------------------------------------------------
    # INNER OOF: untouched inner evaluation predictions only
    # --------------------------------------------------------
    inner_cv = GroupKFold(
        n_splits=N_INNER_SPLITS
    )

    inner_splits = list(
        inner_cv.split(
            outer_train_df,
            outer_train_df["label_num"].astype(int).values,
            outer_train_df["pregnancy_id"].astype(str).values,
        )
    )

    inner_rep_record_parts = {
        rep: []
        for rep in REPRESENTATIONS
    }

    fold_split_audit_rows = []

    for inner_fold, (
        inner_dev_local,
        inner_eval_local
    ) in enumerate(
        inner_splits,
        start=1,
    ):
        print("\n" + "-" * 80)
        print(
            f"INNER FOLD {inner_fold}/3: "
            "fit / early-stop / untouched evaluation"
        )
        print("-" * 80)

        (
            es_candidate_id,
            inner_fit_record_idx,
            inner_earlystop_record_idx,
            inner_eval_record_idx,
        ) = make_inner_fit_earlystop_eval_split(
            outer_train_record_idx=outer_train_record_idx,
            inner_dev_local=inner_dev_local,
            inner_eval_local=inner_eval_local,
            outer_fold=outer_fold,
            inner_fold=inner_fold,
        )

        fit_df = record_df.iloc[
            inner_fit_record_idx
        ].copy()

        es_df = record_df.iloc[
            inner_earlystop_record_idx
        ].copy()

        eval_df = record_df.iloc[
            inner_eval_record_idx
        ].copy()

        validate_threeway_partition(
            fit_df,
            es_df,
            eval_df,
            f"Outer {outer_fold}/Inner {inner_fold}",
        )

        fold_split_audit_rows.append(
            {
                "outer_fold": outer_fold,
                "inner_fold": inner_fold,
                "earlystop_candidate_id": es_candidate_id,
                "fit_records": len(fit_df),
                "fit_pregnancies": fit_df["pregnancy_id"].nunique(),
                "fit_early": int((fit_df["label_num"] == 1).sum()),
                "fit_later": int((fit_df["label_num"] == 0).sum()),
                "earlystop_records": len(es_df),
                "earlystop_pregnancies": es_df[
                    "pregnancy_id"
                ].nunique(),
                "earlystop_early": int(
                    (es_df["label_num"] == 1).sum()
                ),
                "earlystop_later": int(
                    (es_df["label_num"] == 0).sum()
                ),
                "evaluation_records": len(eval_df),
                "evaluation_pregnancies": eval_df[
                    "pregnancy_id"
                ].nunique(),
                "evaluation_early": int(
                    (eval_df["label_num"] == 1).sum()
                ),
                "evaluation_later": int(
                    (eval_df["label_num"] == 0).sum()
                ),
                "pregnancy_overlap": 0,
            }
        )

        inner_fit_seg = record_indices_to_segment_indices(
            inner_fit_record_idx
        )

        inner_earlystop_seg = record_indices_to_segment_indices(
            inner_earlystop_record_idx
        )

        inner_eval_seg = record_indices_to_segment_indices(
            inner_eval_record_idx
        )

        for rep in REPRESENTATIONS:
            run_name = (
                f"outer{outer_fold}_inner{inner_fold}_"
                "fit_es_eval"
            )

            seg_pred = train_predict_representation(
                representation=rep,
                train_segment_indices=inner_fit_seg,
                validation_segment_indices=inner_earlystop_seg,
                test_segment_indices=inner_eval_seg,
                run_name=run_name,
                save_final_model=False,
            )

            rec_pred = aggregate_segment_predictions(
                seg_pred
            )

            rec_pred["inner_fold"] = inner_fold

            inner_rep_record_parts[rep].append(
                rec_pred
            )

    split_audit_df = pd.DataFrame(
        fold_split_audit_rows
    )
    split_audit_df.to_csv(
        split_audit_path,
        index=False,
    )
    all_inner_split_audit_rows.extend(
        fold_split_audit_rows
    )

    inner_pred_dict = {
        rep: pd.concat(
            inner_rep_record_parts[rep],
            ignore_index=True,
        )
        for rep in REPRESENTATIONS
    }

    # Every outer-development record must appear exactly once
    # as an UNTOUCHED inner-evaluation prediction.
    expected_inner_records = set(
        outer_train_df["record_name"].astype(str)
    )

    for rep in REPRESENTATIONS:
        obtained = set(
            inner_pred_dict[rep][
                "record_name"
            ].astype(str)
        )

        if obtained != expected_inner_records:
            missing = sorted(
                expected_inner_records - obtained
            )
            extra = sorted(
                obtained - expected_inner_records
            )
            raise RuntimeError(
                f"{rep}/outer{outer_fold}: "
                f"corrected inner OOF coverage error. "
                f"missing={missing[:5]}, extra={extra[:5]}"
            )

        if inner_pred_dict[rep][
            "record_name"
        ].duplicated().any():
            raise RuntimeError(
                f"{rep}/outer{outer_fold}: "
                "duplicate corrected inner OOF records."
            )

        inner_pred_dict[rep].to_csv(
            fold_dir / f"inner_oof_{rep}.csv",
            index=False,
        )

    inner_merged = merge_rep_record_predictions(
        inner_pred_dict
    )

    if len(inner_merged) != len(outer_train_df):
        raise RuntimeError(
            f"Outer {outer_fold}: corrected inner OOF merged "
            f"records={len(inner_merged)}, expected={len(outer_train_df)}"
        )

    inner_merged.to_csv(
        fold_dir / "inner_oof_all_representations.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Fusion/calibration/threshold selection:
    # ONLY untouched inner-evaluation OOF predictions
    # --------------------------------------------------------
    (
        selected,
        selected_calibrator,
        raw_weight_search,
        calibrated_candidates,
        threshold_search,
    ) = select_a2_corrected_configuration(
        inner_merged
    )

    selected_weights = selected[
        ["w_RGB", "w_S1", "w_S2", "w_S3"]
    ].to_numpy(dtype=float)

    selected_threshold = float(
        selected["threshold"]
    )

    print("\nSelected corrected weights:", selected_weights)
    print(
        "Platt slope:",
        float(selected["platt_slope"]),
    )
    print("Threshold:", selected_threshold)

    raw_weight_search.to_csv(
        fold_dir / "weight_search_raw.csv",
        index=False,
    )
    calibrated_candidates.to_csv(
        fold_dir / "weight_search_calibrated_candidates.csv",
        index=False,
    )
    threshold_search.to_csv(
        fold_dir / "threshold_search.csv",
        index=False,
    )

    # --------------------------------------------------------
    # FINAL OUTER MODEL:
    # fit / early-stop from outer-development only;
    # outer test remains untouched.
    # --------------------------------------------------------
    (
        final_es_split_id,
        final_train_record_idx,
        final_val_record_idx,
    ) = final_outer_fit_earlystop_split(
        outer_train_record_idx,
        outer_fold,
    )

    final_train_df = record_df.iloc[
        final_train_record_idx
    ].copy()

    final_val_df = record_df.iloc[
        final_val_record_idx
    ].copy()

    validate_partition(
        final_train_df,
        final_val_df,
        f"Outer {outer_fold} final fit/early-stop",
    )

    # Outer test must be pregnancy-disjoint from both.
    validate_partition(
        final_train_df,
        outer_test_df,
        f"Outer {outer_fold} final fit/test",
    )
    validate_partition(
        final_val_df,
        outer_test_df,
        f"Outer {outer_fold} final early-stop/test",
    )

    final_train_seg = record_indices_to_segment_indices(
        final_train_record_idx
    )
    final_val_seg = record_indices_to_segment_indices(
        final_val_record_idx
    )
    outer_test_seg = record_indices_to_segment_indices(
        outer_test_record_idx
    )

    outer_rep_dict = {}

    for rep in REPRESENTATIONS:
        run_name = f"outer{outer_fold}_final_corrected"

        seg_pred = train_predict_representation(
            representation=rep,
            train_segment_indices=final_train_seg,
            validation_segment_indices=final_val_seg,
            test_segment_indices=outer_test_seg,
            run_name=run_name,
            save_final_model=True,
        )

        rec_pred = aggregate_segment_predictions(
            seg_pred
        )

        outer_rep_dict[rep] = rec_pred

        rec_pred.to_csv(
            fold_dir
            / f"outer_test_{rep}_record_predictions.csv",
            index=False,
        )

    outer_merged = merge_rep_record_predictions(
        outer_rep_dict
    )

    outer_matrix = outer_merged[
        ["prob_RGB", "prob_S1", "prob_S2", "prob_S3"]
    ].astype(float).to_numpy()

    raw_ensemble = outer_matrix @ selected_weights

    calibrated_ensemble = apply_platt(
        selected_calibrator,
        raw_ensemble,
    )

    y_outer = outer_merged[
        "label_num"
    ].astype(int).to_numpy()

    y_pred = (
        calibrated_ensemble >= selected_threshold
    ).astype(int)

    outer_result = outer_merged.copy()
    outer_result["prob_ensemble_raw"] = raw_ensemble
    outer_result[
        "prob_ensemble_calibrated"
    ] = calibrated_ensemble
    outer_result["pred_label_num"] = y_pred
    outer_result["outer_fold"] = outer_fold
    outer_result["threshold"] = selected_threshold
    outer_result["platt_slope"] = float(
        selected["platt_slope"]
    )
    outer_result["platt_intercept"] = float(
        selected["platt_intercept"]
    )

    for rep, weight in zip(
        REPRESENTATIONS,
        selected_weights,
    ):
        outer_result[f"w_{rep}"] = float(weight)

    outer_metrics = calculate_binary_metrics(
        y_outer,
        calibrated_ensemble,
        y_pred,
    )

    outer_metrics["raw_auc"] = roc_auc_score(
        y_outer,
        raw_ensemble,
    )
    outer_metrics["calibrated_auc"] = roc_auc_score(
        y_outer,
        calibrated_ensemble,
    )

    config_row = {
        "outer_fold": outer_fold,
        "w_RGB": selected_weights[0],
        "w_S1": selected_weights[1],
        "w_S2": selected_weights[2],
        "w_S3": selected_weights[3],
        "threshold": selected_threshold,
        "platt_slope": float(
            selected["platt_slope"]
        ),
        "platt_intercept": float(
            selected["platt_intercept"]
        ),
        "inner_balanced_accuracy": float(
            selected["inner_balanced_accuracy"]
        ),
        "inner_auc": float(
            selected["inner_auc"]
        ),
        "n_inner_records": len(inner_merged),
        "n_inner_pregnancies": inner_merged[
            "pregnancy_id"
        ].nunique(),
        "inner_oof_protocol": (
            "pregnancy-disjoint fit/early-stop/"
            "untouched-evaluation"
        ),
        "final_earlystop_split_id": int(
            final_es_split_id
        ),
    }

    fold_metric_row = {
        "outer_fold": outer_fold,
        **outer_metrics,
        "threshold": selected_threshold,
        "platt_slope": config_row["platt_slope"],
        "platt_intercept": config_row[
            "platt_intercept"
        ],
        "n_test_records": len(outer_result),
        "n_test_pregnancies": outer_result[
            "pregnancy_id"
        ].nunique(),
    }

    outer_result.to_csv(
        final_fold_path,
        index=False,
    )
    pd.DataFrame([config_row]).to_csv(
        config_path,
        index=False,
    )
    pd.DataFrame([fold_metric_row]).to_csv(
        metric_path,
        index=False,
    )

    all_outer_parts.append(outer_result)
    all_config_rows.append(config_row)
    all_fold_metric_rows.append(fold_metric_row)

    print(
        "\nCORRECTED OUTER raw/cal AUC:",
        round(outer_metrics["raw_auc"], 4),
        "/",
        round(outer_metrics["calibrated_auc"], 4),
    )

    del selected_calibrator
    gc.collect()

master_oof = (
    pd.concat(
        all_outer_parts,
        ignore_index=True,
    )
    .sort_values(
        ["outer_fold", "record_name"]
    )
    .reset_index(drop=True)
)

selected_configs_df = pd.DataFrame(
    all_config_rows
).sort_values("outer_fold")

fold_metrics_df = pd.DataFrame(
    all_fold_metric_rows
).sort_values("outer_fold")

inner_split_audit_df = pd.DataFrame(
    all_inner_split_audit_rows
).sort_values(
    ["outer_fold", "inner_fold"]
)

if len(master_oof) != 126:
    raise RuntimeError(
        f"Corrected master OOF must contain 126 records; "
        f"found {len(master_oof)}."
    )

if master_oof["record_name"].duplicated().any():
    raise RuntimeError(
        "Corrected master OOF contains duplicate records."
    )

if master_oof["pregnancy_id"].nunique() != 91:
    raise RuntimeError(
        "Corrected master OOF must contain 91 pregnancies."
    )

master_oof.to_csv(
    PRED_DIR / "master_oof_predictions.csv",
    index=False,
)
selected_configs_df.to_csv(
    TABLE_DIR / "selected_fold_configurations.csv",
    index=False,
)
fold_metrics_df.to_csv(
    TABLE_DIR / "fold_metrics.csv",
    index=False,
)
inner_split_audit_df.to_csv(
    TABLE_DIR / "inner_threeway_split_audit_all_folds.csv",
    index=False,
)

print("\nCORRECTED NESTED-CV BASELINE COMPLETED.")
display(selected_configs_df)
display(fold_metrics_df)
display(inner_split_audit_df)


In [ ]:

# ============================================================
# 17 — Corrected primary mean outer-fold ROC-AUC
#           + 20,000-replicate fold-stratified pregnancy bootstrap
# ============================================================

def representation_and_fusion_scores(df):
    out = df.copy()

    out["prob_equal_RGB_S1_S2"] = (
        out["prob_RGB"].astype(float)
        + out["prob_S1"].astype(float)
        + out["prob_S2"].astype(float)
    ) / 3.0

    out["prob_equal_all4"] = (
        out["prob_RGB"].astype(float)
        + out["prob_S1"].astype(float)
        + out["prob_S2"].astype(float)
        + out["prob_S3"].astype(float)
    ) / 4.0

    return out


master_oof = representation_and_fusion_scores(
    master_oof
)

score_map = {
    "RGB": "prob_RGB",
    "S1": "prob_S1",
    "S2": "prob_S2",
    "S3": "prob_S3",
    "Equal-weight RGB+S1+S2": "prob_equal_RGB_S1_S2",
    "Equal-weight RGB+S1+S2+S3": "prob_equal_all4",
    "Corrected optimized weighted ensemble": "prob_ensemble_raw",
}

fold_auc_rows = []

for model_name, score_col in score_map.items():
    for fold, g in sorted(
        master_oof.groupby("outer_fold")
    ):
        y = g["label_num"].astype(int).to_numpy()
        s = g[score_col].astype(float).to_numpy()

        fold_auc_rows.append(
            {
                "model": model_name,
                "outer_fold": int(fold),
                "n_records": int(len(g)),
                "n_pregnancies": int(
                    g["pregnancy_id"].nunique()
                ),
                "roc_auc": float(
                    roc_auc_score(y, s)
                ),
            }
        )

fold_auc_corrected_df = pd.DataFrame(
    fold_auc_rows
)

auc_summary_rows = []

for model_name, g in fold_auc_corrected_df.groupby(
    "model",
    sort=False,
):
    # Primary: arithmetic mean of the five fold-specific AUCs.
    macro_auc = float(
        g["roc_auc"].mean()
    )
    sd_auc = float(
        g["roc_auc"].std(ddof=1)
    )

    # Secondary record-count-weighted mean.
    weighted_auc = float(
        np.average(
            g["roc_auc"],
            weights=g["n_records"],
        )
    )

    score_col = score_map[model_name]
    pooled_auc = float(
        roc_auc_score(
            master_oof["label_num"].astype(int),
            master_oof[score_col].astype(float),
        )
    )

    auc_summary_rows.append(
        {
            "model": model_name,
            "mean_outer_fold_auc_primary": macro_auc,
            "outer_fold_auc_sd": sd_auc,
            "record_count_weighted_mean_auc_secondary": weighted_auc,
            "pooled_auc_secondary": pooled_auc,
        }
    )

auc_summary_corrected_df = pd.DataFrame(
    auc_summary_rows
)


def fold_stratified_pregnancy_bootstrap_macro_auc(
    df,
    score_col,
    n_boot=20000,
    seed=20261019,
):
    rng = np.random.default_rng(seed)

    fold_data = {}

    for fold, g in sorted(
        df.groupby("outer_fold")
    ):
        g = g.reset_index(drop=True).copy()

        pids = g[
            "pregnancy_id"
        ].astype(str).drop_duplicates().to_numpy()

        pid_to_idx = {
            pid: g.index[
                g["pregnancy_id"].astype(str) == pid
            ].to_numpy()
            for pid in pids
        }

        fold_data[int(fold)] = (
            g,
            pids,
            pid_to_idx,
        )

    values = np.empty(
        n_boot,
        dtype=float,
    )

    valid = 0
    attempts = 0
    max_attempts = n_boot * 20

    while (
        valid < n_boot
        and attempts < max_attempts
    ):
        attempts += 1
        fold_aucs = []
        ok = True

        for fold, (
            g,
            pids,
            pid_to_idx,
        ) in fold_data.items():
            sampled_pids = rng.choice(
                pids,
                size=len(pids),
                replace=True,
            )

            idx = np.concatenate(
                [
                    pid_to_idx[pid]
                    for pid in sampled_pids
                ]
            )

            b = g.iloc[idx]
            y = b["label_num"].astype(int).to_numpy()

            if np.unique(y).size < 2:
                ok = False
                break

            fold_aucs.append(
                roc_auc_score(
                    y,
                    b[score_col].astype(float).to_numpy(),
                )
            )

        if not ok or len(fold_aucs) != N_OUTER_SPLITS:
            continue

        values[valid] = float(
            np.mean(fold_aucs)
        )
        valid += 1

    if valid != n_boot:
        raise RuntimeError(
            f"Only {valid}/{n_boot} valid bootstrap "
            f"replicates after {attempts} attempts."
        )

    return {
        "bootstrap_values": values,
        "ci95_low": float(
            np.quantile(values, 0.025)
        ),
        "ci95_high": float(
            np.quantile(values, 0.975)
        ),
        "attempts": int(attempts),
    }


primary_boot = (
    fold_stratified_pregnancy_bootstrap_macro_auc(
        master_oof,
        "prob_ensemble_raw",
        n_boot=N_BOOTSTRAP_PRIMARY_AUC,
        seed=PRIMARY_BOOTSTRAP_SEED,
    )
)

primary_row = (
    auc_summary_corrected_df.loc[
        auc_summary_corrected_df["model"]
        == "Corrected optimized weighted ensemble"
    ]
    .iloc[0]
    .copy()
)

primary_ci_df = pd.DataFrame(
    [
        {
            "model": "Corrected optimized weighted ensemble",
            "mean_outer_fold_auc_primary": float(
                primary_row[
                    "mean_outer_fold_auc_primary"
                ]
            ),
            "outer_fold_auc_sd": float(
                primary_row["outer_fold_auc_sd"]
            ),
            "ci95_low": primary_boot["ci95_low"],
            "ci95_high": primary_boot["ci95_high"],
            "record_count_weighted_mean_auc_secondary": float(
                primary_row[
                    "record_count_weighted_mean_auc_secondary"
                ]
            ),
            "pooled_auc_secondary": float(
                primary_row[
                    "pooled_auc_secondary"
                ]
            ),
            "bootstrap_replicates": int(
                N_BOOTSTRAP_PRIMARY_AUC
            ),
            "bootstrap_seed": int(
                PRIMARY_BOOTSTRAP_SEED
            ),
            "bootstrap_unit": (
                "pregnancy sampled with replacement "
                "separately within outer fold"
            ),
        }
    ]
)

fold_auc_corrected_df.to_csv(
    TABLE_DIR / "corrected_fold_auc.csv",
    index=False,
)

auc_summary_corrected_df.to_csv(
    TABLE_DIR / "corrected_auc_summary.csv",
    index=False,
)

primary_ci_df.to_csv(
    TABLE_DIR
    / "corrected_primary_auc_fold_stratified_bootstrap.csv",
    index=False,
)

np.save(
    TABLE_DIR
    / "corrected_primary_auc_bootstrap_draws.npy",
    primary_boot["bootstrap_values"],
)

print("=" * 92)
print("CORRECTED PRIMARY DISCRIMINATION SUMMARY")
print("=" * 92)
display(fold_auc_corrected_df)
display(auc_summary_corrected_df)
display(primary_ci_df)


In [ ]:

# ============================================================
# 18. THRESHOLD-METRIC BOOTSTRAP + CALIBRATION / ECE
# ============================================================

# Threshold-dependent metrics condition on each record's already selected
# held-out binary decision. They do NOT re-estimate threshold/calibration.

def fixed_decision_metrics(df):
    y = df["label_num"].astype(int).to_numpy()
    pred = df["pred_label_num"].astype(int).to_numpy()
    prob = df["prob_ensemble_calibrated"].astype(float).to_numpy()
    return calculate_binary_metrics(y, prob, pred)

def pregnancy_cluster_bootstrap_fixed_decisions(
    df, n_boot=N_BOOTSTRAP_THRESHOLD, seed=THRESHOLD_BOOTSTRAP_SEED
):
    rng = np.random.default_rng(seed)
    pids = df["pregnancy_id"].astype(str).drop_duplicates().to_numpy()
    pid_to_idx = {
        pid: df.index[df["pregnancy_id"].astype(str) == pid].to_numpy()
        for pid in pids
    }
    metrics = ["accuracy","precision","sensitivity","specificity","f1","balanced_accuracy"]
    store = {m: [] for m in metrics}

    valid = 0
    while valid < n_boot:
        sampled = rng.choice(pids, size=len(pids), replace=True)
        idx = np.concatenate([pid_to_idx[pid] for pid in sampled])
        b = df.loc[idx].copy()
        if b["label_num"].nunique() < 2:
            continue
        res = fixed_decision_metrics(b)
        for m in metrics:
            store[m].append(float(res[m]))
        valid += 1

    rows = []
    observed = fixed_decision_metrics(df)
    for m in metrics:
        vals = np.asarray(store[m], dtype=float)
        rows.append({
            "metric": m,
            "estimate": float(observed[m]),
            "ci95_low": float(np.quantile(vals, 0.025)),
            "ci95_high": float(np.quantile(vals, 0.975)),
            "bootstrap_replicates": int(n_boot),
        })
    return pd.DataFrame(rows)

threshold_ci_df = pregnancy_cluster_bootstrap_fixed_decisions(master_oof)
save_table(threshold_ci_df, "table_09_threshold_metrics_pregnancy_bootstrap")

# Add primary AUC CI from 20k fold-stratified bootstrap.
auc_row = pd.DataFrame([{
    "metric": "mean_outer_fold_raw_auc",
    "estimate": float(primary_ci_df["mean_outer_fold_auc_primary"].iloc[0]),
    "ci95_low": float(primary_ci_df["ci95_low"].iloc[0]),
    "ci95_high": float(primary_ci_df["ci95_high"].iloc[0]),
    "bootstrap_replicates": int(N_BOOTSTRAP_PRIMARY_AUC),
}])
table_09 = pd.concat([threshold_ci_df, auc_row], ignore_index=True)
save_table(table_09, "table_09_pregnancy_wise_bootstrap_CI_final")

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    # quantile bins, robust to probability concentration
    q = np.linspace(0, 1, n_bins + 1)
    edges = np.unique(np.quantile(y_prob, q))
    if len(edges) < 3:
        edges = np.linspace(0, 1, n_bins + 1)
    bins = np.digitize(y_prob, edges[1:-1], right=True)
    ece = 0.0
    rows = []
    for b in range(max(bins)+1):
        mask = bins == b
        if not np.any(mask):
            continue
        conf = float(np.mean(y_prob[mask]))
        obs = float(np.mean(y_true[mask]))
        frac = float(np.mean(mask))
        ece += frac * abs(obs - conf)
        rows.append({"bin": b+1, "n": int(mask.sum()), "mean_predicted": conf, "observed_fraction": obs})
    return float(ece), pd.DataFrame(rows)

y_all = master_oof["label_num"].astype(int).to_numpy()
p_raw = master_oof["prob_ensemble_raw"].astype(float).to_numpy()
p_cal = master_oof["prob_ensemble_calibrated"].astype(float).to_numpy()

raw_ece, raw_rel = expected_calibration_error(y_all, p_raw, 10)
cal_ece, cal_rel = expected_calibration_error(y_all, p_cal, 10)

calibration_summary = pd.DataFrame([
    {
        "score": "Raw weighted ensemble",
        "pooled_auc_secondary": roc_auc_score(y_all, p_raw),
        "brier": brier_score_loss(y_all, p_raw),
        "ece_10_quantile_bins": raw_ece,
    },
    {
        "score": "Fold-specific Platt-calibrated ensemble",
        "pooled_auc_secondary": roc_auc_score(y_all, p_cal),
        "brier": brier_score_loss(y_all, p_cal),
        "ece_10_quantile_bins": cal_ece,
    },
])
save_table(calibration_summary, "table_08_calibration_summary")
save_table(raw_rel, "calibration_reliability_raw")
save_table(cal_rel, "calibration_reliability_calibrated")

print("Table 9")
display(table_09)
print("Calibration")
display(calibration_summary)


In [ ]:

# ============================================================
# 19. PAIRED-PREGNANCY + PAIRED VS SINGLETON SENSITIVITY
# ============================================================

pairing_lookup = dict(
    zip(record_meta["record_name"].astype(str), record_meta["pairing_status"].astype(str))
)
master_oof["pairing_status"] = master_oof["record_name"].astype(str).map(pairing_lookup)

# Matched 35 pregnancies.
paired_ids = preg_comp.loc[
    preg_comp["pairing_status"] == "paired", "pregnancy_id"
].astype(str).tolist()

paired_rows = []
for pid in sorted(paired_ids):
    g = master_oof.loc[master_oof["pregnancy_id"].astype(str) == pid].copy()
    early = g.loc[g["label_num"].astype(int) == 1]
    later = g.loc[g["label_num"].astype(int) == 0]
    if len(early) != 1 or len(later) != 1:
        raise RuntimeError(f"Expected exactly one early and one later record for paired pregnancy {pid}")
    er = early.iloc[0]
    lr = later.iloc[0]
    paired_rows.append({
        "pregnancy_id": pid,
        "early_record": er["record_name"],
        "later_record": lr["record_name"],
        "early_raw_score": float(er["prob_ensemble_raw"]),
        "later_raw_score": float(lr["prob_ensemble_raw"]),
        "raw_difference_early_minus_later": float(er["prob_ensemble_raw"] - lr["prob_ensemble_raw"]),
        "early_calibrated_score": float(er["prob_ensemble_calibrated"]),
        "later_calibrated_score": float(lr["prob_ensemble_calibrated"]),
        "calibrated_difference_early_minus_later": float(er["prob_ensemble_calibrated"] - lr["prob_ensemble_calibrated"]),
        "early_higher_raw": bool(er["prob_ensemble_raw"] > lr["prob_ensemble_raw"]),
    })

paired_df = pd.DataFrame(paired_rows)
assert len(paired_df) == EXPECTED_PAIRED_PREGNANCIES

n_positive = int(paired_df["early_higher_raw"].sum())
n_negative = int(len(paired_df) - n_positive)
binom = binomtest(n_positive, n=len(paired_df), p=0.5, alternative="two-sided")
binom_ci = binom.proportion_ci(confidence_level=0.95, method="exact")

# Pregnancy-pair bootstrap for mean raw difference.
rng = np.random.default_rng(PAIRED_BOOTSTRAP_SEED)
diff = paired_df["raw_difference_early_minus_later"].to_numpy(float)
boot_mean = np.empty(N_BOOTSTRAP_PAIRED, dtype=float)
for b in range(N_BOOTSTRAP_PAIRED):
    boot_mean[b] = float(np.mean(rng.choice(diff, size=len(diff), replace=True)))

# Paired sign-flip permutation test on the mean raw difference.
PERMUTATION_REPLICATES = 100000
perm_rng = np.random.default_rng(PAIRED_BOOTSTRAP_SEED + 1)
observed_mean_diff = float(np.mean(diff))
perm_means = np.empty(PERMUTATION_REPLICATES, dtype=float)
for b in range(PERMUTATION_REPLICATES):
    signs = perm_rng.choice(np.array([-1.0, 1.0]), size=len(diff), replace=True)
    perm_means[b] = float(np.mean(diff * signs))
paired_permutation_p = float(
    (np.sum(np.abs(perm_means) >= abs(observed_mean_diff)) + 1)
    / (PERMUTATION_REPLICATES + 1)
)

paired_summary = pd.DataFrame([{
    "paired_pregnancies": len(paired_df),
    "early_score_higher_n": n_positive,
    "early_score_higher_percent": 100.0 * n_positive / len(paired_df),
    "exact_binomial_ci95_low": float(binom_ci.low),
    "exact_binomial_ci95_high": float(binom_ci.high),
    "reverse_order_n": n_negative,
    "ties_n": int((paired_df["raw_difference_early_minus_later"] == 0).sum()),
    "exact_two_sided_sign_p": float(binom.pvalue),
    "paired_signflip_permutation_replicates": PERMUTATION_REPLICATES,
    "paired_signflip_permutation_p": paired_permutation_p,
    "mean_raw_difference": float(np.mean(diff)),
    "median_raw_difference": float(np.median(diff)),
    "mean_raw_difference_boot_ci95_low": float(np.quantile(boot_mean, 0.025)),
    "mean_raw_difference_boot_ci95_high": float(np.quantile(boot_mean, 0.975)),
    "mean_calibrated_difference": float(paired_df["calibrated_difference_early_minus_later"].mean()),
    "median_calibrated_difference": float(paired_df["calibrated_difference_early_minus_later"].median()),
}])

save_table(paired_df, "paired_pregnancy_score_differences")
save_table(paired_summary, "paired_pregnancy_summary")

def subgroup_performance(df, label):
    y = df["label_num"].astype(int).to_numpy()
    pred = df["pred_label_num"].astype(int).to_numpy()
    prob_cal = df["prob_ensemble_calibrated"].astype(float).to_numpy()
    m = calculate_binary_metrics(y, prob_cal, pred)

    fold_aucs = []
    for fold, g in sorted(df.groupby("outer_fold")):
        if g["label_num"].nunique() == 2:
            fold_aucs.append(
                roc_auc_score(g["label_num"].astype(int), g["prob_ensemble_raw"].astype(float))
            )
    return {
        "Subgroup": label,
        "Pregnancies": int(df["pregnancy_id"].nunique()),
        "Records": int(len(df)),
        "Early": int((df["label_num"] == 1).sum()),
        "Later": int((df["label_num"] == 0).sum()),
        "Accuracy": m["accuracy"],
        "Sensitivity": m["sensitivity"],
        "Specificity": m["specificity"],
        "F1": m["f1"],
        "Balanced accuracy": m["balanced_accuracy"],
        "Mean outer-fold raw ROC-AUC": float(np.mean(fold_aucs)),
        "Outer-fold AUC SD": float(np.std(fold_aucs, ddof=1)),
        "Valid AUC folds": len(fold_aucs),
    }

table_10 = pd.DataFrame([
    subgroup_performance(master_oof.loc[master_oof["pairing_status"] == "paired"].copy(), "Paired pregnancies"),
    subgroup_performance(master_oof.loc[master_oof["pairing_status"] == "singleton"].copy(), "Singleton pregnancies"),
])
save_table(table_10, "table_10_paired_singleton_performance")

display(paired_summary)
display(table_10)


In [ ]:

# ============================================================
# 20. METADATA / ACQUISITION-CONFOUNDER ANALYSES
# ============================================================

def first_float(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    m = re.search(r"[-+]?\d+(?:\.\d+)?", str(value))
    return float(m.group(0)) if m else np.nan

meta = record_meta.copy()
meta["gestation_delivery_week"] = meta["gestation_delivery_week_raw"].map(first_float)
meta["recording_week_rectime"] = meta["recording_week_rectime_raw"].map(first_float)
meta["maternal_age_years_header"] = meta["maternal_age_raw"].map(first_float)
meta["maternal_weight_kg"] = meta["maternal_weight_raw"].map(first_float)
meta["maternal_height_cm"] = meta["maternal_height_raw"].map(first_float)
meta["newborn_weight_g"] = meta["newborn_weight_raw"].map(first_float)

# Cross-source documented correction retained explicitly, never silently.
meta["maternal_age_years"] = meta["maternal_age_years_header"].copy()
meta.loc[meta["record_name"].astype(str) == "icehg1235", "maternal_age_years"] = 34.0

fold_lookup = dict(zip(master_oof["record_name"].astype(str), master_oof["outer_fold"].astype(int)))
meta["outer_fold"] = meta["record_name"].astype(str).map(fold_lookup).astype(int)

metadata_vars = [
    "delivery_mode", "recording_week_rectime", "gestation_delivery_week",
    "maternal_age_years", "maternal_weight_kg", "maternal_height_cm",
    "placental_position", "newborn_weight_g", "recording_duration_minutes",
    "pair_record_id"
]

availability_rows = []
for col in metadata_vars:
    n_av = int(meta[col].notna().sum())
    availability_rows.append({
        "Metadata variable": col,
        "Available, n": n_av,
        "Unavailable, n": int(len(meta)-n_av),
        "Availability, %": 100.0*n_av/len(meta),
    })

for col, status in [
    ("Acquisition-batch identifier", "Not provided"),
    ("Equipment/device-state variable", "Not provided"),
    ("Expert-labeled artifact count/frequency", "Not provided"),
]:
    availability_rows.append({
        "Metadata variable": col, "Available, n": 0,
        "Unavailable, n": len(meta), "Availability, %": 0.0, "Status": status
    })

metadata_availability = pd.DataFrame(availability_rows)
save_table(metadata_availability, "table_S3B_metadata_availability")

# Descriptive cohort/acquisition characteristics.
continuous = {
    "Recording week": "recording_week_rectime",
    "Maternal age": "maternal_age_years",
    "Maternal weight": "maternal_weight_kg",
    "Maternal height": "maternal_height_cm",
    "Gestational age at delivery": "gestation_delivery_week",
    "Newborn weight": "newborn_weight_g",
    "Recording duration": "recording_duration_minutes",
    "Expected 120-s windows": "expected_120s_segments",
}

desc_rows = []
for display_name, col in continuous.items():
    for period, g in meta.groupby("period"):
        vals = pd.to_numeric(g[col], errors="coerce").dropna()
        desc_rows.append({
            "Variable": display_name, "Group": period,
            "n": len(vals), "Mean": vals.mean(), "SD": vals.std(ddof=1),
            "Median": vals.median(), "Q1": vals.quantile(.25), "Q3": vals.quantile(.75),
            "Min": vals.min(), "Max": vals.max(),
        })
characteristics = pd.DataFrame(desc_rows)
save_table(characteristics, "table_S_metadata_characteristics_early_later")


# Reviewer 3 Comment 6:
# Descriptive maternal/acquisition characteristics jointly by recording period
# and paired/singleton status. This is descriptive only and does not enter model fitting.
pairing_desc_rows = []
for display_name, col in continuous.items():
    for (period, pairing_status), g in meta.groupby(["period","pairing_status"], dropna=False):
        vals = pd.to_numeric(g[col], errors="coerce").dropna()
        pairing_desc_rows.append({
            "Variable": display_name,
            "Period": period,
            "Pairing status": pairing_status,
            "n": int(len(vals)),
            "Mean": float(vals.mean()) if len(vals) else np.nan,
            "SD": float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
            "Median": float(vals.median()) if len(vals) else np.nan,
            "Q1": float(vals.quantile(.25)) if len(vals) else np.nan,
            "Q3": float(vals.quantile(.75)) if len(vals) else np.nan,
            "Min": float(vals.min()) if len(vals) else np.nan,
            "Max": float(vals.max()) if len(vals) else np.nan,
        })
pairing_characteristics = pd.DataFrame(pairing_desc_rows)
save_table(
    pairing_characteristics,
    "table_S_metadata_characteristics_period_by_pairing_status",
)

categorical_pairing_rows = []
for col in ["placental_position","delivery_mode"]:
    for (period, pairing_status), g in meta.groupby(["period","pairing_status"], dropna=False):
        denom = int(len(g))
        vc = g[col].fillna("Missing").astype(str).value_counts(dropna=False)
        for level, n in vc.items():
            categorical_pairing_rows.append({
                "Variable": col,
                "Period": period,
                "Pairing status": pairing_status,
                "Level": level,
                "n": int(n),
                "Percent within group": (100.0 * int(n) / denom) if denom else np.nan,
                "Group N": denom,
            })
categorical_pairing_characteristics = pd.DataFrame(categorical_pairing_rows)
save_table(
    categorical_pairing_characteristics,
    "table_S_categorical_characteristics_period_by_pairing_status",
)

# Metadata-only diagnostic baseline: Rectime and identifiers are explicitly excluded.
full_numeric = [
    "maternal_age_years", "maternal_weight_kg", "maternal_height_cm",
    "gestation_delivery_week", "newborn_weight_g",
    "recording_duration_minutes", "expected_120s_segments",
]
full_categorical = ["placental_position", "delivery_mode", "pairing_status"]
reduced_numeric = ["maternal_age_years", "maternal_weight_kg", "maternal_height_cm",
                   "recording_duration_minutes", "expected_120s_segments"]
reduced_categorical = ["placental_position", "pairing_status"]

def make_metadata_pipeline(numeric_cols, categorical_cols):
    prep = ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
    ])
    return Pipeline([
        ("prep", prep),
        ("clf", LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000, random_state=42)),
    ])

def metadata_oof_predictions(data, numeric_cols, categorical_cols, model_name):
    parts = []
    for fold in range(1, N_OUTER_SPLITS+1):
        tr = data.loc[data["outer_fold"] != fold].copy()
        te = data.loc[data["outer_fold"] == fold].copy()
        pipe = make_metadata_pipeline(numeric_cols, categorical_cols)
        pipe.fit(tr[numeric_cols+categorical_cols], tr["label_num"].astype(int))
        p = pipe.predict_proba(te[numeric_cols+categorical_cols])[:,1]
        out = te[["record_name","pregnancy_id","label_num","outer_fold"]].copy()
        out["prob_early"] = p
        out["model"] = model_name
        parts.append(out)
    return pd.concat(parts, ignore_index=True)

meta_oof_full = metadata_oof_predictions(meta, full_numeric, full_categorical, "metadata_full_age34")
meta_oof_reduced = metadata_oof_predictions(meta, reduced_numeric, reduced_categorical, "metadata_reduced")

# Missing-age sensitivity: set icehg1235 to NaN; imputation occurs within training fold.
meta_missing_age = meta.copy()
meta_missing_age.loc[meta_missing_age["record_name"].astype(str) == "icehg1235", "maternal_age_years"] = np.nan
meta_oof_missing_age = metadata_oof_predictions(
    meta_missing_age, full_numeric, full_categorical, "metadata_full_age_missing"
)

# Identifier-only diagnostic, intentionally separate from primary metadata baseline.
meta["record_id_numeric"] = meta["record_name"].astype(str).str.extract(r"(\d+)")[0].astype(float)
def id_only_oof(data):
    parts=[]
    for fold in range(1,N_OUTER_SPLITS+1):
        tr=data.loc[data["outer_fold"]!=fold].copy()
        te=data.loc[data["outer_fold"]==fold].copy()
        model=Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000, random_state=42))
        ])
        model.fit(tr[["record_id_numeric"]], tr["label_num"].astype(int))
        p=model.predict_proba(te[["record_id_numeric"]])[:,1]
        out=te[["record_name","pregnancy_id","label_num","outer_fold"]].copy()
        out["prob_early"]=p
        out["model"]="record_id_only"
        parts.append(out)
    return pd.concat(parts, ignore_index=True)

meta_oof_id = id_only_oof(meta)
metadata_oof = pd.concat(
    [meta_oof_full, meta_oof_missing_age, meta_oof_reduced, meta_oof_id],
    ignore_index=True
)
save_table(metadata_oof, "metadata_diagnostic_oof_predictions", directory=META_ROOT)

def summarize_continuous_oof(g):
    fold_auc=[]
    for fold, fg in sorted(g.groupby("outer_fold")):
        fold_auc.append(roc_auc_score(fg["label_num"].astype(int), fg["prob_early"].astype(float)))
    y=g["label_num"].astype(int).to_numpy()
    p=g["prob_early"].astype(float).to_numpy()
    pred=(p>=0.5).astype(int)
    m=calculate_binary_metrics(y,p,pred)
    return {
        "model": g["model"].iloc[0],
        "mean_outer_fold_auc": float(np.mean(fold_auc)),
        "outer_fold_auc_sd": float(np.std(fold_auc,ddof=1)),
        "pooled_auc_secondary": float(roc_auc_score(y,p)),
        "balanced_accuracy_at_0.5": float(m["balanced_accuracy"]),
    }

metadata_summary = pd.DataFrame([
    summarize_continuous_oof(g) for _,g in metadata_oof.groupby("model", sort=False)
])
save_table(metadata_summary, "metadata_diagnostic_summary", directory=META_ROOT)

# Delivery-mode subgroup performance using original held-out ensemble decisions/scores.
mode_lookup = dict(zip(meta["record_name"].astype(str), meta["delivery_mode"].astype(str)))
master_oof["delivery_mode"] = master_oof["record_name"].astype(str).map(mode_lookup)

delivery_rows=[]
for mode,g in master_oof.groupby("delivery_mode"):
    if len(g) == 0:
        continue
    row=subgroup_performance(g.copy(), str(mode))
    row["Delivery mode"]=mode
    delivery_rows.append(row)
delivery_summary=pd.DataFrame(delivery_rows)
save_table(delivery_summary, "table_S1_delivery_mode_performance")

# Paired fold-stratified pregnancy bootstrap: EHG vs metadata_full_age34.
comparison = master_oof[
    ["record_name","pregnancy_id","outer_fold","label_num","prob_ensemble_raw"]
].merge(
    meta_oof_full[["record_name","prob_early"]],
    on="record_name", how="inner", validate="one_to_one"
).rename(columns={"prob_early":"prob_metadata"})

def mean_fold_auc_col(df, score_col):
    return float(np.mean([
        roc_auc_score(g["label_num"].astype(int), g[score_col].astype(float))
        for _,g in sorted(df.groupby("outer_fold"))
    ]))

def paired_fold_cluster_bootstrap_delta(df, col_a, col_b, n_boot=20000, seed=20261020):
    rng=np.random.default_rng(seed)
    fold_data={}
    for fold,g in sorted(df.groupby("outer_fold")):
        g=g.reset_index(drop=True).copy()
        pids=g["pregnancy_id"].astype(str).drop_duplicates().to_numpy()
        pid_to_idx={pid:g.index[g["pregnancy_id"].astype(str)==pid].to_numpy() for pid in pids}
        fold_data[int(fold)]=(g,pids,pid_to_idx)
    vals=[]
    while len(vals)<n_boot:
        deltas=[]
        ok=True
        for fold,(g,pids,pid_to_idx) in fold_data.items():
            sampled=rng.choice(pids,size=len(pids),replace=True)
            b=g.iloc[np.concatenate([pid_to_idx[p] for p in sampled])]
            if b["label_num"].nunique()<2:
                ok=False; break
            deltas.append(
                roc_auc_score(b["label_num"],b[col_a]) -
                roc_auc_score(b["label_num"],b[col_b])
            )
        if ok:
            vals.append(float(np.mean(deltas)))
    vals=np.asarray(vals)
    observed=mean_fold_auc_col(df,col_a)-mean_fold_auc_col(df,col_b)
    p=min(1.0, 2.0*min((np.sum(vals<=0)+1)/(len(vals)+1),(np.sum(vals>=0)+1)/(len(vals)+1)))
    return {
        "delta_auc": observed,
        "ci95_low": float(np.quantile(vals,.025)),
        "ci95_high": float(np.quantile(vals,.975)),
        "two_sided_p": float(p),
    }

ehg_meta_delta = paired_fold_cluster_bootstrap_delta(
    comparison, "prob_ensemble_raw", "prob_metadata", n_boot=20000, seed=20261020
)

table_S2_fold = []
for fold,g in sorted(comparison.groupby("outer_fold")):
    a=roc_auc_score(g["label_num"],g["prob_metadata"])
    b=roc_auc_score(g["label_num"],g["prob_ensemble_raw"])
    table_S2_fold.append({
        "Outer fold": int(fold),
        "Records": len(g),
        "Pregnancies": g["pregnancy_id"].nunique(),
        "Metadata-only AUC": a,
        "EHG AUC": b,
        "EHG minus metadata": b-a,
    })
table_S2=pd.DataFrame(table_S2_fold)
save_table(table_S2, "table_S2_metadata_vs_ehg_fold_auc")

save_json({
    "age_anomaly_record":"icehg1235",
    "header_age": float(meta.loc[meta["record_name"]=="icehg1235","maternal_age_years_header"].iloc[0]),
    "documented_table_age_used_primary":34.0,
    "rectime_excluded_from_metadata_classifier":True,
    "identifiers_excluded_from_primary_metadata_classifier":True,
    "ehg_vs_metadata_paired_bootstrap":ehg_meta_delta,
}, "metadata_confounder_audit", META_ROOT)

display(metadata_summary)
display(delivery_summary)
display(table_S2)
print("EHG vs metadata:", ehg_meta_delta)


In [ ]:

# ============================================================
# 21. SIMPLE SIGNAL BASELINES
#     Welch spectral features + Logistic Regression (primary simple baseline)
#     SVM and Random Forest are exploratory companion baselines.
# ============================================================

SPECTRAL_FEATURE_PATH = BASELINE_ROOT / "record_spectral_features.csv"

def spectral_features_1d(x, fs=FS):
    x = np.asarray(x, dtype=np.float64)
    f, pxx = welch(x, fs=fs, nperseg=min(4096, len(x)), noverlap=None)
    mask = (f >= FILTER_LOW_HZ) & (f <= FILTER_HIGH_HZ)
    f = f[mask]
    pxx = np.maximum(pxx[mask], 0)
    total = np.trapz(pxx, f) if len(f) > 1 else 0.0
    eps = 1e-12
    peak_f = float(f[np.argmax(pxx)]) if len(f) else np.nan
    csum = np.cumsum(pxx)
    med_f = float(f[np.searchsorted(csum, 0.5*csum[-1])]) if len(f) and csum[-1] > 0 else np.nan
    centroid = float(np.sum(f*pxx)/(np.sum(pxx)+eps)) if len(f) else np.nan

    bands = {
        "bp_008_03": (0.08,0.30),
        "bp_03_10": (0.30,1.00),
        "bp_10_30": (1.00,3.00),
        "bp_30_50": (3.00,5.00),
    }
    out = {
        "peak_frequency": peak_f,
        "median_frequency": med_f,
        "spectral_centroid": centroid,
        "rms": float(np.sqrt(np.mean(x**2))),
        "std": float(np.std(x, ddof=1)),
        "mad": float(np.median(np.abs(x-np.median(x)))),
    }
    for name,(lo,hi) in bands.items():
        m=(f>=lo)&(f<=(hi if hi==5.0 else hi))
        bp=np.trapz(pxx[m],f[m]) if np.sum(m)>1 else 0.0
        out[name]=float(bp/(total+eps))
    return out

if SPECTRAL_FEATURE_PATH.exists():
    spectral_df=pd.read_csv(SPECTRAL_FEATURE_PATH)
else:
    rows=[]
    for i,row in enumerate(record_meta.sort_values("record_name").itertuples(),start=1):
        filtered=np.load(FILTERED_ROOT/f"{row.record_name}.npz")["filtered"]
        feat={
            "record_name":row.record_name,
            "pregnancy_id":row.pregnancy_id,
            "label_num":int(row.label_num),
            "outer_fold":int(fold_lookup[row.record_name]),
        }
        for ch in range(3):
            sf=spectral_features_1d(filtered[:,ch])
            for key,val in sf.items():
                feat[f"S{ch+1}_{key}"]=val
        rows.append(feat)
        if i%20==0:
            print(f"Spectral features {i}/{len(record_meta)}")
    spectral_df=pd.DataFrame(rows)
    spectral_df.to_csv(SPECTRAL_FEATURE_PATH,index=False)

feature_cols=[c for c in spectral_df.columns if c.startswith(("S1_","S2_","S3_"))]

def baseline_oof(model_name):
    parts=[]
    for fold in range(1,N_OUTER_SPLITS+1):
        tr=spectral_df.loc[spectral_df["outer_fold"]!=fold].copy()
        te=spectral_df.loc[spectral_df["outer_fold"]==fold].copy()

        if model_name=="Logistic Regression":
            model=Pipeline([
                ("impute",SimpleImputer(strategy="median")),
                ("scale",StandardScaler()),
                ("clf",LogisticRegression(C=1.0,solver="lbfgs",max_iter=5000,random_state=42)),
            ])
        elif model_name=="RBF-SVM":
            model=Pipeline([
                ("impute",SimpleImputer(strategy="median")),
                ("scale",StandardScaler()),
                ("clf",SVC(C=1.0,kernel="rbf",gamma="scale",probability=True,random_state=42)),
            ])
        elif model_name=="Random Forest":
            model=Pipeline([
                ("impute",SimpleImputer(strategy="median")),
                ("clf",RandomForestClassifier(n_estimators=500,max_features="sqrt",
                                              random_state=42,n_jobs=-1,class_weight="balanced")),
            ])
        else:
            raise ValueError(model_name)

        model.fit(tr[feature_cols],tr["label_num"].astype(int))
        p=model.predict_proba(te[feature_cols])[:,1]
        out=te[["record_name","pregnancy_id","label_num","outer_fold"]].copy()
        out["prob_early"]=p
        out["model"]=model_name
        parts.append(out)
    return pd.concat(parts,ignore_index=True)

baseline_names=["Logistic Regression","RBF-SVM","Random Forest"]
baseline_oof_df=pd.concat([baseline_oof(name) for name in baseline_names],ignore_index=True)
save_table(baseline_oof_df,"simple_spectral_baseline_oof",directory=BASELINE_ROOT)

baseline_summary=[]
for name,g in baseline_oof_df.groupby("model",sort=False):
    fold_aucs=[
        roc_auc_score(fg["label_num"],fg["prob_early"])
        for _,fg in sorted(g.groupby("outer_fold"))
    ]
    baseline_summary.append({
        "Model":name,
        "Mean outer-fold ROC-AUC":float(np.mean(fold_aucs)),
        "Outer-fold AUC SD":float(np.std(fold_aucs,ddof=1)),
        "Pooled AUC secondary":float(roc_auc_score(g["label_num"],g["prob_early"])),
    })
baseline_summary_df=pd.DataFrame(baseline_summary)
save_table(baseline_summary_df,"simple_spectral_baseline_summary",directory=BASELINE_ROOT)

# Paired primary EHG-vs-simple Logistic Regression comparison (descriptive + bootstrap).
lr_oof=baseline_oof_df.loc[baseline_oof_df["model"]=="Logistic Regression"].copy()
ehg_lr=master_oof[
    ["record_name","pregnancy_id","outer_fold","label_num","prob_ensemble_raw"]
].merge(
    lr_oof[["record_name","prob_early"]],
    on="record_name",how="inner",validate="one_to_one"
).rename(columns={"prob_early":"prob_spectral_lr"})
ehg_vs_lr=paired_fold_cluster_bootstrap_delta(
    ehg_lr,"prob_ensemble_raw","prob_spectral_lr",n_boot=20000,seed=20261022
)
save_json(ehg_vs_lr,"ehg_vs_spectral_lr_bootstrap",BASELINE_ROOT)

display(baseline_summary_df)
print("EHG vs spectral LR:",ehg_vs_lr)


## Repeated-seed robustness

Five prespecified seeds retrain only the final representation models on the same fixed outer folds and the same deterministic final fit/early-stop split. Fold-specific fusion weights remain fixed at values selected by the corrected baseline inner OOF procedure. No seed is selected on held-out performance.


In [ ]:

# ============================================================
# 22 — Reviewer 1 Comment 4
# Fixed-split repeated-seed training sensitivity analysis
# ============================================================

# Methodological purpose:
# - Outer-fold assignments are fixed.
# - Preprocessing, architecture, and training hyperparameters are fixed.
# - Fold-specific fusion weights are fixed at the corrected baseline values.
# - The final four representation models are retrained under five independent
#   seed settings.
# - NO seed is selected based on held-out performance.
#
# This isolates stochastic variability of the final trained models from
# fold-to-fold partition variability. It does NOT claim to estimate every
# possible source of model-development uncertainty.

# Preserve baseline globals so the notebook can continue safely.
_BASELINE_GLOBALS = {
    "OUT_DIR": OUT_DIR,
    "MODEL_DIR": MODEL_DIR,
    "PRED_DIR": PRED_DIR,
    "FOLD_DIR": FOLD_DIR,
    "TABLE_DIR": TABLE_DIR,
    "FIG_DIR": FIG_DIR,
    "EXPORT_DIR": EXPORT_DIR,
    "RANDOM_STATE": RANDOM_STATE,
}

baseline_config_lookup = (
    selected_configs_df
    .set_index("outer_fold")
    .copy()
)

repeat_fold_rows = []
repeat_oof_parts = []

for base_seed in REPEATED_SEEDS:
    print("\n" + "=" * 100)
    print(f"REPEATED-SEED RUN: base seed {base_seed}")
    print("=" * 100)

    # New output root per seed, making the run resume-safe.
    seed_root = COMMENT4_DIR / f"seed_{base_seed}"

    OUT_DIR = seed_root
    MODEL_DIR = seed_root / "models"
    PRED_DIR = seed_root / "predictions"
    FOLD_DIR = seed_root / "fold_results"
    TABLE_DIR = seed_root / "tables"
    FIG_DIR = seed_root / "figures"
    EXPORT_DIR = seed_root / "export"

    for p in [
        OUT_DIR, MODEL_DIR, PRED_DIR, FOLD_DIR,
        TABLE_DIR, FIG_DIR, EXPORT_DIR,
    ]:
        p.mkdir(parents=True, exist_ok=True)

    # The global seed setting changes; data partitions do not.
    RANDOM_STATE = int(base_seed)
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    tf.random.set_seed(RANDOM_STATE)

    seed_outer_parts = []

    for outer_fold, (
        outer_train_record_idx,
        outer_test_record_idx,
    ) in enumerate(
        outer_splits,
        start=1,
    ):
        print(
            f"\nSeed {base_seed} | "
            f"Outer fold {outer_fold}/5"
        )

        outer_train_record_idx = np.asarray(
            outer_train_record_idx,
            dtype=int,
        )
        outer_test_record_idx = np.asarray(
            outer_test_record_idx,
            dtype=int,
        )

        # Exactly the same deterministic final fit/early-stop split
        # used by the corrected baseline.
        (
            final_es_split_id,
            final_train_record_idx,
            final_val_record_idx,
        ) = final_outer_fit_earlystop_split(
            outer_train_record_idx,
            outer_fold,
        )

        final_train_seg = (
            record_indices_to_segment_indices(
                final_train_record_idx
            )
        )
        final_val_seg = (
            record_indices_to_segment_indices(
                final_val_record_idx
            )
        )
        outer_test_seg = (
            record_indices_to_segment_indices(
                outer_test_record_idx
            )
        )

        outer_rep_dict = {}

        for rep in REPRESENTATIONS:
            run_name = (
                f"seed{base_seed}_outer{outer_fold}_"
                "final_fixedsplit"
            )

            seg_pred = train_predict_representation(
                representation=rep,
                train_segment_indices=final_train_seg,
                validation_segment_indices=final_val_seg,
                test_segment_indices=outer_test_seg,
                run_name=run_name,
                save_final_model=False,
            )

            rec_pred = aggregate_segment_predictions(
                seg_pred
            )

            outer_rep_dict[rep] = rec_pred

        outer_merged = merge_rep_record_predictions(
            outer_rep_dict
        )

        config = baseline_config_lookup.loc[
            outer_fold
        ]

        fixed_weights = config[
            ["w_RGB", "w_S1", "w_S2", "w_S3"]
        ].to_numpy(dtype=float)

        prob_matrix = outer_merged[
            ["prob_RGB", "prob_S1", "prob_S2", "prob_S3"]
        ].astype(float).to_numpy()

        weighted_raw = prob_matrix @ fixed_weights

        y_outer = outer_merged[
            "label_num"
        ].astype(int).to_numpy()

        result = outer_merged.copy()
        result["base_seed"] = int(base_seed)
        result["outer_fold"] = int(outer_fold)
        result["prob_ensemble_raw"] = weighted_raw
        result["w_RGB"] = float(fixed_weights[0])
        result["w_S1"] = float(fixed_weights[1])
        result["w_S2"] = float(fixed_weights[2])
        result["w_S3"] = float(fixed_weights[3])
        result["final_earlystop_split_id"] = int(
            final_es_split_id
        )

        # The actual effective TensorFlow seed used by the original helper
        # is base seed + deterministic CRC32(run name) offset.
        for rep in REPRESENTATIONS:
            rep_run_name = (
                f"{rep}_seed{base_seed}_"
                f"outer{outer_fold}_final_fixedsplit"
            )
            result[
                f"effective_seed_{rep}"
            ] = int(
                base_seed
                + (
                    zlib.crc32(
                        rep_run_name.encode("utf-8")
                    )
                    % 100000
                )
            )

        rep_aucs = {
            rep: float(
                roc_auc_score(
                    y_outer,
                    outer_merged[
                        f"prob_{rep}"
                    ].astype(float),
                )
            )
            for rep in REPRESENTATIONS
        }

        ensemble_auc = float(
            roc_auc_score(
                y_outer,
                weighted_raw,
            )
        )

        repeat_fold_rows.append(
            {
                "base_seed": int(base_seed),
                "outer_fold": int(outer_fold),
                "n_records": int(len(outer_merged)),
                "n_pregnancies": int(
                    outer_merged[
                        "pregnancy_id"
                    ].nunique()
                ),
                "RGB_auc": rep_aucs["RGB"],
                "S1_auc": rep_aucs["S1"],
                "S2_auc": rep_aucs["S2"],
                "S3_auc": rep_aucs["S3"],
                "fixed_weighted_ensemble_auc": ensemble_auc,
                "w_RGB": float(fixed_weights[0]),
                "w_S1": float(fixed_weights[1]),
                "w_S2": float(fixed_weights[2]),
                "w_S3": float(fixed_weights[3]),
                "final_earlystop_split_id": int(
                    final_es_split_id
                ),
            }
        )

        seed_outer_parts.append(result)
        repeat_oof_parts.append(result)

        fold_out = (
            FOLD_DIR
            / f"outer_{outer_fold}"
        )
        fold_out.mkdir(
            parents=True,
            exist_ok=True,
        )
        result.to_csv(
            fold_out
            / "outer_record_predictions_fixed_weights.csv",
            index=False,
        )

        print(
            "Fixed-weight raw AUC:",
            round(ensemble_auc, 4),
        )

    seed_oof = (
        pd.concat(
            seed_outer_parts,
            ignore_index=True,
        )
        .sort_values(
            ["outer_fold", "record_name"]
        )
        .reset_index(drop=True)
    )

    if len(seed_oof) != 126:
        raise RuntimeError(
            f"Seed {base_seed}: expected 126 OOF records, "
            f"found {len(seed_oof)}."
        )

    seed_oof.to_csv(
        PRED_DIR
        / "master_oof_fixed_weight_repeated_seed.csv",
        index=False,
    )

# Restore baseline globals.
OUT_DIR = _BASELINE_GLOBALS["OUT_DIR"]
MODEL_DIR = _BASELINE_GLOBALS["MODEL_DIR"]
PRED_DIR = _BASELINE_GLOBALS["PRED_DIR"]
FOLD_DIR = _BASELINE_GLOBALS["FOLD_DIR"]
TABLE_DIR = _BASELINE_GLOBALS["TABLE_DIR"]
FIG_DIR = _BASELINE_GLOBALS["FIG_DIR"]
EXPORT_DIR = _BASELINE_GLOBALS["EXPORT_DIR"]
RANDOM_STATE = _BASELINE_GLOBALS["RANDOM_STATE"]

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

repeat_fold_df = pd.DataFrame(
    repeat_fold_rows
).sort_values(
    ["base_seed", "outer_fold"]
)

repeat_oof_df = pd.concat(
    repeat_oof_parts,
    ignore_index=True,
)

repeat_fold_df.to_csv(
    COMMENT4_DIR
    / "comment4_repeated_seed_fold_auc.csv",
    index=False,
)

repeat_oof_df.to_csv(
    COMMENT4_DIR
    / "comment4_repeated_seed_all_oof_predictions.csv",
    index=False,
)

display(repeat_fold_df)


In [ ]:

# ============================================================
# 23 — Comment 4 summary, variability audit, manuscript-ready output
# ============================================================

# Seed-wise primary summary: arithmetic mean of the five outer-fold AUCs.
seed_summary_rows = []

for base_seed, g in repeat_fold_df.groupby(
    "base_seed",
    sort=True,
):
    seed_oof = repeat_oof_df.loc[
        repeat_oof_df["base_seed"] == base_seed
    ].copy()

    seed_summary_rows.append(
        {
            "base_seed": int(base_seed),
            "mean_outer_fold_auc": float(
                g[
                    "fixed_weighted_ensemble_auc"
                ].mean()
            ),
            "outer_fold_auc_sd": float(
                g[
                    "fixed_weighted_ensemble_auc"
                ].std(ddof=1)
            ),
            "min_outer_fold_auc": float(
                g[
                    "fixed_weighted_ensemble_auc"
                ].min()
            ),
            "max_outer_fold_auc": float(
                g[
                    "fixed_weighted_ensemble_auc"
                ].max()
            ),
            "secondary_pooled_auc": float(
                roc_auc_score(
                    seed_oof["label_num"].astype(int),
                    seed_oof[
                        "prob_ensemble_raw"
                    ].astype(float),
                )
            ),
        }
    )

seed_summary_df = pd.DataFrame(
    seed_summary_rows
).sort_values("base_seed")

# Training stochasticity is summarized by variability of the five
# seed-specific mean outer-fold AUC values.
seed_means = seed_summary_df[
    "mean_outer_fold_auc"
].to_numpy(float)

training_variability_df = pd.DataFrame(
    [
        {
            "n_seed_settings": int(
                len(seed_summary_df)
            ),
            "seed_settings": ",".join(
                map(
                    str,
                    seed_summary_df[
                        "base_seed"
                    ].astype(int).tolist(),
                )
            ),
            "mean_of_seed_mean_outer_fold_auc": float(
                np.mean(seed_means)
            ),
            "sd_of_seed_mean_outer_fold_auc": float(
                np.std(seed_means, ddof=1)
            ),
            "min_seed_mean_outer_fold_auc": float(
                np.min(seed_means)
            ),
            "max_seed_mean_outer_fold_auc": float(
                np.max(seed_means)
            ),
            "range_seed_mean_outer_fold_auc": float(
                np.max(seed_means)
                - np.min(seed_means)
            ),
            "no_seed_selected_on_test_performance": True,
            "outer_splits_fixed": True,
            "preprocessing_fixed": True,
            "architecture_fixed": True,
            "training_hyperparameters_fixed": True,
            "fusion_weights_fixed_from_corrected_inner_oof": True,
        }
    ]
)

# Fold-wise across-seed variability can show whether one outer partition
# is consistently more difficult than the others.
fold_across_seed_df = (
    repeat_fold_df
    .groupby(
        "outer_fold",
        as_index=False,
    )
    .agg(
        mean_auc_across_seeds=(
            "fixed_weighted_ensemble_auc",
            "mean",
        ),
        sd_auc_across_seeds=(
            "fixed_weighted_ensemble_auc",
            "std",
        ),
        min_auc_across_seeds=(
            "fixed_weighted_ensemble_auc",
            "min",
        ),
        max_auc_across_seeds=(
            "fixed_weighted_ensemble_auc",
            "max",
        ),
    )
)

seed_summary_df.to_csv(
    COMMENT4_DIR
    / "comment4_repeated_seed_summary.csv",
    index=False,
)

training_variability_df.to_csv(
    COMMENT4_DIR
    / "comment4_training_variability_summary.csv",
    index=False,
)

fold_across_seed_df.to_csv(
    COMMENT4_DIR
    / "comment4_fold_across_seed_variability.csv",
    index=False,
)

audit = {
    "analysis": (
        "Reviewer 1 Comment 4 fixed-split repeated-seed "
        "training sensitivity analysis"
    ),
    "corrected_nested_cv_root": str(
        _BASELINE_GLOBALS["OUT_DIR"]
    ),
    "comment4_output_root": str(
        COMMENT4_DIR
    ),
    "records": int(
        master_oof.shape[0]
    ),
    "pregnancies": int(
        master_oof["pregnancy_id"].nunique()
    ),
    "outer_folds": int(
        N_OUTER_SPLITS
    ),
    "repeated_seed_settings": [
        int(x)
        for x in REPEATED_SEEDS
    ],
    "repeated_seed_protocol": {
        "outer_fold_assignments": "fixed",
        "preprocessing": "fixed",
        "architecture": "fixed",
        "training_hyperparameters": "fixed",
        "final_fit_earlystop_split": "fixed within outer fold",
        "fusion_weights": (
            "fixed at fold-specific values selected from the "
            "corrected untouched inner-evaluation OOF predictions"
        ),
        "seed_selection": (
            "none; all prespecified seed settings retained"
        ),
        "primary_seed_metric": (
            "arithmetic mean of five outer-fold raw ROC-AUC values"
        ),
        "purpose": (
            "quantify final-model stochastic training variability "
            "separately from fold-to-fold partition variability"
        ),
    },
    "important_scope_note": (
        "This repeated-seed analysis conditions on the corrected "
        "fold-specific fusion configuration and does not estimate "
        "every possible source of model-development uncertainty."
    ),
    "outputs": [
        "comment4_repeated_seed_fold_auc.csv",
        "comment4_repeated_seed_all_oof_predictions.csv",
        "comment4_repeated_seed_summary.csv",
        "comment4_training_variability_summary.csv",
        "comment4_fold_across_seed_variability.csv",
        "comment4_analysis_audit.json",
    ],
}

with open(
    COMMENT4_DIR / "comment4_analysis_audit.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        audit,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("=" * 96)
print("REVIEWER 1 — COMMENT 4 REPEATED-SEED SUMMARY")
print("=" * 96)

print("\nSeed-wise results:")
display(seed_summary_df)

print("\nAcross-seed training variability:")
display(training_variability_df)

print("\nFold-specific variability across seeds:")
display(fold_across_seed_df)

print("\nInterpretation guardrails:")
print(
    "- No seed was selected or discarded based on held-out performance."
)
print(
    "- The five pregnancy-wise outer partitions were kept fixed."
)
print(
    "- Architecture, preprocessing, and training hyperparameters were fixed."
)
print(
    "- Fold-specific fusion weights were frozen from the corrected "
    "inner-OOF baseline."
)
print(
    "- Therefore this analysis targets stochastic final-model training "
    "variability; it is not a second hyperparameter search."
)
print(
    "- The fixed-OOF bootstrap and the repeated-seed analysis quantify "
    "different sources of uncertainty and should not be described as "
    "interchangeable."
)

print("\nAll Comment 4 outputs:")
print(COMMENT4_DIR)


In [ ]:

# ============================================================
# 24. CORRECTED ARCHITECTURE ABLATION:
#     CNN-only RGB / Transformer-only RGB / full CNN–Transformer RGB
# ============================================================

def classification_head_from_vector(x, prefix="head"):
    x=layers.Dense(128,activation="relu",kernel_regularizer=regularizers.l2(L2_WEIGHT),
                   name=f"{prefix}_dense_128")(x)
    x=layers.Dropout(DROPOUT,name=f"{prefix}_dropout_128")(x)
    x=layers.Dense(64,activation="relu",kernel_regularizer=regularizers.l2(L2_WEIGHT),
                   name=f"{prefix}_dense_64")(x)
    x=layers.Dropout(DROPOUT,name=f"{prefix}_dropout_64")(x)
    return layers.Dense(1,activation="sigmoid",dtype="float32",name="probability")(x)

def build_a2_cnn_only():
    inputs=layers.Input((IMAGE_SIZE,IMAGE_SIZE,3),name="RGB_scalogram")
    x=cnn_backbone(inputs)
    x=layers.GlobalAveragePooling2D(name="cnn_gap")(x)
    out=classification_head_from_vector(x)
    model=models.Model(inputs,out,name="A2_CNN_only_RGB")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",metrics=[keras.metrics.AUC(name="auc")]
    )
    return model

def build_a2_transformer_only():
    inputs=layers.Input((IMAGE_SIZE,IMAGE_SIZE,3),name="RGB_scalogram")
    # 224/32 = 7 -> 49 non-overlapping image tokens.
    x=layers.Conv2D(
        EMBED_DIM,kernel_size=32,strides=32,padding="valid",
        kernel_regularizer=regularizers.l2(L2_WEIGHT),name="patch_projection"
    )(inputs)
    x=layers.Reshape((49,EMBED_DIM),name="patch_tokens")(x)
    x=AddPositionEmbedding(name="position_embedding")(x)
    for b in range(1,NUM_TRANSFORMER_BLOCKS+1):
        x=transformer_encoder_block(x,b)
    x=layers.GlobalAveragePooling1D(name="token_gap")(x)
    out=classification_head_from_vector(x)
    model=models.Model(inputs,out,name="A2_Transformer_only_RGB")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",metrics=[keras.metrics.AUC(name="auc")]
    )
    return model

def build_ablation_model(architecture):
    if architecture=="cnn_only":
        return build_a2_cnn_only()
    if architecture=="transformer_only":
        return build_a2_transformer_only()
    raise ValueError(architecture)

complexity_rows=[]
for label,builder in [
    ("CNN-only RGB",build_a2_cnn_only),
    ("Transformer-only RGB",build_a2_transformer_only),
    ("CNN–Transformer RGB",lambda:build_a2_cnn_transformer("RGB")),
]:
    K.clear_session()
    m=builder()
    complexity_rows.append({"Configuration":label,"Parameters":int(m.count_params())})
    del m; K.clear_session(); gc.collect()
complexity_df=pd.DataFrame(complexity_rows)
save_table(complexity_df,"architecture_parameter_counts",directory=ABLATION_ROOT)
display(complexity_df)

def train_predict_ablation(
    architecture, train_segment_indices, validation_segment_indices,
    test_segment_indices, run_name
):
    pred_dir=ABLATION_ROOT/"predictions"/architecture
    hist_dir=ABLATION_ROOT/"histories"/architecture
    pred_dir.mkdir(parents=True,exist_ok=True); hist_dir.mkdir(parents=True,exist_ok=True)
    pred_path=pred_dir/f"{run_name}_segment_predictions.csv"
    hist_path=hist_dir/f"{run_name}_history.csv"

    if not FORCE_RETRAIN_ABLATION and pred_path.exists():
        print("Ablation resume ->",pred_path)
        return pd.read_csv(pred_path)

    run_seed=set_run_seed(f"{architecture}_RGB_{run_name}")
    K.clear_session(); gc.collect()
    model=build_ablation_model(architecture)
    train_gen=A2ScalogramSequence("RGB",train_segment_indices,labels_all,BATCH_SIZE,shuffle=True)
    val_gen=A2ScalogramSequence("RGB",validation_segment_indices,labels_all,BATCH_SIZE,shuffle=False)
    test_gen=A2ScalogramSequence("RGB",test_segment_indices,labels_all,BATCH_SIZE,shuffle=False)
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=PATIENCE,
                                      restore_best_weights=True,verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_auc",mode="max",factor=.5,
                                          patience=2,min_lr=1e-7,verbose=1),
        keras.callbacks.TerminateOnNaN(),
    ]
    hist=model.fit(train_gen,validation_data=val_gen,epochs=EPOCHS,callbacks=callbacks,verbose=1)
    pd.DataFrame(hist.history).to_csv(hist_path,index=False)
    probs=model.predict(test_gen,verbose=1).reshape(-1)
    lookup=(segment_df.set_index("segment_index")
            .loc[np.asarray(test_segment_indices,dtype=int)].reset_index())
    pred=pd.DataFrame({
        "segment_index":np.asarray(test_segment_indices,dtype=int),
        "record_name":lookup["record_name"].astype(str).values,
        "pregnancy_id":lookup["pregnancy_id"].astype(str).values,
        "label_text":lookup["label_text"].astype(str).values,
        "label_num":lookup["label_num"].astype(int).values,
        "prob_early_raw":probs.astype(float),
    })
    pred.to_csv(pred_path,index=False)
    del model,train_gen,val_gen,test_gen,hist
    K.clear_session();gc.collect()
    return pred

def run_corrected_ablation_architecture(architecture):
    out_parts=[]
    arch_root=ABLATION_ROOT/"fold_results"/architecture
    arch_root.mkdir(parents=True,exist_ok=True)

    for outer_fold,(outer_train_idx,outer_test_idx) in enumerate(outer_splits,start=1):
        fold_dir=arch_root/f"outer_{outer_fold}"
        fold_dir.mkdir(parents=True,exist_ok=True)
        final_path=fold_dir/"outer_record_predictions.csv"
        if not FORCE_RETRAIN_ABLATION and final_path.exists():
            out_parts.append(pd.read_csv(final_path)); continue

        outer_train_idx=np.asarray(outer_train_idx,dtype=int)
        outer_test_idx=np.asarray(outer_test_idx,dtype=int)
        outer_train_df=record_df.iloc[outer_train_idx].copy()
        inner_cv=GroupKFold(n_splits=N_INNER_SPLITS)
        inner_splits=list(inner_cv.split(
            outer_train_df,outer_train_df["label_num"].astype(int).values,
            outer_train_df["pregnancy_id"].astype(str).values
        ))
        inner_parts=[]
        for inner_fold,(dev_local,eval_local) in enumerate(inner_splits,start=1):
            _,fit_idx,es_idx,eval_idx=make_inner_fit_earlystop_eval_split(
                outer_train_idx,dev_local,eval_local,outer_fold,inner_fold
            )
            seg_pred=train_predict_ablation(
                architecture,
                record_indices_to_segment_indices(fit_idx),
                record_indices_to_segment_indices(es_idx),
                record_indices_to_segment_indices(eval_idx),
                f"outer{outer_fold}_inner{inner_fold}_fit_es_eval"
            )
            rec=aggregate_segment_predictions(seg_pred)
            rec["inner_fold"]=inner_fold
            inner_parts.append(rec)
        inner_oof=pd.concat(inner_parts,ignore_index=True)
        if inner_oof["record_name"].nunique()!=len(outer_train_df):
            raise RuntimeError(f"{architecture} outer {outer_fold}: invalid inner OOF coverage")

        y_inner=inner_oof["label_num"].astype(int).to_numpy()
        p_inner=inner_oof["prob_early_raw"].astype(float).to_numpy()
        cal,slope,intercept=fit_positive_platt(y_inner,p_inner)
        if cal is None:
            raise RuntimeError(f"{architecture} outer {outer_fold}: non-positive Platt slope")
        p_inner_cal=apply_platt(cal,p_inner)
        thr_row,_=select_robust_threshold(y_inner,p_inner_cal)
        threshold=float(thr_row["threshold"])

        _,final_fit_idx,final_es_idx=final_outer_fit_earlystop_split(outer_train_idx,outer_fold)
        seg_pred=train_predict_ablation(
            architecture,
            record_indices_to_segment_indices(final_fit_idx),
            record_indices_to_segment_indices(final_es_idx),
            record_indices_to_segment_indices(outer_test_idx),
            f"outer{outer_fold}_final_corrected"
        )
        rec=aggregate_segment_predictions(seg_pred)
        rec["prob_raw"]=rec["prob_early_raw"].astype(float)
        rec["prob_calibrated"]=apply_platt(cal,rec["prob_raw"].to_numpy())
        rec["pred_label_num"]=(rec["prob_calibrated"]>=threshold).astype(int)
        rec["outer_fold"]=outer_fold
        rec["architecture"]=architecture
        rec["threshold"]=threshold
        rec["platt_slope"]=slope
        rec["platt_intercept"]=intercept
        rec.to_csv(final_path,index=False)
        inner_oof.to_csv(fold_dir/"inner_oof_RGB.csv",index=False)
        out_parts.append(rec)
        del cal;gc.collect()

    oof=(pd.concat(out_parts,ignore_index=True)
         .sort_values(["outer_fold","record_name"]).reset_index(drop=True))
    oof.to_csv(ABLATION_ROOT/f"{architecture}_RGB_corrected_oof.csv",index=False)
    return oof

if RUN_ARCHITECTURE_ABLATION:
    cnn_oof=run_corrected_ablation_architecture("cnn_only")
    transformer_oof=run_corrected_ablation_architecture("transformer_only")
else:
    cnn_oof=pd.read_csv(ABLATION_ROOT/"cnn_only_RGB_corrected_oof.csv")
    transformer_oof=pd.read_csv(ABLATION_ROOT/"transformer_only_RGB_corrected_oof.csv")

# Hybrid RGB raw scores come directly from primary corrected OOF.
hybrid_oof=master_oof[
    ["record_name","pregnancy_id","label_num","outer_fold","prob_RGB"]
].copy().rename(columns={"prob_RGB":"prob_raw"})

def auc_summary_from_oof(name,df,score="prob_raw"):
    aucs=[
        roc_auc_score(g["label_num"].astype(int),g[score].astype(float))
        for _,g in sorted(df.groupby("outer_fold"))
    ]
    return {
        "Configuration":name,
        "Mean outer-fold raw ROC-AUC":float(np.mean(aucs)),
        "Outer-fold AUC SD":float(np.std(aucs,ddof=1)),
        "Secondary pooled raw ROC-AUC":float(roc_auc_score(df["label_num"],df[score])),
        **{f"Fold {i+1} AUC":float(v) for i,v in enumerate(aucs)}
    }

architecture_summary=pd.DataFrame([
    auc_summary_from_oof("CNN-only RGB",cnn_oof),
    auc_summary_from_oof("Transformer-only RGB",transformer_oof),
    auc_summary_from_oof("CNN–Transformer RGB",hybrid_oof),
])
save_table(architecture_summary,"architecture_ablation_summary",directory=ABLATION_ROOT)
save_table(architecture_summary,"table_11_architecture_ablation_final")
display(architecture_summary)


In [ ]:

# ============================================================
# 24B. ARCHITECTURE-ABLATION REPEATED-SEED SENSITIVITY
#      Hybrid RGB reuses the main repeated-seed RGB predictions.
#      CNN-only / Transformer-only final models are retrained on the
#      same fixed outer folds and deterministic final fit/early-stop split.
# ============================================================

ARCH_SEED_ROOT = ABLATION_ROOT / "repeated_seed"
ARCH_SEED_ROOT.mkdir(parents=True, exist_ok=True)

baseline_random_state = RANDOM_STATE
arch_seed_fold_rows = []
arch_seed_oof_parts = []

# Hybrid RGB: reuse already trained primary repeated-seed models.
for base_seed in REPEATED_SEEDS:
    h = repeat_oof_df.loc[repeat_oof_df["base_seed"].astype(int) == int(base_seed)].copy()
    for fold, g in sorted(h.groupby("outer_fold")):
        arch_seed_fold_rows.append({
            "base_seed": int(base_seed),
            "architecture": "CNN–Transformer RGB",
            "outer_fold": int(fold),
            "raw_auc": float(roc_auc_score(g["label_num"].astype(int), g["prob_RGB"].astype(float))),
        })

# CNN-only and Transformer-only repeated final-model training.
for base_seed in REPEATED_SEEDS:
    RANDOM_STATE = int(base_seed)
    random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE); tf.random.set_seed(RANDOM_STATE)

    for architecture, label in [
        ("cnn_only", "CNN-only RGB"),
        ("transformer_only", "Transformer-only RGB"),
    ]:
        seed_parts = []
        for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(outer_splits, start=1):
            outer_train_idx = np.asarray(outer_train_idx, dtype=int)
            outer_test_idx = np.asarray(outer_test_idx, dtype=int)
            _, fit_idx, es_idx = final_outer_fit_earlystop_split(outer_train_idx, outer_fold)

            sp = train_predict_ablation(
                architecture,
                record_indices_to_segment_indices(fit_idx),
                record_indices_to_segment_indices(es_idx),
                record_indices_to_segment_indices(outer_test_idx),
                f"seed{base_seed}_outer{outer_fold}_final_fixedsplit"
            )
            rec = aggregate_segment_predictions(sp)
            rec["base_seed"] = int(base_seed)
            rec["outer_fold"] = int(outer_fold)
            rec["architecture"] = label
            rec["prob_raw"] = rec["prob_early_raw"].astype(float)
            seed_parts.append(rec)

            auc = roc_auc_score(rec["label_num"].astype(int), rec["prob_raw"].astype(float))
            arch_seed_fold_rows.append({
                "base_seed": int(base_seed),
                "architecture": label,
                "outer_fold": int(outer_fold),
                "raw_auc": float(auc),
            })

        arch_seed_oof_parts.append(pd.concat(seed_parts, ignore_index=True))

# Restore baseline seed state.
RANDOM_STATE = baseline_random_state
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE); tf.random.set_seed(RANDOM_STATE)

arch_seed_fold_df = pd.DataFrame(arch_seed_fold_rows).sort_values(
    ["architecture","base_seed","outer_fold"]
)

arch_seed_summary_rows = []
for (architecture, base_seed), g in arch_seed_fold_df.groupby(["architecture","base_seed"]):
    arch_seed_summary_rows.append({
        "architecture": architecture,
        "base_seed": int(base_seed),
        "mean_outer_fold_auc": float(g["raw_auc"].mean()),
        "outer_fold_auc_sd": float(g["raw_auc"].std(ddof=1)),
    })
architecture_seed_summary_df = pd.DataFrame(arch_seed_summary_rows)

architecture_seed_variability_df = (
    architecture_seed_summary_df.groupby("architecture", as_index=False)
    .agg(
        mean_across_seeds=("mean_outer_fold_auc","mean"),
        between_seed_sd=("mean_outer_fold_auc","std"),
        min_seed_mean_auc=("mean_outer_fold_auc","min"),
        max_seed_mean_auc=("mean_outer_fold_auc","max"),
    )
)
architecture_seed_variability_df["range"] = (
    architecture_seed_variability_df["max_seed_mean_auc"]
    - architecture_seed_variability_df["min_seed_mean_auc"]
)

# Seed-wise architecture differences (same fixed outer partitions).
arch_seed_pivot = architecture_seed_summary_df.pivot(
    index="base_seed", columns="architecture", values="mean_outer_fold_auc"
).reset_index()
if {"CNN–Transformer RGB","CNN-only RGB","Transformer-only RGB"}.issubset(arch_seed_pivot.columns):
    arch_seed_pivot["Hybrid minus CNN-only"] = (
        arch_seed_pivot["CNN–Transformer RGB"] - arch_seed_pivot["CNN-only RGB"]
    )
    arch_seed_pivot["Hybrid minus Transformer-only"] = (
        arch_seed_pivot["CNN–Transformer RGB"] - arch_seed_pivot["Transformer-only RGB"]
    )

arch_seed_fold_df.to_csv(ARCH_SEED_ROOT/"architecture_seed_fold_auc.csv",index=False)
architecture_seed_summary_df.to_csv(ARCH_SEED_ROOT/"architecture_seed_summary.csv",index=False)
architecture_seed_variability_df.to_csv(ARCH_SEED_ROOT/"architecture_seed_variability.csv",index=False)
arch_seed_pivot.to_csv(ARCH_SEED_ROOT/"architecture_seed_differences.csv",index=False)

# Descriptive distribution of paired architecture AUC differences across the
# five prespecified seeds. Seeds are not treated as independent patients and
# no inferential p-value is computed from these five values.
_seed_diff_cols = [
    c for c in ["Hybrid minus CNN-only","Hybrid minus Transformer-only"]
    if c in arch_seed_pivot.columns
]
_seed_diff_rows = []
for c in _seed_diff_cols:
    vals = pd.to_numeric(arch_seed_pivot[c], errors="coerce").dropna()
    _seed_diff_rows.append({
        "Paired AUC difference": c,
        "n_seeds": int(len(vals)),
        "mean": float(vals.mean()) if len(vals) else np.nan,
        "sd": float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
        "median": float(vals.median()) if len(vals) else np.nan,
        "min": float(vals.min()) if len(vals) else np.nan,
        "max": float(vals.max()) if len(vals) else np.nan,
    })
architecture_seed_difference_summary = pd.DataFrame(_seed_diff_rows)
architecture_seed_difference_summary.to_csv(
    ARCH_SEED_ROOT/"architecture_seed_difference_distribution.csv",
    index=False,
)

if arch_seed_oof_parts:
    pd.concat(arch_seed_oof_parts,ignore_index=True).to_csv(
        ARCH_SEED_ROOT/"cnn_transformer_only_repeated_seed_oof.csv",index=False
    )

display(architecture_seed_summary_df)
display(architecture_seed_variability_df)
display(arch_seed_pivot)
display(architecture_seed_difference_summary)


In [ ]:

# ============================================================
# 25. FIVE PRESPECIFIED PAIRED ARCHITECTURE/FUSION COMPARISONS
#     Fold-stratified pregnancy-cluster bootstrap + Holm
# ============================================================

comparison_oof = master_oof[
    ["record_name","pregnancy_id","label_num","outer_fold",
     "prob_RGB","prob_S1","prob_S2","prob_S3","prob_ensemble_raw"]
].copy()

comparison_oof["cnn_transformer_rgb_raw"] = comparison_oof["prob_RGB"].astype(float)
comparison_oof["equal_all4_raw"] = comparison_oof[
    ["prob_RGB","prob_S1","prob_S2","prob_S3"]
].astype(float).mean(axis=1)
comparison_oof["weighted_all4_raw"] = comparison_oof["prob_ensemble_raw"].astype(float)

comparison_oof = comparison_oof.merge(
    cnn_oof[["record_name","prob_raw"]].rename(columns={"prob_raw":"cnn_only_raw"}),
    on="record_name", how="left", validate="one_to_one"
).merge(
    transformer_oof[["record_name","prob_raw"]].rename(columns={"prob_raw":"transformer_only_raw"}),
    on="record_name", how="left", validate="one_to_one"
)

comparison_specs = [
    ("CNN–Transformer RGB − CNN-only RGB", "cnn_transformer_rgb_raw", "cnn_only_raw"),
    ("CNN–Transformer RGB − Transformer-only RGB", "cnn_transformer_rgb_raw", "transformer_only_raw"),
    ("Weighted RGB+S1+S2+S3 − CNN–Transformer RGB", "weighted_all4_raw", "cnn_transformer_rgb_raw"),
    ("Weighted RGB+S1+S2+S3 − CNN-only RGB", "weighted_all4_raw", "cnn_only_raw"),
    ("Equal-weight RGB+S1+S2+S3 − weighted RGB+S1+S2+S3", "equal_all4_raw", "weighted_all4_raw"),
]

comparison_rows=[]
for i,(name,a,b) in enumerate(comparison_specs):
    res=paired_fold_cluster_bootstrap_delta(
        comparison_oof,a,b,n_boot=N_BOOTSTRAP_PAIRED,seed=20261030+i
    )
    comparison_rows.append({
        "Comparison":name,
        "AUC difference":res["delta_auc"],
        "95% CI low":res["ci95_low"],
        "95% CI high":res["ci95_high"],
        "Unadjusted p":res["two_sided_p"],
    })

paired_comparisons=pd.DataFrame(comparison_rows)

def holm_adjust(p_values):
    p=np.asarray(p_values,dtype=float)
    m=len(p)
    order=np.argsort(p)
    sorted_p=p[order]
    adj_sorted=np.empty(m,dtype=float)
    running=0.0
    for i,val in enumerate(sorted_p):
        corrected=min(1.0,(m-i)*val)
        running=max(running,corrected)
        adj_sorted[i]=min(1.0,running)
    adj=np.empty(m,dtype=float)
    adj[order]=adj_sorted
    return adj

paired_comparisons["Holm-adjusted p"]=holm_adjust(paired_comparisons["Unadjusted p"])
save_table(paired_comparisons,"table_12_paired_bootstrap_holm",directory=ABLATION_ROOT)
display(paired_comparisons)


In [ ]:

# ============================================================
# 26. CWT / RESIZING / NORMALIZATION SENSITIVITY — RGB ONLY
#     Same corrected pregnancy-wise nested protocol.
# ============================================================

class VariantRGBSequence(keras.utils.Sequence):
    def __init__(self, cache_path, variant, segment_indices, labels=labels_all,
                 batch_size=BATCH_SIZE, shuffle=False):
        super().__init__()
        self.cache_path=Path(cache_path)
        self.variant=variant
        self.segment_indices=np.asarray(segment_indices,dtype=int).copy()
        self.labels=np.asarray(labels,dtype=np.float32)
        self.batch_size=int(batch_size)
        self.shuffle=bool(shuffle)
        self.X=np.load(self.cache_path,mmap_mode="r")
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.segment_indices)/self.batch_size))

    def __getitem__(self,batch_index):
        ids=self.segment_indices[
            batch_index*self.batch_size:(batch_index+1)*self.batch_size
        ]
        x=self.X[ids].astype(np.float32)
        if self.variant.endswith("uint8"):
            x=x/255.0
        elif self.variant=="legacy_bilinear_minmax_float16":
            # Exact primary min-max concept without uint8 rounding/quantization.
            x=np.clip(x,0,1).astype(np.float32)
        elif self.variant=="bandlimited_logpower_float16":
            # Per-image, per-channel z standardization; no label/fold information used.
            mean=x.mean(axis=(1,2),keepdims=True)
            std=x.std(axis=(1,2),keepdims=True)
            x=(x-mean)/(std+1e-6)
            x=np.clip(x,-6,6).astype(np.float32)
        y=self.labels[ids].astype(np.float32)
        return x,y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.segment_indices)

def train_predict_sensitivity_variant(
    variant, train_idx, val_idx, test_idx, run_name
):
    root=SENSITIVITY_ROOT/variant
    pred_dir=root/"predictions"; hist_dir=root/"histories"; model_dir=root/"models"
    for p in [pred_dir,hist_dir,model_dir]:
        p.mkdir(parents=True,exist_ok=True)
    pred_path=pred_dir/f"{run_name}_segment_predictions.csv"
    if not FORCE_RETRAIN_SENSITIVITY and pred_path.exists():
        print("Sensitivity resume ->",pred_path)
        return pd.read_csv(pred_path)

    cache_path=SENS_VARIANTS[variant]["path"]
    run_seed=set_run_seed(f"sensitivity_{variant}_{run_name}")
    K.clear_session();gc.collect()
    model=build_a2_cnn_transformer("RGB")
    train_gen=VariantRGBSequence(cache_path,variant,train_idx,labels_all,BATCH_SIZE,True)
    val_gen=VariantRGBSequence(cache_path,variant,val_idx,labels_all,BATCH_SIZE,False)
    test_gen=VariantRGBSequence(cache_path,variant,test_idx,labels_all,BATCH_SIZE,False)
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=PATIENCE,
                                      restore_best_weights=True,verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_auc",mode="max",factor=.5,
                                          patience=2,min_lr=1e-7,verbose=1),
        keras.callbacks.TerminateOnNaN(),
    ]
    hist=model.fit(train_gen,validation_data=val_gen,epochs=EPOCHS,callbacks=callbacks,verbose=1)
    pd.DataFrame(hist.history).to_csv(hist_dir/f"{run_name}_history.csv",index=False)
    p=model.predict(test_gen,verbose=1).reshape(-1)
    lookup=(segment_df.set_index("segment_index").loc[np.asarray(test_idx,dtype=int)].reset_index())
    out=pd.DataFrame({
        "segment_index":np.asarray(test_idx,dtype=int),
        "record_name":lookup["record_name"].astype(str).values,
        "pregnancy_id":lookup["pregnancy_id"].astype(str).values,
        "label_text":lookup["label_text"].astype(str).values,
        "label_num":lookup["label_num"].astype(int).values,
        "prob_early_raw":p.astype(float),
    })
    out.to_csv(pred_path,index=False)
    del model,train_gen,val_gen,test_gen,hist
    K.clear_session();gc.collect()
    return out

def run_sensitivity_variant(variant):
    variant_root=SENSITIVITY_ROOT/variant
    fold_root=variant_root/"fold_results"
    fold_root.mkdir(parents=True,exist_ok=True)
    outer_parts=[]

    for outer_fold,(outer_train_idx,outer_test_idx) in enumerate(outer_splits,start=1):
        fold_dir=fold_root/f"outer_{outer_fold}"
        fold_dir.mkdir(parents=True,exist_ok=True)
        final_path=fold_dir/"outer_record_predictions.csv"
        if not FORCE_RETRAIN_SENSITIVITY and final_path.exists():
            outer_parts.append(pd.read_csv(final_path)); continue

        outer_train_idx=np.asarray(outer_train_idx,dtype=int)
        outer_test_idx=np.asarray(outer_test_idx,dtype=int)
        outer_train_df=record_df.iloc[outer_train_idx].copy()
        inner_cv=GroupKFold(n_splits=N_INNER_SPLITS)
        inner_splits=list(inner_cv.split(
            outer_train_df,outer_train_df["label_num"].astype(int).values,
            outer_train_df["pregnancy_id"].astype(str).values
        ))
        inner_parts=[]
        for inner_fold,(dev_local,eval_local) in enumerate(inner_splits,start=1):
            _,fit_idx,es_idx,eval_idx=make_inner_fit_earlystop_eval_split(
                outer_train_idx,dev_local,eval_local,outer_fold,inner_fold
            )
            sp=train_predict_sensitivity_variant(
                variant,
                record_indices_to_segment_indices(fit_idx),
                record_indices_to_segment_indices(es_idx),
                record_indices_to_segment_indices(eval_idx),
                f"outer{outer_fold}_inner{inner_fold}_fit_es_eval"
            )
            rec=aggregate_segment_predictions(sp);rec["inner_fold"]=inner_fold
            inner_parts.append(rec)

        inner_oof=pd.concat(inner_parts,ignore_index=True)
        y_inner=inner_oof["label_num"].astype(int).to_numpy()
        p_inner=inner_oof["prob_early_raw"].astype(float).to_numpy()
        cal,slope,intercept=fit_positive_platt(y_inner,p_inner)
        # Sensitivity comparisons use held-out RAW ROC-AUC. A non-positive
        # inner-development Platt slope is not allowed to invert class rankings.
        # For this sensitivity fold only, retain the raw score orientation and
        # select the classification threshold using untouched inner RAW scores.
        # This fallback is recorded explicitly and never uses outer-test labels.
        calibration_policy = (
            "positive_platt" if cal is not None
            else "uncalibrated_fallback_nonpositive_slope"
        )
        if cal is None:
            print(
                f"SENSITIVITY CALIBRATION FALLBACK -> {variant}/outer{outer_fold}; "
                f"rejected Platt slope={slope:.6g}. Raw probabilities retained."
            )
        pd.DataFrame([{
            "variant": variant,
            "outer_fold": outer_fold,
            "calibration_policy": calibration_policy,
            "fitted_platt_slope": slope,
            "fitted_platt_intercept": intercept,
            "inner_raw_auc": roc_auc_score(y_inner, p_inner),
            "n_inner_records": len(y_inner),
        }]).to_csv(fold_dir/"sensitivity_calibration_diagnostic.csv", index=False)
        pcal=apply_platt(cal,p_inner)
        thr,_=select_robust_threshold(y_inner,pcal)
        threshold=float(thr["threshold"])

        _,fit_idx,es_idx=final_outer_fit_earlystop_split(outer_train_idx,outer_fold)
        sp=train_predict_sensitivity_variant(
            variant,
            record_indices_to_segment_indices(fit_idx),
            record_indices_to_segment_indices(es_idx),
            record_indices_to_segment_indices(outer_test_idx),
            f"outer{outer_fold}_final_corrected"
        )
        rec=aggregate_segment_predictions(sp)
        rec["prob_raw"]=rec["prob_early_raw"].astype(float)
        rec["prob_calibrated"]=apply_platt(cal,rec["prob_raw"].to_numpy())
        rec["pred_label_num"]=(rec["prob_calibrated"]>=threshold).astype(int)
        rec["outer_fold"]=outer_fold
        rec["variant"]=variant
        rec["threshold"]=threshold
        rec["calibration_policy"]=calibration_policy
        rec["fitted_platt_slope"]=slope
        rec["fitted_platt_intercept"]=intercept
        # In fallback folds, prob_calibrated retains the raw values for
        # backward-compatible downstream schema; calibration_policy discloses it.
        rec.to_csv(final_path,index=False)
        inner_oof.to_csv(fold_dir/"inner_oof_RGB.csv",index=False)
        outer_parts.append(rec)
        del cal;gc.collect()

    oof=(pd.concat(outer_parts,ignore_index=True)
         .sort_values(["outer_fold","record_name"]).reset_index(drop=True))
    oof.to_csv(variant_root/"master_oof_RGB.csv",index=False)
    return oof

sensitivity_oofs={}
if RUN_CWT_SENSITIVITY:
    for variant in SENS_VARIANTS:
        sensitivity_oofs[variant]=run_sensitivity_variant(variant)
else:
    for variant in SENS_VARIANTS:
        sensitivity_oofs[variant]=pd.read_csv(SENSITIVITY_ROOT/variant/"master_oof_RGB.csv")

# Include primary RGB for comparison.
sensitivity_summary_rows=[{
    "Variant":"primary_legacy_bilinear_uint8",
    **auc_summary_from_oof("primary",master_oof.rename(columns={"prob_RGB":"prob_raw"})[
        ["record_name","pregnancy_id","label_num","outer_fold","prob_raw"]
    ])
}]
# Clean nested name field.
sensitivity_summary_rows[0].pop("Configuration",None)

for variant,oof in sensitivity_oofs.items():
    row=auc_summary_from_oof(variant,oof)
    row["Variant"]=variant
    row.pop("Configuration",None)
    sensitivity_summary_rows.append(row)

sensitivity_summary=pd.DataFrame(sensitivity_summary_rows)
save_table(sensitivity_summary,"cwt_normalization_resizing_sensitivity_summary",directory=SENSITIVITY_ROOT)

# Paired comparisons versus primary RGB.
primary_rgb=master_oof[
    ["record_name","pregnancy_id","label_num","outer_fold","prob_RGB"]
].copy().rename(columns={"prob_RGB":"primary_raw"})

sens_delta_rows=[]
for i,(variant,oof) in enumerate(sensitivity_oofs.items()):
    comp=primary_rgb.merge(
        oof[["record_name","prob_raw"]].rename(columns={"prob_raw":"sensitivity_raw"}),
        on="record_name",how="inner",validate="one_to_one"
    )
    res=paired_fold_cluster_bootstrap_delta(
        comp,"sensitivity_raw","primary_raw",n_boot=20000,seed=20261100+i
    )
    sens_delta_rows.append({"Variant":variant,**res})
sensitivity_deltas=pd.DataFrame(sens_delta_rows)
save_table(sensitivity_deltas,"cwt_sensitivity_paired_bootstrap_vs_primary",directory=SENSITIVITY_ROOT)

# Reviewer 3: direct test of dependence on potentially boundary-affected regions.
boundary_effect_comparison = pd.DataFrame()
if (
    "bandlimited_bilinear_uint8" in sensitivity_oofs
    and "bandlimited_boundary_masked_uint8" in sensitivity_oofs
):
    _unmasked = sensitivity_oofs["bandlimited_bilinear_uint8"][
        ["record_name","pregnancy_id","label_num","outer_fold","prob_raw"]
    ].rename(columns={"prob_raw":"unmasked_raw"})
    _masked = sensitivity_oofs["bandlimited_boundary_masked_uint8"][
        ["record_name","prob_raw"]
    ].rename(columns={"prob_raw":"masked_raw"})
    _bc = _unmasked.merge(
        _masked,
        on="record_name",
        how="inner",
        validate="one_to_one",
    )
    _res = paired_fold_cluster_bootstrap_delta(
        _bc,
        "masked_raw",
        "unmasked_raw",
        n_boot=20000,
        seed=20261199,
    )
    boundary_effect_comparison = pd.DataFrame([{
        "Comparison": "boundary-masked minus unmasked 0.08–5.0 Hz RGB",
        **_res,
    }])
    save_table(
        boundary_effect_comparison,
        "cwt_boundary_effect_paired_bootstrap",
        directory=SENSITIVITY_ROOT,
    )

display(sensitivity_summary)
display(sensitivity_deltas)
if len(boundary_effect_comparison):
    display(boundary_effect_comparison)


In [ ]:

# ============================================================
# 27. RESULTS FIGURES — 600-DPI PNG + VECTOR PDF
#     Figure 1–8 + supplementary reviewer figures.
# ============================================================

# -----------------------------
# Figure 1. Abdominal electrode configuration / bipolar derivations
# -----------------------------
fig,ax=plt.subplots(figsize=(6.5,6))
ax.set_aspect("equal")
ax.axis("off")
coords={"E1":(0,1),"E2":(1,1),"E3":(1,0),"E4":(0,0)}
for name,(x,y) in coords.items():
    circ=plt.Circle((x,y),0.08,fill=False,linewidth=2)
    ax.add_patch(circ)
    ax.text(x,y,name,ha="center",va="center",fontsize=11)
# S1 = E2-E1; S2 = E2-E3; S3 = E4-E3
ax.add_patch(FancyArrowPatch((0.1,1.12),(0.9,1.12),arrowstyle="->",mutation_scale=14))
ax.text(.5,1.18,"S1 = E2 − E1",ha="center")
ax.add_patch(FancyArrowPatch((1.12,.9),(1.12,.1),arrowstyle="->",mutation_scale=14))
ax.text(1.20,.5,"S2 = E2 − E3",rotation=-90,va="center")
ax.add_patch(FancyArrowPatch((.9,-.12),(.1,-.12),arrowstyle="->",mutation_scale=14))
ax.text(.5,-.22,"S3 = E4 − E3",ha="center")
ax.set_xlim(-.4,1.45);ax.set_ylim(-.4,1.4)
ax.set_title("ICEHG DS abdominal electrode configuration and bipolar derivations")
save_figure(fig,"figure_01_electrode_configuration")
plt.close(fig)

# -----------------------------
# Figure 2. Proposed workflow
# -----------------------------
fig,ax=plt.subplots(figsize=(14,5))
ax.axis("off")
labels=[
    "ICEHG DS\n126 records / 91 pregnancies",
    "0.08–5 Hz filtering\n120-s windows / 60-s stride",
    "CWT scalograms\nRGB + S1 + S2 + S3",
    "Corrected nested CV\nfit / early-stop / untouched eval",
    "Fusion + Platt + threshold\ninner OOF only",
    "Held-out outer evaluation\nbootstrap / sensitivity / Grad-CAM",
]
xs=np.linspace(.08,.92,len(labels))
for x,label in zip(xs,labels):
    box=FancyBboxPatch((x-.07,.38),.14,.24,boxstyle="round,pad=0.02",fill=False)
    ax.add_patch(box)
    ax.text(x,.50,label,ha="center",va="center",fontsize=9)
for a,b in zip(xs[:-1],xs[1:]):
    ax.add_patch(FancyArrowPatch((a+.07,.50),(b-.07,.50),arrowstyle="->",mutation_scale=14))
ax.set_xlim(0,1);ax.set_ylim(0,1)
ax.set_title("Pregnancy-wise leakage-controlled multi-representation EHG classification framework")
save_figure(fig,"figure_02_framework")
plt.close(fig)

# -----------------------------
# Figure 3. Representative filtered early/later EHG segments
# Deterministic, downstream-model-independent selection used in manuscript.
# -----------------------------
requested_examples=[("icehg1449","Early-recorded"),("icehg1565","Later-recorded")]
available=set(record_meta["record_name"].astype(str))
if not all(name in available for name,_ in requested_examples):
    early_name=record_meta.loc[record_meta["label_num"]==1].sort_values("record_name")["record_name"].iloc[0]
    later_name=record_meta.loc[record_meta["label_num"]==0].sort_values("record_name")["record_name"].iloc[0]
    requested_examples=[(early_name,"Early-recorded"),(later_name,"Later-recorded")]

start_s,stop_s=300,420
i0,i1=int(start_s*FS),int(stop_s*FS)
t=np.arange(i0,i1)/FS
fig,axes=plt.subplots(3,2,figsize=(13,7),sharex=True)
for col,(record_name,period_label) in enumerate(requested_examples):
    filt=np.load(FILTERED_ROOT/f"{record_name}.npz")["filtered"]
    for ch in range(3):
        axes[ch,col].plot(t,filt[i0:i1,ch],linewidth=.6)
        axes[ch,col].set_ylabel(f"S{ch+1}")
        axes[ch,col].grid(alpha=.2)
        if ch==0:
            axes[ch,col].set_title(f"{period_label}: {record_name}")
    axes[-1,col].set_xlabel("Time (s)")
fig.suptitle("Representative 120-s band-pass-filtered EHG segments (300–420 s)")
fig.tight_layout()
save_figure(fig,"figure_03_filtered_ehg_early_later")
plt.close(fig)

# -----------------------------
# Figure 4. Held-out confusion matrix
# -----------------------------
cm=confusion_matrix(master_oof["label_num"].astype(int),master_oof["pred_label_num"].astype(int),labels=[0,1])
fig,ax=plt.subplots(figsize=(5.5,5))
im=ax.imshow(cm)
for i in range(2):
    for j in range(2):
        ax.text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=15)
ax.set_xticks([0,1],["Later","Early"])
ax.set_yticks([0,1],["Later","Early"])
ax.set_xlabel("Predicted recording period")
ax.set_ylabel("True recording period")
ax.set_title("Held-out outer-fold confusion matrix")
fig.tight_layout()
save_figure(fig,"figure_04_confusion_matrix")
plt.close(fig)

# -----------------------------
# Figure 5. Pooled ROC curves — secondary descriptive visualization
# -----------------------------
roc_scores={
    "RGB":"prob_RGB",
    "S1":"prob_S1",
    "S2":"prob_S2",
    "S3":"prob_S3",
    "Weighted ensemble":"prob_ensemble_raw",
}
fig,ax=plt.subplots(figsize=(7,6))
y=master_oof["label_num"].astype(int).to_numpy()
for name,col in roc_scores.items():
    fpr,tpr,_=roc_curve(y,master_oof[col].astype(float))
    auc=roc_auc_score(y,master_oof[col].astype(float))
    ax.plot(fpr,tpr,label=f"{name} (pooled AUC={auc:.3f})")
ax.plot([0,1],[0,1],"--",linewidth=1)
ax.set_xlabel("False-positive rate")
ax.set_ylabel("True-positive rate")
ax.set_title("Cross-validated pooled ROC curves (secondary descriptive analysis)")
ax.legend(fontsize=8)
ax.grid(alpha=.2)
fig.tight_layout()
save_figure(fig,"figure_05_pooled_roc_secondary")
plt.close(fig)

# -----------------------------
# Figure 6. Reliability / calibration
# -----------------------------
fig,ax=plt.subplots(figsize=(6.5,6))
ax.plot([0,1],[0,1],"--",label="Perfect calibration")
ax.plot(raw_rel["mean_predicted"],raw_rel["observed_fraction"],marker="o",
        label=f"Raw (ECE={raw_ece:.3f})")
ax.plot(cal_rel["mean_predicted"],cal_rel["observed_fraction"],marker="o",
        label=f"Platt-calibrated (ECE={cal_ece:.3f})")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed early-recorded fraction")
ax.set_title("Reliability diagram")
ax.legend()
ax.grid(alpha=.2)
fig.tight_layout()
save_figure(fig,"figure_06_reliability_calibration")
plt.close(fig)

# -----------------------------
# Figure 7. Architecture-ablation ROC + weighted ensemble reference
# -----------------------------
arch_plot=master_oof[
    ["record_name","label_num","prob_ensemble_raw","prob_RGB"]
].merge(
    cnn_oof[["record_name","prob_raw"]].rename(columns={"prob_raw":"cnn"}),
    on="record_name",validate="one_to_one"
).merge(
    transformer_oof[["record_name","prob_raw"]].rename(columns={"prob_raw":"transformer"}),
    on="record_name",validate="one_to_one"
)
fig,ax=plt.subplots(figsize=(7,6))
for name,col in [
    ("CNN-only RGB","cnn"),
    ("Transformer-only RGB","transformer"),
    ("CNN–Transformer RGB","prob_RGB"),
    ("Weighted ensemble reference","prob_ensemble_raw"),
]:
    fpr,tpr,_=roc_curve(arch_plot["label_num"],arch_plot[col])
    auc=roc_auc_score(arch_plot["label_num"],arch_plot[col])
    ax.plot(fpr,tpr,label=f"{name} (pooled AUC={auc:.3f})")
ax.plot([0,1],[0,1],"--",linewidth=1)
ax.set_xlabel("False-positive rate");ax.set_ylabel("True-positive rate")
ax.set_title("Architecture-ablation pooled ROC curves (secondary descriptive analysis)")
ax.legend(fontsize=8);ax.grid(alpha=.2)
fig.tight_layout()
save_figure(fig,"figure_07_architecture_ablation_roc")
plt.close(fig)

# -----------------------------
# Figure 8. Grad-CAM — score-based, heatmap-independent median selection
# -----------------------------
def select_gradcam_records(df):
    selected=[]
    for label_num, group_name in [(1,"Early-recorded"),(0,"Later-recorded")]:
        g=df.loc[df["label_num"].astype(int)==label_num].copy()
        med=float(g["prob_RGB"].astype(float).median())
        g["distance_to_class_median_rgb"]=np.abs(g["prob_RGB"].astype(float)-med)
        r=g.sort_values(["distance_to_class_median_rgb","record_name"]).iloc[0].copy()
        r["selection_group"]=group_name
        r["class_median_rgb_score"]=med
        selected.append(r)
    return pd.DataFrame(selected)

gradcam_selected=select_gradcam_records(master_oof)
save_table(gradcam_selected,"table_gradcam_selected_records",directory=GRADCAM_ROOT)

def gradcam_for_image(model,image,layer_name="token_projection"):
    grad_model=keras.Model(model.inputs,[model.get_layer(layer_name).output,model.output])
    x=tf.convert_to_tensor(image[None,...],dtype=tf.float32)
    with tf.GradientTape() as tape:
        conv_out,pred=grad_model(x,training=False)
        score=pred[:,0]
    grads=tape.gradient(score,conv_out)
    weights=tf.reduce_mean(grads,axis=(1,2),keepdims=True)
    heat=tf.reduce_sum(weights*conv_out,axis=-1)
    heat=tf.nn.relu(heat)[0]
    heat=heat/(tf.reduce_max(heat)+1e-8)
    heat=tf.image.resize(heat[...,None],(IMAGE_SIZE,IMAGE_SIZE),method="bilinear")[...,0]
    return heat.numpy(),float(pred.numpy().ravel()[0])

rgb_cache=np.load(SCALOGRAM_PATHS["RGB"],mmap_mode="r")
fig,axes=plt.subplots(1,2,figsize=(16,7),sharex=True,sharey=True)
axes=np.atleast_1d(axes).ravel()

gradcam_detail_rows=[]
for ax,(_,row) in zip(axes,gradcam_selected.iterrows()):
    fold=int(row["outer_fold"])
    model_path=MODEL_DIR/"RGB"/f"outer{fold}_final_corrected_best.keras"
    model=keras.models.load_model(
        model_path,
        custom_objects={"AddPositionEmbedding":AddPositionEmbedding},
        compile=False,
    )
    seg_pred_path=PRED_DIR/"RGB"/f"outer{fold}_final_corrected_segment_predictions.csv"
    seg_pred=pd.read_csv(seg_pred_path)
    rec_seg=seg_pred.loc[seg_pred["record_name"].astype(str)==str(row["record_name"])].copy()
    # Segment selection is score-based and independent of heatmap appearance:
    # choose segment probability closest to its held-out record-level RGB mean.
    rec_seg["distance_to_record_mean"]=np.abs(
        rec_seg["prob_early_raw"].astype(float)-float(row["prob_RGB"])
    )
    chosen=rec_seg.sort_values(["distance_to_record_mean","segment_index"]).iloc[0]
    idx=int(chosen["segment_index"])
    image=rgb_cache[idx].astype(np.float32)/255.0
    heat,pred=gradcam_for_image(model,image,"token_projection")

    ax.imshow(image,extent=[0,120,128,1],aspect="auto")
    ax.imshow(heat,alpha=.45,cmap="jet",extent=[0,120,128,1],aspect="auto")
    ax.set_xlabel("Time (s)", fontsize=11)
    ticks=np.array([1,32,64,96,128],dtype=int)
    pseudo=pywt.scale2frequency(WAVELET,ticks)/SAMPLING_PERIOD
    ax.set_yticks(ticks,[f"{s}\n{f:.2f} Hz" for s,f in zip(ticks,pseudo)])
    ax.set_ylabel("Scale / nominal pseudo-frequency", fontsize=11)
    ax.tick_params(axis="both", labelsize=9)
    ax.set_title(
        f"{row['selection_group']}: {row['record_name']} | fold {fold}\n"
        f"record RGB p={row['prob_RGB']:.3f}, segment p={pred:.3f}"
    )
    gradcam_detail_rows.append({
        "selection_group":row["selection_group"],
        "record_name":row["record_name"],
        "outer_fold":fold,
        "label_num":int(row["label_num"]),
        "record_rgb_score":float(row["prob_RGB"]),
        "class_median_rgb_score":float(row["class_median_rgb_score"]),
        "segment_index":idx,
        "selection_rule":"record score closest to true-class median RGB score; segment score closest to record RGB mean",
        "heatmap_used_for_selection":False,
        "model_path":str(model_path)
    })
    del model;K.clear_session();gc.collect()

fig.suptitle("Grad-CAM examples selected by a score-based, heatmap-independent protocol", fontsize=14)
fig.tight_layout()
save_figure(fig,"figure_08_gradcam_score_based_examples")
plt.close(fig)
save_table(pd.DataFrame(gradcam_detail_rows),"gradcam_selection_audit",directory=GRADCAM_ROOT)

# -----------------------------
# Supplementary Figure S1. Paired within-pregnancy score differences
# -----------------------------
fig,ax=plt.subplots(figsize=(7,6))
for _,r in paired_df.sort_values("raw_difference_early_minus_later").iterrows():
    ax.plot([0,1],[r["later_raw_score"],r["early_raw_score"]],marker="o",alpha=.6)
ax.set_xticks([0,1],["Later recording","Early recording"])
ax.set_ylabel("Raw weighted-ensemble early-class probability")
ax.set_title("Within-pregnancy paired score ordering (35 pregnancies)")
ax.grid(alpha=.2)
fig.tight_layout()
save_figure(fig,"figure_S01_paired_score_ordering")
plt.close(fig)

# Supplementary Figure S2. Repeated-seed robustness
if "seed_summary_df" in globals():
    fig,ax=plt.subplots(figsize=(7,5))
    ax.errorbar(
        seed_summary_df["base_seed"].astype(str),
        seed_summary_df["mean_outer_fold_auc"],
        yerr=seed_summary_df["outer_fold_auc_sd"],
        marker="o",capsize=4
    )
    ax.set_xlabel("Prespecified base seed")
    ax.set_ylabel("Mean outer-fold raw ROC-AUC ± fold SD")
    ax.set_title("Fixed-split repeated-seed sensitivity")
    ax.grid(alpha=.2)
    fig.tight_layout()
    save_figure(fig,"figure_S02_repeated_seed_robustness")
    plt.close(fig)

# Supplementary Figure S3. CWT/normalization/resizing sensitivity
fig,ax=plt.subplots(figsize=(9,5.5))
x=np.arange(len(sensitivity_summary))
ax.errorbar(
    x,
    sensitivity_summary["Mean outer-fold raw ROC-AUC"],
    yerr=sensitivity_summary["Outer-fold AUC SD"],
    fmt="o",capsize=5
)
ax.set_xticks(x,sensitivity_summary["Variant"],rotation=25,ha="right")
ax.set_ylabel("Mean outer-fold raw ROC-AUC ± fold SD")
ax.set_title("CWT, resizing, and representation sensitivity (RGB)")
ax.grid(alpha=.2)
fig.tight_layout()
save_figure(fig,"figure_S03_cwt_normalization_resizing_sensitivity")
plt.close(fig)

# Supplementary Figure S4. Delivery-mode exploratory AUCs
if len(delivery_summary):
    fig,ax=plt.subplots(figsize=(7,5))
    ax.errorbar(
        delivery_summary["Delivery mode"],
        delivery_summary["Mean outer-fold raw ROC-AUC"],
        yerr=delivery_summary["Outer-fold AUC SD"],
        fmt="o",capsize=5
    )
    ax.set_ylabel("Mean valid outer-fold raw ROC-AUC")
    ax.set_title("Exploratory delivery-mode subgroup discrimination")
    ax.tick_params(axis="x",rotation=20)
    ax.grid(alpha=.2)
    fig.tight_layout()
    save_figure(fig,"figure_S04_delivery_mode_auc")
    plt.close(fig)


# Supplementary Figure S5. Architecture repeated-seed variability
if "architecture_seed_summary_df" in globals() and len(architecture_seed_summary_df):
    fig,ax=plt.subplots(figsize=(8,5.5))
    for architecture,g in architecture_seed_summary_df.groupby("architecture"):
        ax.plot(g["base_seed"].astype(str),g["mean_outer_fold_auc"],marker="o",label=architecture)
    ax.set_xlabel("Prespecified base seed")
    ax.set_ylabel("Mean outer-fold raw ROC-AUC")
    ax.set_title("Architecture-ablation repeated-seed sensitivity")
    ax.legend(fontsize=8)
    ax.grid(alpha=.2)
    fig.tight_layout()
    save_figure(fig,"figure_S05_architecture_repeated_seed")
    plt.close(fig)


# Supplementary Figure S6. CWT/scalogram representation variants
rep_row=record_meta.sort_values(["period","record_name"]).iloc[0]
seg_idx=int(segment_df.loc[segment_df["record_name"]==rep_row.record_name,"segment_index"].iloc[0])
primary_rgb=np.load(SCALOGRAM_PATHS["RGB"],mmap_mode="r")[seg_idx].astype(np.float32)/255.0
images=[("Primary legacy\nbilinear uint8",primary_rgb)]
for variant in ["legacy_bilinear_minmax_float16","legacy_bicubic_uint8",
                "bandlimited_bilinear_uint8","bandlimited_logpower_float16",
                "bandlimited_boundary_masked_uint8"]:
    arr=np.load(SENS_VARIANTS[variant]["path"],mmap_mode="r")[seg_idx].astype(np.float32)
    if variant.endswith("uint8"):
        arr=arr/255.0
    else:
        lo=np.nanpercentile(arr,1);hi=np.nanpercentile(arr,99)
        arr=np.clip((arr-lo)/(hi-lo+1e-6),0,1)
    images.append((variant.replace("_","\n"),arr))
fig,axes=plt.subplots(1,6,figsize=(22,4.5))
for ax,(title,img) in zip(axes,images):
    ax.imshow(img)
    ax.set_title(title,fontsize=8)
    ax.set_xlabel("Time")
    ax.set_ylabel("CWT row")
fig.suptitle("CWT/scalogram sensitivity representations for the same EHG segment")
fig.tight_layout()
save_figure(fig,"figure_S06_scalogram_variants")
plt.close(fig)

print("All manuscript and supplementary figures generated.")


In [ ]:

# ============================================================
# 27A. MANUSCRIPT-READY TABLES FROM THE SAME FINAL RUN
# ============================================================

# Table 2 — CWT / representation parameters
table_02=pd.DataFrame([
    ["Sampling frequency",f"{FS:g} Hz"],
    ["Input band-pass",f"{FILTER_LOW_HZ:.2f}–{FILTER_HIGH_HZ:.1f} Hz; order {FILTER_ORDER}"],
    ["Segmentation",f"{WINDOW_SECONDS} s windows; {STRIDE_SECONDS} s stride (50% overlap)"],
    ["Primary wavelet",f"Morlet ({WAVELET})"],
    ["Primary scales","128 integer scales, 1–128"],
    ["Primary CWT computation","PyWavelets convolution (`conv`)"],
    ["Primary scalogram value","Wavelet power |CWT|²"],
    ["Primary image normalization","Per-segment/channel min–max -> uint8"],
    ["Primary resize",f"PIL bilinear -> {IMAGE_SIZE}×{IMAGE_SIZE}"],
    ["Primary model input","float32 / 255"],
    ["Quantization sensitivity","Legacy scales/bilinear with float16 [0,1] cache; no uint8 rounding"],
    ["Resize sensitivity","Legacy scales/uint8 with bicubic instead of bilinear"],
    ["Sensitivity physical frequency grid",f"{N_FREQUENCIES} geometrically spaced frequencies, {FILTER_LOW_HZ:.2f}–{FILTER_HIGH_HZ:.1f} Hz"],
    ["Sensitivity float representation","log1p(|CWT|²), float16 cache, per-image/channel z standardization"],
    ["Boundary-effect sensitivity","0.08–5.0 Hz physical Morlet grid; scale-dependent support mask; masked coefficients set to common neutral value before bilinear resize"],
],columns=["Parameter","Value"])
save_table(table_02,"table_02_cwt_parameters_final")

# Table 3 — architecture/training settings
table_03=pd.DataFrame([
    ["Optimizer","Adam"],
    ["Initial learning rate",LEARNING_RATE],
    ["Batch size",BATCH_SIZE],
    ["Maximum epochs",EPOCHS],
    ["Early-stopping patience",PATIENCE],
    ["Loss","Binary cross-entropy"],
    ["Dropout",DROPOUT],
    ["L2 coefficient",L2_WEIGHT],
    ["Transformer embedding",EMBED_DIM],
    ["Attention heads",NUM_HEADS],
    ["Feed-forward dimension",FF_DIM],
    ["Transformer blocks",NUM_TRANSFORMER_BLOCKS],
    ["Numeric precision","float32"],
],columns=["Setting","Value"])
save_table(table_03,"table_03_architecture_training_settings_final")

# Table 4 — validation design
table_04=pd.DataFrame([
    ["Outer folds",N_OUTER_SPLITS],
    ["Inner evaluation folds",N_INNER_SPLITS],
    ["Grouping unit","Pregnancy"],
    ["Inner development split","Pregnancy-disjoint fit + early-stop"],
    ["Inner configuration predictions","Untouched inner-evaluation pregnancies only"],
    ["Fusion/calibration/threshold selection","Corrected untouched inner OOF only"],
    ["Outer test use","Exactly once for final held-out evaluation"],
    ["Primary AUC summary","Arithmetic mean of five outer-fold raw ROC-AUC values"],
    ["Pooled AUC","Secondary descriptive only"],
],columns=["Validation item","Specification"])
save_table(table_04,"table_04_nested_validation_settings_final")

# Table 5 — ablation/sensitivity configurations
table_05=pd.DataFrame([
    ["CNN-only RGB","Residual CNN + SE; no token projection/positional embedding/Transformer; GAP -> dense head"],
    ["Transformer-only RGB","32×32 non-overlapping RGB patches -> 49 tokens; positional embedding + 2 Transformer blocks -> dense head"],
    ["CNN–Transformer RGB","Residual CNN + SE -> 7×7 feature map -> token projection -> positional embedding + 2 Transformer blocks"],
    ["Equal RGB+S1+S2","Arithmetic mean of RGB, S1, S2 raw probabilities"],
    ["Equal all four","Arithmetic mean of RGB, S1, S2, S3 raw probabilities"],
    ["Optimized all four","Non-negative weights; each 0.10–0.50 in 0.05 increments; sum=1; selected from corrected inner OOF"],
    ["Boundary-masked RGB sensitivity","0.08–5.0 Hz Morlet RGB with scale-dependent finite-support boundary masking; same pregnancy-wise validation policy"],
],columns=["Configuration","Definition"])
save_table(table_05,"table_05_ablation_and_fusion_configurations_final")

# Table 6 — representation/fusion discrimination
table_06=auc_summary_corrected_df.copy()
save_table(table_06,"table_06_representation_fusion_auc_final")

# Table 7 — fold-specific selected configuration and held-out performance
table_07=selected_configs_df.merge(
    fold_metrics_df,
    on="outer_fold",how="left",suffixes=("_config","_metric")
)
save_table(table_07,"table_07_fold_configuration_and_performance_final")

# Reviewer 3 Comment 4 — explicit selected Platt parameters by outer fold.
_platt_cols = [
    "outer_fold","w_RGB","w_S1","w_S2","w_S3",
    "platt_slope","platt_intercept","threshold",
]
table_S_platt = selected_configs_df[
    [c for c in _platt_cols if c in selected_configs_df.columns]
].copy()
save_table(
    table_S_platt,
    "table_S_platt_parameters_by_outer_fold",
)

# Confusion matrix table
conf_table=pd.DataFrame(
    cm,index=["True later","True early"],columns=["Predicted later","Predicted early"]
)
save_table(conf_table,"table_confusion_matrix_final",index=True)

# Tables 8–12 from revised results sequence.
save_table(calibration_summary,"table_08_calibration_final")
save_table(table_09,"table_09_bootstrap_final")
save_table(table_10,"table_10_paired_singleton_final")
save_table(architecture_summary,"table_11_architecture_ablation_final")
save_table(paired_comparisons,"table_12_paired_comparisons_holm_final")

# Grad-CAM selections are retained as a reproducibility/support table, not assigned
# a main-manuscript table number because the literature-comparison table follows later.
gradcam_support=gradcam_selected[["selection_group","record_name","outer_fold","label_num","prob_RGB","class_median_rgb_score"]].copy()
save_table(gradcam_support,"gradcam_selected_records_support")
print("Manuscript-ready Tables 2–12 generated from this run.")


## Statistical and interpretive guardrails

- Confidence intervals are **not Holm-adjusted**; Holm correction applies only to the five prespecified paired-comparison p-values.
- Architecture configurations are **not parameter matched**. Parameter counts are reported, and any architecture comparison should be described as a comparison of the tested configurations rather than proof of an isolated architectural causal effect.
- CWT/normalization/resizing alternatives are **prespecified sensitivity analyses**. They are not used to select or replace the primary representation based on outer-test performance.
- The 35-pregnancy matched analysis is a sensitivity analysis of within-pregnancy score ordering; it does not establish a physiological gestational trajectory.
- Delivery-mode analyses are exploratory because some subgroup/fold combinations are small.
- The signal-integrity audit detects gross technical abnormalities only; it cannot establish the absence of subtle acquisition artifacts.
- Grad-CAM retains positive contributions after ReLU. Record and segment selection is score-based and heatmap-independent.


In [ ]:

# ============================================================
# 28. FINAL TABLE WORKBOOK, REVIEWER-COVERAGE MANIFEST,
#     REPRODUCIBILITY AUDIT, ZENODO EXPORT
# ============================================================

# Final key-result table.
primary_metrics = fixed_decision_metrics(master_oof)
primary_auc_record = primary_ci_df.iloc[0]

final_key_results = pd.DataFrame([
    {"Result":"Records","Value":len(master_oof)},
    {"Result":"Pregnancies","Value":master_oof["pregnancy_id"].nunique()},
    {"Result":"Accuracy","Value":primary_metrics["accuracy"]},
    {"Result":"Sensitivity","Value":primary_metrics["sensitivity"]},
    {"Result":"Specificity","Value":primary_metrics["specificity"]},
    {"Result":"F1","Value":primary_metrics["f1"]},
    {"Result":"Balanced accuracy","Value":primary_metrics["balanced_accuracy"]},
    {"Result":"Primary mean outer-fold raw ROC-AUC","Value":float(primary_auc_record["mean_outer_fold_auc_primary"])},
    {"Result":"Outer-fold AUC SD","Value":float(primary_auc_record["outer_fold_auc_sd"])},
    {"Result":"Primary AUC CI low","Value":float(primary_auc_record["ci95_low"])},
    {"Result":"Primary AUC CI high","Value":float(primary_auc_record["ci95_high"])},
    {"Result":"Secondary pooled raw ROC-AUC","Value":float(primary_auc_record["pooled_auc_secondary"])},
])
save_table(final_key_results,"FINAL_key_results")

# Reviewer 3 auxiliary audit tables — no additional model fitting.
# These make already-computed information explicit for the response letter.

# Fold composition + principal comparator AUCs.
_fold_comp_rows = []
for fold, g in sorted(master_oof.groupby("outer_fold")):
    row = {
        "outer_fold": int(fold),
        "test_records": int(len(g)),
        "test_pregnancies": int(g["pregnancy_id"].nunique()),
        "early_records": int((g["label_num"].astype(int) == 1).sum()),
        "later_records": int((g["label_num"].astype(int) == 0).sum()),
        "optimized_ensemble_auc": float(
            roc_auc_score(g["label_num"].astype(int), g["prob_ensemble_raw"].astype(float))
        ),
    }
    # Architecture AUCs are already computed in architecture_summary; copy them here.
    for arch_name in ["CNN-only RGB","Transformer-only RGB","CNN–Transformer RGB"]:
        _a = architecture_summary.loc[
            architecture_summary["Configuration"] == arch_name,
            f"Fold {int(fold)} AUC",
        ]
        if len(_a):
            row[arch_name + " AUC"] = float(_a.iloc[0])
    _fold_comp_rows.append(row)
reviewer3_fold_composition = pd.DataFrame(_fold_comp_rows)
save_table(reviewer3_fold_composition, "reviewer3_fold_composition_and_auc_audit")

# Fusion-search specification, including the distance used for equal-weight proximity.
fusion_search_spec = pd.DataFrame([
    ["Weight nonnegativity", True],
    ["Per-representation weight minimum", MIN_WEIGHT],
    ["Per-representation weight maximum", MAX_WEIGHT],
    ["Weight increment", WEIGHT_STEP],
    ["Sum-to-one constraint", 1.0],
    ["Admissible candidate vectors", 375],
    ["Equal-weight reference", "(0.25, 0.25, 0.25, 0.25)"],
    ["Proximity metric", "Squared Euclidean (squared L2) distance to equal-weight vector"],
    ["Shortlisted candidates", TOP_WEIGHT_CANDIDATES],
    ["Threshold range", f"{THRESHOLD_MIN:.3f}–{THRESHOLD_MAX:.3f}"],
    ["Threshold increment", THRESHOLD_STEP],
    ["Calibration", f"Platt logistic regression, C={PLATT_C}, lbfgs, max_iter=2000"],
    ["Rejected calibration", "Candidate discarded when fitted Platt slope <= 0"],
], columns=["Fusion-search item","Specification"])
save_table(fusion_search_spec, "reviewer3_fusion_search_specification")

# Explicit dataset / seeds manifest.
dataset_seed_manifest = pd.DataFrame([
    ["Dataset", "PhysioNet ICEHG DS"],
    ["Dataset version", "1.0.1"],
    ["Dataset source", PHYSIONET_URL],
    ["Primary random state", RANDOM_STATE],
    ["Repeated seeds", ",".join(map(str, REPEATED_SEEDS))],
    ["Outer folds", N_OUTER_SPLITS],
    ["Inner folds", N_INNER_SPLITS],
    ["Grouping unit", "pregnancy"],
    ["Window / stride", f"{WINDOW_SECONDS}s / {STRIDE_SECONDS}s"],
], columns=["Item","Value"])
save_table(dataset_seed_manifest, "reviewer3_dataset_and_seed_manifest")

# Primary training-history / computational-burden summary.
# This uses already saved histories/models and does not retrain anything.
_primary_history_root = PROJECT_ROOT / "histories"
_primary_history_files = sorted(_primary_history_root.rglob("*_history.csv"))
_epoch_counts = []
for _hp in _primary_history_files:
    try:
        _h = pd.read_csv(_hp)
        _epoch_counts.append({
            "history_file": str(_hp.relative_to(PROJECT_ROOT)),
            "epochs_trained": int(len(_h)),
        })
    except Exception:
        pass
primary_training_histories = pd.DataFrame(_epoch_counts)
if len(primary_training_histories):
    save_table(primary_training_histories, "primary_training_history_epochs")
    primary_training_summary = pd.DataFrame([{
        "training_trajectories_with_history": int(len(primary_training_histories)),
        "mean_epochs": float(primary_training_histories["epochs_trained"].mean()),
        "median_epochs": float(primary_training_histories["epochs_trained"].median()),
        "min_epochs": int(primary_training_histories["epochs_trained"].min()),
        "max_epochs": int(primary_training_histories["epochs_trained"].max()),
    }])
else:
    primary_training_summary = pd.DataFrame([{
        "training_trajectories_with_history": 0,
        "mean_epochs": np.nan,
        "median_epochs": np.nan,
        "min_epochs": np.nan,
        "max_epochs": np.nan,
    }])
save_table(primary_training_summary, "primary_training_history_summary")

# Parameter counts for the four primary representation-specific hybrid streams.
_stream_param_rows = []
for _rep in REPRESENTATIONS:
    K.clear_session()
    _m = build_a2_cnn_transformer(_rep)
    _stream_param_rows.append({
        "representation": _rep,
        "parameters": int(_m.count_params()),
    })
    del _m
    K.clear_session()
primary_stream_parameters = pd.DataFrame(_stream_param_rows)
save_table(primary_stream_parameters, "primary_stream_parameter_counts")
primary_ensemble_parameter_summary = pd.DataFrame([{
    "four_stream_total_parameters": int(primary_stream_parameters["parameters"].sum())
}])
save_table(primary_ensemble_parameter_summary, "primary_four_stream_parameter_total")

# Archived final model-file sizes by outer fold.
_model_size_rows = []
for _fold in range(1, N_OUTER_SPLITS + 1):
    _row = {"outer_fold": _fold}
    _total = 0
    for _rep in REPRESENTATIONS:
        _mp = MODEL_DIR / _rep / f"outer{_fold}_final_corrected_best.keras"
        _bytes = int(_mp.stat().st_size) if _mp.exists() else np.nan
        _row[f"{_rep}_bytes"] = _bytes
        if np.isfinite(_bytes):
            _total += int(_bytes)
    _row["four_stream_total_bytes"] = _total if _total > 0 else np.nan
    _row["four_stream_total_MB"] = (_total / (1024**2)) if _total > 0 else np.nan
    _model_size_rows.append(_row)
primary_model_size_by_fold = pd.DataFrame(_model_size_rows)
save_table(primary_model_size_by_fold, "primary_final_model_size_by_fold")


# Reviewer-coverage manifest.
coverage = pd.DataFrame([
    ["Paired pregnancies / within-pregnancy ordering","paired_pregnancy_summary.csv; table_10_paired_singleton_performance.csv","Completed"],
    ["Maternal/acquisition characteristics by period and pairing status","12_metadata_confounder/table_S_metadata_characteristics_period_by_pairing_status.csv; table_S_categorical_characteristics_period_by_pairing_status.csv","Completed"],
    ["Clinical/acquisition confounding","12_metadata_confounder/*; table_S1/S2/S3","Completed"],
    ["Mean outer-fold AUC instead of pooled primary","corrected_auc_summary.csv / FINAL_key_results.csv","Completed"],
    ["Separate early-stop vs untouched inner evaluation","fold_results/*/inner_threeway_split_audit.csv","Completed"],
    ["CWT frequency alignment sensitivity","10_sensitivity/cwt_normalization_resizing_sensitivity_summary.csv","Completed"],
    ["CWT large-scale boundary-effect sensitivity","10_sensitivity/table_S_cwt_boundary_support_audit.csv; 10_sensitivity/cwt_boundary_effect_paired_bootstrap.csv","Completed"],
    ["Normalization/uint8/resizing sensitivity","10_sensitivity/*","Completed"],
    ["Simple signal baseline","11_simple_baselines/*","Completed"],
    ["Training-history / model complexity summary","07_tables/primary_training_history_summary.csv; primary_stream_parameter_counts.csv; primary_final_model_size_by_fold.csv","Completed"],
    ["Repeated-seed stochasticity","13_repeated_seed/*","Completed"],
    ["Fusion transparency / equal weighting","primary fusion tables + selected configurations","Completed"],
    ["Architecture ablation","14_architecture_ablation/*","Completed"],
    ["Architecture ablation seed robustness","14_architecture_ablation/repeated_seed/*","Completed"],
    ["Seed-wise paired AUC-difference distribution","14_architecture_ablation/repeated_seed/architecture_seed_difference_distribution.csv","Completed"],
    ["Paired bootstrap + Holm","14_architecture_ablation/table_12_paired_bootstrap_holm.csv","Completed"],
    ["Grad-CAM score-based selection + readable axes","15_gradcam/*; figure_08_gradcam_score_based_examples.*","Completed"],
    ["Reproducibility / Zenodo package","16_zenodo_export/*","Completed"],
],columns=["Reviewer issue","Notebook outputs","Status"])
save_table(coverage,"reviewer_coverage_manifest")


# Reviewer 3 response-ready output index.
reviewer3_output_index = pd.DataFrame([
    ["Comment 1", "07_tables/inner_threeway_split_audit_all_folds.csv", "Fit / early-stop / untouched inner-evaluation roles"],
    ["Comment 2", "10_sensitivity/cwt_normalization_resizing_sensitivity_summary.csv", "Bandwidth-constrained CWT results"],
    ["Comment 2", "10_sensitivity/table_S_cwt_boundary_support_audit.csv", "Scale-dependent boundary support audit"],
    ["Comment 2", "10_sensitivity/cwt_boundary_effect_paired_bootstrap.csv", "Boundary-masked vs unmasked direct comparison"],
    ["Comment 3", "environment_versions.json; pip_freeze.txt; 16_zenodo_export/output_manifest.csv", "Pinned/reported environment and archive manifest"],
    ["Comment 4", "07_tables/table_S_platt_parameters_by_outer_fold.csv", "Selected Platt slope/intercept by outer fold"],
    ["Comment 4", "07_tables/reviewer3_fusion_search_specification.csv", "Complete fusion grid / L2 tie-break specification"],
    ["Comment 5", "14_architecture_ablation/repeated_seed/*", "Architecture-specific repeated-seed robustness"],
    ["Comment 5", "14_architecture_ablation/repeated_seed/architecture_seed_difference_distribution.csv", "Distribution of paired seed-wise AUC differences"],
    ["Comment 5", "07_tables/reviewer3_fold_composition_and_auc_audit.csv", "Per-fold composition and principal comparator AUCs"],
    ["Comment 6", "paired_pregnancy_summary.csv; table_10_paired_singleton_performance.csv", "35 paired pregnancies"],
    ["Comment 6", "12_metadata_confounder/table_S_metadata_characteristics_period_by_pairing_status.csv", "Maternal/acquisition characteristics by period and paired/singleton status"],
    ["Comment 7", "11_simple_baselines/*", "Welch spectral LR/SVM/RF baselines"],
    ["Minor", "08_figures/figure_08_gradcam_score_based_examples.png/.pdf", "Enlarged Grad-CAM with readable time and scale/pseudo-frequency axes"],
], columns=["Reviewer 3 item","Output","Purpose"])
save_table(reviewer3_output_index, "reviewer3_output_index")

# Combine Table S3A + S3B into an Excel-friendly form while keeping CSVs.
table_s3_note = pd.DataFrame([{
    "Note": (
        "The signal-integrity audit detects gross technical abnormalities but does not demonstrate "
        "the absence of subtler acquisition artifacts. Pair_RecID absence in singleton records is structural. "
        "The public ICEHG DS does not provide explicit acquisition-batch identifiers, equipment/device-state "
        "variables, or expert-labeled artifact counts."
    )
}])
save_table(table_s3_note,"table_S3_note")

# Export all CSV tables into one workbook.
workbook_path = EXPORT_DIR / "ICEHG_FINAL_ALL_TABLES.xlsx"
csv_sources = []
for root in [TABLE_DIR, META_ROOT, BASELINE_ROOT, SENSITIVITY_ROOT, ABLATION_ROOT, GRADCAM_ROOT, REPEAT_ROOT]:
    csv_sources.extend(sorted(Path(root).rglob("*.csv")))

used_sheet_names=set()
with pd.ExcelWriter(workbook_path,engine="openpyxl") as writer:
    for path in csv_sources:
        try:
            df=pd.read_csv(path)
        except Exception:
            continue
        stem=path.stem[:31]
        sheet=stem
        k=1
        while sheet in used_sheet_names:
            suffix=f"_{k}"
            sheet=(stem[:31-len(suffix)]+suffix)
            k+=1
        used_sheet_names.add(sheet)
        df.to_excel(writer,sheet_name=sheet,index=False)

# Structural reproducibility checks.
checks = []
def add_check(name, condition, detail=""):
    checks.append({"Check":name,"Pass":bool(condition),"Detail":str(detail)})

add_check("126 records",len(master_oof)==126,len(master_oof))
add_check("91 pregnancies",master_oof["pregnancy_id"].nunique()==91,master_oof["pregnancy_id"].nunique())
add_check("62 early records",int((master_oof["label_num"]==1).sum())==62,int((master_oof["label_num"]==1).sum()))
add_check("64 later records",int((master_oof["label_num"]==0).sum())==64,int((master_oof["label_num"]==0).sum()))
add_check("35 paired pregnancies",len(paired_df)==35,len(paired_df))
add_check("56 singleton pregnancies",int((preg_comp["pairing_status"]=="singleton").sum())==56,
          int((preg_comp["pairing_status"]=="singleton").sum()))
add_check("5 outer folds",master_oof["outer_fold"].nunique()==5,sorted(master_oof["outer_fold"].unique()))
add_check("One held-out prediction per record",master_oof["record_name"].nunique()==len(master_oof),
          master_oof["record_name"].nunique())
add_check("No missing primary probabilities",
          not master_oof[["prob_RGB","prob_S1","prob_S2","prob_S3","prob_ensemble_raw"]].isna().any().any())
add_check("All primary Platt slopes positive",(master_oof["platt_slope"].astype(float)>0).all())
add_check("Primary mean AUC finite",np.isfinite(float(primary_auc_record["mean_outer_fold_auc_primary"])))
add_check("Primary CI ordered",
          float(primary_auc_record["ci95_low"]) <= float(primary_auc_record["mean_outer_fold_auc_primary"]) <=
          float(primary_auc_record["ci95_high"]))
add_check("Signal reads 126/126",int(integrity_df["read_success"].sum())==126,int(integrity_df["read_success"].sum()))
add_check("Boundary-support audit has 128 rows",len(boundary_audit)==N_FREQUENCIES,len(boundary_audit))
add_check("Boundary sensitivity result exists",
          "bandlimited_boundary_masked_uint8" in set(sensitivity_summary["Variant"].astype(str)),
          list(sensitivity_summary["Variant"].astype(str)))

audit_df=pd.DataFrame(checks)
save_table(audit_df,"FINAL_reproducibility_audit")
display(audit_df)

if not audit_df["Pass"].all():
    raise RuntimeError("One or more final reproducibility checks failed. Inspect FINAL_reproducibility_audit.csv")

# Figure manifest.
figure_files=sorted(FIG_DIR.glob("*.png"))+sorted(FIG_DIR.glob("*.pdf"))
figure_manifest=pd.DataFrame([{
    "file":p.name,
    "path":str(p),
    "size_bytes":p.stat().st_size,
    "sha256":sha256_file(p),
} for p in figure_files])
save_table(figure_manifest,"figure_manifest")

# Output manifest for provenance.
manifest_rows=[]
for p in PROJECT_ROOT.rglob("*"):
    if p.is_file():
        # Skip raw downloaded signal files from checksum manifest to keep this cell practical.
        if DATA_ROOT in p.parents:
            continue
        try:
            manifest_rows.append({
                "relative_path":str(p.relative_to(PROJECT_ROOT)),
                "size_bytes":p.stat().st_size,
            })
        except Exception:
            pass
manifest_df=pd.DataFrame(manifest_rows)
manifest_df.to_csv(EXPORT_DIR/"output_manifest.csv",index=False)

# README for Zenodo/GitHub.
readme = f"""# ICEHG recording-period classification — final reviewer rebuild

Dataset: PhysioNet ICEHG DS v1.0.1
Records: {len(master_oof)}
Pregnancies: {master_oof['pregnancy_id'].nunique()}
Primary validation: 5 outer / 3 inner pregnancy-wise GroupKFold.

Corrected inner protocol:
outer-development -> inner fit / pregnancy-disjoint early-stop / untouched inner evaluation.
Only untouched inner-evaluation OOF predictions are used for fusion weight, Platt calibration,
and threshold selection.

Primary representation:
Morlet (`morl`), integer scales 1–128, PyWavelets convolution, 0.08–5 Hz prefilter,
120-s windows with 60-s stride, per-segment/channel min-max to uint8,
PIL bilinear 224x224, float32/255 model input.

Reviewer sensitivities:
- legacy CWT + bicubic resize;
- 0.08–5 Hz physical Morlet frequency grid + uint8/bilinear;
- 0.08–5 Hz physical Morlet grid + log1p(power) floating-point input;\n- 0.08–5 Hz physical Morlet grid + scale-dependent finite-support boundary masking;
- simple Welch spectral baselines;
- repeated seeds 2026–2030;
- CNN-only and Transformer-only architecture ablations;
- paired/singleton, metadata-confounder, delivery-mode, calibration, bootstrap, and Grad-CAM analyses.

Primary AUC is the arithmetic mean of held-out outer-fold raw ROC-AUC values.
Pooled AUC is secondary descriptive only.

No alternative CWT pipeline, architecture, or random seed is selected on outer-test performance.
"""
with open(EXPORT_DIR/"README_reproduce.md","w",encoding="utf-8") as f:
    f.write(readme)

# Copy lightweight provenance files into export folder.
for src in [
    PROJECT_ROOT/"environment_versions.json",
    PROJECT_ROOT/"pip_freeze.txt",
    TABLE_DIR/"FINAL_key_results.csv",
    TABLE_DIR/"FINAL_reproducibility_audit.csv",
    TABLE_DIR/"reviewer_coverage_manifest.csv",
]:
    if src.exists():
        shutil.copy2(src,EXPORT_DIR/src.name)

# Create a compact code/results ZIP. Large raw data, scalogram arrays, and .keras models remain
# in the project tree and are referenced in the manifest; set INCLUDE_LARGE_ARTIFACTS=True if needed.
INCLUDE_LARGE_ARTIFACTS = False
zip_stage = EXPORT_DIR / "_zip_stage"
if zip_stage.exists():
    shutil.rmtree(zip_stage)
zip_stage.mkdir(parents=True,exist_ok=True)

for folder in [TABLE_DIR,FIG_DIR,META_ROOT,BASELINE_ROOT,SENSITIVITY_ROOT,ABLATION_ROOT,GRADCAM_ROOT,REPEAT_ROOT]:
    target=zip_stage/folder.name
    def ignore_large(directory,names):
        ignored=[]
        if not INCLUDE_LARGE_ARTIFACTS:
            for n in names:
                p=Path(directory)/n
                if p.is_file() and p.stat().st_size>100*1024*1024:
                    ignored.append(n)
        return ignored
    shutil.copytree(folder,target,dirs_exist_ok=True,ignore=ignore_large)

for p in [EXPORT_DIR/"README_reproduce.md",EXPORT_DIR/"output_manifest.csv",
          PROJECT_ROOT/"environment_versions.json",PROJECT_ROOT/"pip_freeze.txt"]:
    if p.exists():
        shutil.copy2(p,zip_stage/p.name)

zip_path=shutil.make_archive(
    str(EXPORT_DIR/"ICEHG_FINAL_REVIEWER_REBUILD_results"),
    "zip", root_dir=zip_stage
)
shutil.rmtree(zip_stage)

print("="*100)
print("FINAL REPRODUCIBILITY AUDIT: PASS")
print("="*100)
display(final_key_results)
print("All tables workbook:",workbook_path)
print("Results ZIP:",zip_path)
print("Project root:",PROJECT_ROOT)


## Reviewer 3 v3 additions

- Adds **one new model-based sensitivity experiment**: `bandlimited_boundary_masked_uint8`.
- Adds a 128-row support-based boundary audit and a direct paired pregnancy-cluster bootstrap comparison of masked vs unmasked 0.08–5 Hz CWT representations.
- Reuses completed CLEAN v2 primary folds and existing sensitivity/ablation/baseline outputs whenever their result files are present (`FORCE_* = False`).
- Adds explicit fold-wise Platt slope/intercept export, fusion-grid specification, fold-composition/AUC audit, dataset/seed manifest, Reviewer 3 output index, and a larger Figure 8.
- Does **not** add alternative segment-duration training or alternative outer-fold partition retraining; those remain scope/limitations rather than required new experiments.

- Adds descriptive maternal/acquisition summaries by **recording period × paired/singleton status**.
- Adds a descriptive distribution summary of paired architecture AUC differences across the five prespecified seeds.
- Adds primary training-epoch, parameter-count, and archived model-size summaries without retraining.
